In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


# New Section

In [ ]:
# This cell loads the prepared AMI baseline dataset from the existing project structure.

import os
import pandas as pd

PROJECT_DIR = "/content/drive/MyDrive/MTechIndProj/MoM_Project"
BASELINE_PATH = os.path.join(
    PROJECT_DIR,
    "data",
    "processed",
    "ami_baseline_dataset.csv"
)

print("Project directory:", PROJECT_DIR)
print("Baseline dataset:", BASELINE_PATH)
print("Dataset exists:", os.path.exists(BASELINE_PATH))

if not os.path.exists(BASELINE_PATH):
    raise FileNotFoundError(f"Dataset not found: {BASELINE_PATH}")

baseline_df = pd.read_csv(
    BASELINE_PATH,
    encoding="utf-8"
)

print("\nDataset loaded successfully.")
print("Shape:", baseline_df.shape)
print("Columns:", baseline_df.columns.tolist())
print("Meetings:", baseline_df["meeting_id"].tolist())

Project directory: /content/drive/MyDrive/MTechIndProj/MoM_Project
Baseline dataset: /content/drive/MyDrive/MTechIndProj/MoM_Project/data/processed/ami_baseline_dataset.csv
Dataset exists: True

Dataset loaded successfully.
Shape: (10, 6)
Columns: ['meeting_id', 'transcript', 'reference_summary', 'clean_transcript', 'timestamped_words', 'clean_words']
Meetings: ['ES2004a', 'ES2004b', 'ES2004c', 'ES2004d', 'ES2005a', 'ES2005b', 'ES2005c', 'ES2006a', 'ES2006b', 'ES2008a']


In [ ]:
# This cell compares the raw and existing cleaned transcript for ES2004a before we send it to the summarization model.

meeting_id = "ES2004a"

row = baseline_df.loc[
    baseline_df["meeting_id"] == meeting_id
].iloc[0]

print("=" * 70)
print("RAW TRANSCRIPT — FIRST 1500 CHARACTERS")
print("=" * 70)
print(row["transcript"][:1500])

print("\n" + "=" * 70)
print("CLEAN TRANSCRIPT — FIRST 1500 CHARACTERS")
print("=" * 70)
print(row["clean_transcript"][:1500])

print("\n" + "=" * 70)
print("CLEAN WORDS — FIRST 1000 CHARACTERS")
print("=" * 70)
print(str(row["clean_words"])[:1000])

RAW TRANSCRIPT — FIRST 1500 CHARACTERS
[0.37 - 0.95] A: Hmm
[0.95 - 1.53] A: hmm
[1.53 - 1.76] A: hmm
[1.76 - 1.76] A: .
[10.99 - 11.02] B: Are
[11.02 - 12.13] B: we
[12.13 - 12.29] B: we're
[12.29 - 12.42] B: not
[12.42 - 12.62] B: allowed
[12.62 - 12.70] B: to
[12.70 - 12.84] B: dim
[12.84 - 12.91] B: the
[12.91 - 13.18] B: lights
[13.18 - 13.31] B: so
[13.31 - 13.53] B: people
[13.53 - 13.71] B: can
[13.71 - 13.81] B: see
[13.81 - 13.96] B: that
[13.96 - 13.99] B: a
[13.99 - 14.15] B: bit
[14.15 - 14.53] B: better
[14.53 - 14.53] B: ?
[17.88 - 18.15] A: Yeah
[18.15 - 18.15] A: .
[18.87 - 19.70] B: Okay
[19.70 - 19.70] B: ,
[19.70 - 19.99] B: that's
[19.99 - 20.29] B: fine
[20.29 - 20.29] B: .
[22.37 - 22.50] B: Am
[22.50 - 22.56] B: I
[22.56 - 22.78] B: supposed
[22.78 - 22.84] B: to
[22.84 - 22.90] B: be
[22.90 - 23.28] B: standing
[23.28 - 23.44] B: up
[23.44 - 23.81] B: there
[23.81 - 23.81] B: ?
[25.15 - 25.23] D: So
[25.18 - 25.60] B: Okay
[25.23 - 25.33] D: we've
[25.33 - 25.4

In [ ]:
# This cell reconstructs the word-level AMI transcript into readable speaker utterances for BART input.

import re
import pandas as pd

def reconstruct_transcript(clean_transcript):
    """
    Convert AMI word-level speaker annotations into readable text.

    Example:
        B: Hello
        B: everybody
        A: Hi
        A: ,
        A: good
        A: morning
        B: .

    becomes:
        B: Hello everybody.
        A: Hi, good morning.
    """

    if not isinstance(clean_transcript, str):
        return ""

    lines = clean_transcript.splitlines()

    utterances = []
    current_speaker = None
    current_words = []

    punctuation = {".", ",", "?", "!", ";", ":"}

    def flush_utterance():
        nonlocal current_speaker, current_words

        if not current_words:
            return

        text = ""

        for word in current_words:
            word = word.strip()

            if not word:
                continue

            if word in punctuation:
                text = text.rstrip() + word
            else:
                if text:
                    text += " "
                text += word

        text = re.sub(r"\s+", " ", text).strip()

        if text:
            utterances.append(f"{current_speaker}: {text}")

        current_words = []

    for line in lines:

        line = line.strip()

        if not line:
            continue

        match = re.match(r"^([A-Z]):\s*(.*)$", line)

        if not match:
            continue

        speaker = match.group(1)
        word = match.group(2).strip()

        # Speaker changed → finish previous utterance
        if current_speaker is not None and speaker != current_speaker:
            flush_utterance()

        current_speaker = speaker

        if word:
            current_words.append(word)

        # Sentence-ending punctuation → finish utterance
        if word in {".", "?", "!"}:
            flush_utterance()

    # Add final unfinished utterance
    flush_utterance()

    return "\n".join(utterances)


# Test on ES2004a
test_clean = baseline_df.loc[
    baseline_df["meeting_id"] == "ES2004a",
    "clean_transcript"
].iloc[0]

bart_test = reconstruct_transcript(test_clean)

print("=" * 70)
print("BART-READY TRANSCRIPT — ES2004a")
print("=" * 70)
print(bart_test[:3000])

BART-READY TRANSCRIPT — ES2004a
A: Hmm hmm hmm.
B: Are we we're not allowed to dim the lights so people can see that a bit better?
A: Yeah.
B: Okay, that's fine.
B: Am I supposed to be standing up there?
D: So
B: Okay
D: we've got both
B: .
D: of these clipped on?
D: She gonna answer me
B: Yeah
D: or not
B: , I've got
D: ?
D: Right, both of them, okay.
B: Yes.
D: God.
D: Jesus, it's gonna fall off.
A: Okay.
A: Yep, yep.
A: Okay.
B: Okay
A: Tu tu tu tu
B: .
B: Hello everybody
A: Hi, good morning
B: .
B: Um
A: .
B: I'm Sarah, the Project Manager and this is our first meeting, surprisingly enough.
B: Okay, this is our agenda, um we will do some stuff, get to know each other a bit better to feel more comfortable with each other.
B: Um then we'll go do tool training, talk about the project plan, discuss our own ideas and everything um and we've got twenty five minutes to do that, as far as I can understand.
B: Now, we're developing a remote control which you probably already know.
B: Um, we

In [ ]:
# This cell creates a speaker-neutral BART input while preserving the chronological content of the AMI transcript.

def create_bart_input(reconstructed_transcript):
    """
    Remove AMI speaker labels while preserving the reconstructed
    chronological transcript.
    """

    if not isinstance(reconstructed_transcript, str):
        return ""

    lines = reconstructed_transcript.splitlines()
    cleaned_lines = []

    for line in lines:
        line = line.strip()

        if not line:
            continue

        # Remove speaker label such as A:, B:, C:, D:
        line = re.sub(r"^[A-Z]:\s*", "", line)

        # Normalize whitespace
        line = re.sub(r"\s+", " ", line).strip()

        if line:
            cleaned_lines.append(line)

    # Join utterances into paragraphs
    text = " ".join(cleaned_lines)

    # Normalize spaces before punctuation
    text = re.sub(r"\s+([,.!?;:])", r"\1", text)

    # Normalize repeated spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text


bart_input_test = create_bart_input(bart_test)

print("=" * 70)
print("SPEAKER-NEUTRAL BART INPUT — ES2004a")
print("=" * 70)
print(bart_input_test[:3000])

print("\n" + "=" * 70)
print("WORD COUNT")
print("=" * 70)
print(len(bart_input_test.split()))

SPEAKER-NEUTRAL BART INPUT — ES2004a
Hmm hmm hmm. Are we we're not allowed to dim the lights so people can see that a bit better? Yeah. Okay, that's fine. Am I supposed to be standing up there? So Okay we've got both. of these clipped on? She gonna answer me Yeah or not, I've got? Right, both of them, okay. Yes. God. Jesus, it's gonna fall off. Okay. Yep, yep. Okay. Okay Tu tu tu tu. Hello everybody Hi, good morning. Um. I'm Sarah, the Project Manager and this is our first meeting, surprisingly enough. Okay, this is our agenda, um we will do some stuff, get to know each other a bit better to feel more comfortable with each other. Um then we'll go do tool training, talk about the project plan, discuss our own ideas and everything um and we've got twenty five minutes to do that, as far as I can understand. Now, we're developing a remote control which you probably already know. Um, we want it to be original, something that's uh people haven't thought of, that's not out in the shops, um, t

In [ ]:
# This cell creates BART-ready inputs for all meetings and checks their lengths before chunking.

baseline_df["bart_input"] = baseline_df["clean_transcript"].apply(
    lambda x: create_bart_input(reconstruct_transcript(x))
)

baseline_df["bart_input_words"] = baseline_df["bart_input"].apply(
    lambda x: len(x.split())
)

print("=" * 70)
print("BART INPUT LENGTHS")
print("=" * 70)

print(
    baseline_df[
        ["meeting_id", "bart_input_words"]
    ].to_string(index=False)
)

print("\nAverage BART input words:",
      round(baseline_df["bart_input_words"].mean()))

print("Maximum BART input words:",
      baseline_df["bart_input_words"].max())

print("Minimum BART input words:",
      baseline_df["bart_input_words"].min())

BART INPUT LENGTHS
meeting_id  bart_input_words
   ES2004a              2614
   ES2004b              6763
   ES2004c              6968
   ES2004d              6134
   ES2005a               747
   ES2005b              6190
   ES2005c              6694
   ES2006a              2777
   ES2006b              6201
   ES2008a              2506

Average BART input words: 4759
Maximum BART input words: 6968
Minimum BART input words: 747


In [ ]:
# This cell splits each BART-ready meeting transcript into overlapping chunks that can be processed safely by BART.

CHUNK_WORDS = 900
OVERLAP_WORDS = 100


def chunk_text(text, chunk_words=CHUNK_WORDS, overlap_words=OVERLAP_WORDS):
    """Split text into overlapping word-based chunks."""

    words = text.split()

    if not words:
        return []

    chunks = []
    start = 0

    while start < len(words):
        end = min(start + chunk_words, len(words))

        chunk = " ".join(words[start:end])
        chunks.append(chunk)

        if end >= len(words):
            break

        start = end - overlap_words

    return chunks


# Test chunking on ES2004a
test_chunks = chunk_text(
    baseline_df.loc[
        baseline_df["meeting_id"] == "ES2004a",
        "bart_input"
    ].iloc[0]
)

print("=" * 70)
print("CHUNKING TEST — ES2004a")
print("=" * 70)

print("Total words:", len(
    baseline_df.loc[
        baseline_df["meeting_id"] == "ES2004a",
        "bart_input"
    ].iloc[0].split()
))

print("Number of chunks:", len(test_chunks))

for i, chunk in enumerate(test_chunks, 1):
    print(f"\nChunk {i}: {len(chunk.split())} words")
    print(chunk[:300] + "...")

CHUNKING TEST — ES2004a
Total words: 2614
Number of chunks: 4

Chunk 1: 900 words
Hmm hmm hmm. Are we we're not allowed to dim the lights so people can see that a bit better? Yeah. Okay, that's fine. Am I supposed to be standing up there? So Okay we've got both. of these clipped on? She gonna answer me Yeah or not, I've got? Right, both of them, okay. Yes. God. Jesus, it's gonna ...

Chunk 2: 900 words
Uh. It's not a vampire bat honestly Okay, yeah.. Uh and somewhere there's a body behind Okay. That's, some my dreadful sort of that's the worst yet bird, that's. it's meant to be an eagle A seagu Ah Eagle right eagle, okay, right., not you okay can a seagull tell. it's a flying animal could. have be...

Chunk 3: 900 words
they're. when you've got the main things on the front of it and a section opens up or something to the other functions where you can do sound or options Oh yeah kind. of recording, things like that inside it Mm-hmm.. 'Cause it doesn't make when you pick it up it doesn't

In [ ]:
# This cell loads the BART summarization model on the available Tesla T4 GPU for a single-chunk baseline test.

import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL_NAME = "facebook/bart-large-cnn"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Model:", MODEL_NAME)
print("Device:", DEVICE)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

bart_model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME
).to(DEVICE)

bart_model.eval()

print("BART loaded successfully.")

Model: facebook/bart-large-cnn
Device: cuda


config.json:   0%|          | 0.00/1.58k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

BART loaded successfully.


In [ ]:
# This cell generates a first BART summary from one ES2004a transcript chunk to verify the model output before processing all meetings.

import torch

test_chunk = test_chunks[0]

inputs = tokenizer(
    test_chunk,
    return_tensors="pt",
    max_length=1024,
    truncation=True
)

inputs = {
    key: value.to(DEVICE)
    for key, value in inputs.items()
}

with torch.no_grad():
    summary_ids = bart_model.generate(
        **inputs,
        max_length=180,
        min_length=60,
        num_beams=4,
        length_penalty=2.0,
        no_repeat_ngram_size=3,
        early_stopping=True
    )

test_summary = tokenizer.decode(
    summary_ids[0],
    skip_special_tokens=True
)

print("=" * 70)
print("BART TEST SUMMARY — ES2004a CHUNK 1")
print("=" * 70)
print(test_summary)

BART TEST SUMMARY — ES2004a CHUNK 1
Sarah, the Project Manager, introduces the team. They will work on a remote control that can be controlled by a dog. The team then draw their favourite animal on a white board. The group then discuss their ideas and work on the design. The project is due to be completed by the end of the year.


In [ ]:
# This cell creates stable BART token chunks without altering BPE spacing during decoding.

BART_CHUNK_TOKENS = 900
BART_OVERLAP_TOKENS = 100


def chunk_text_by_tokens(
    text,
    tokenizer,
    chunk_tokens=BART_CHUNK_TOKENS,
    overlap_tokens=BART_OVERLAP_TOKENS
):
    """Create overlapping chunks directly from BART token IDs."""

    token_ids = tokenizer.encode(
        text,
        add_special_tokens=False
    )

    if not token_ids:
        return []

    chunks = []
    start = 0

    while start < len(token_ids):

        end = min(
            start + chunk_tokens,
            len(token_ids)
        )

        chunk_ids = token_ids[start:end]

        chunk_text = tokenizer.decode(
            chunk_ids,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False
        ).strip()

        chunks.append({
            "text": chunk_text,
            "token_count": len(chunk_ids),
            "start_token": start,
            "end_token": end
        })

        if end >= len(token_ids):
            break

        start = end - overlap_tokens

    return chunks


# Test the corrected token-aware chunking on ES2004a.

es2004a_bart_input = baseline_df.loc[
    baseline_df["meeting_id"] == "ES2004a",
    "bart_input"
].iloc[0]

token_chunks = chunk_text_by_tokens(
    es2004a_bart_input,
    tokenizer
)

print("=" * 70)
print("CORRECTED TOKEN-AWARE CHUNKING — ES2004a")
print("=" * 70)

total_tokens = len(
    tokenizer.encode(
        es2004a_bart_input,
        add_special_tokens=False
    )
)

print("Total input tokens:", total_tokens)
print("Number of chunks:", len(token_chunks))

for i, chunk in enumerate(token_chunks, 1):
    print(
        f"\nChunk {i}: "
        f"{chunk['token_count']} tokens "
        f"({chunk['start_token']} → {chunk['end_token']})"
    )
    print(chunk["text"][:300] + "...")

CORRECTED TOKEN-AWARE CHUNKING — ES2004a
Total input tokens: 3463
Number of chunks: 5

Chunk 1: 900 tokens (0 → 900)
Hmm hmm hmm. Are we we're not allowed to dim the lights so people can see that a bit better? Yeah. Okay, that's fine. Am I supposed to be standing up there? So Okay we've got both. of these clipped on? She gonna answer me Yeah or not, I've got? Right, both of them, okay. Yes. God. Jesus, it's gonna ...

Chunk 2: 900 tokens (800 → 1700)
gonna be because that looks like a beak now, so. Crocodile? Gonna be Yeah a, it bird can be. a crocodile, it can be Is a it crocodile gonna be. Well it was it was it's an gonna at be first a bird firstly. it was an attempt at a T_ Rex and then it sort O of changed into a pelican but it can be a croc...

Chunk 3: 900 tokens (1600 → 2500)
mm.. Um, and profit aim is fifty million Euros, which is uh In our first year? Yi yes, um yeah, I presume so Mm-hmm. So. Um then You've got market range international and you did say earlier it's got to be 

In [ ]:
# This cell generates an abstractive BART summary for each token-aware ES2004a chunk.

def summarize_chunk(chunk_text):
    inputs = tokenizer(
        chunk_text,
        return_tensors="pt",
        max_length=1024,
        truncation=False
    )

    inputs = {
        key: value.to(DEVICE)
        for key, value in inputs.items()
    }

    with torch.no_grad():
        summary_ids = bart_model.generate(
            **inputs,
            max_length=180,
            min_length=40,
            num_beams=4,
            length_penalty=2.0,
            no_repeat_ngram_size=3,
            early_stopping=True
        )

    return tokenizer.decode(
        summary_ids[0],
        skip_special_tokens=True
    )


chunk_summaries = []

print("=" * 70)
print("BART CHUNK SUMMARIZATION — ES2004a")
print("=" * 70)

for i, chunk in enumerate(token_chunks, 1):

    print(f"\nProcessing chunk {i}/{len(token_chunks)}...")

    summary = summarize_chunk(chunk["text"])

    chunk_summaries.append(summary)

    print(f"\nCHUNK {i} SUMMARY:")
    print(summary)

print("\n" + "=" * 70)
print("ALL CHUNK SUMMARIES GENERATED")
print("=" * 70)
print("Number of summaries:", len(chunk_summaries))

BART CHUNK SUMMARIZATION — ES2004a

Processing chunk 1/5...

CHUNK 1 SUMMARY:
Sarah, the Project Manager, introduces the team. They will work on a remote control that can be controlled by a dog. The team then draw their favourite animal on the white board. The group then discuss their ideas and work on the design.

Processing chunk 2/5...

CHUNK 2 SUMMARY:
It was an attempt at a T_ Rex and then it sort O of changed into a pelican but it can be a crocodile now actually. We've got a selling price at twenty five Euros, which I don't actually know what that is in Pounds, at all. The profit aim is fifty million Euros.

Processing chunk 3/5...

CHUNK 3 SUMMARY:
The company is aiming for a profit of fifty million Euros in its first year. It is aimed at the international market, not the business market. The company is also targeting the older generation.

Processing chunk 4/5...

CHUNK 4 SUMMARY:
I don't know how for twenty fi, or twelve Euros fifty how much of a excellent screen you could get

In [ ]:
# This cell combines the five independently generated BART chunk summaries into one baseline meeting-level summary input.

combined_chunk_summary = " ".join(chunk_summaries)

print("=" * 70)
print("COMBINED CHUNK SUMMARIES — ES2004a")
print("=" * 70)

print(combined_chunk_summary)

print("\n" + "=" * 70)
print("COMBINED SUMMARY TOKEN COUNT")
print("=" * 70)

combined_tokens = tokenizer.encode(
    combined_chunk_summary,
    add_special_tokens=False
)

print("Tokens:", len(combined_tokens))

COMBINED CHUNK SUMMARIES — ES2004a
Sarah, the Project Manager, introduces the team. They will work on a remote control that can be controlled by a dog. The team then draw their favourite animal on the white board. The group then discuss their ideas and work on the design. It was an attempt at a T_ Rex and then it sort O of changed into a pelican but it can be a crocodile now actually. We've got a selling price at twenty five Euros, which I don't actually know what that is in Pounds, at all. The profit aim is fifty million Euros. The company is aiming for a profit of fifty million Euros in its first year. It is aimed at the international market, not the business market. The company is also targeting the older generation. I don't know how for twenty fi, or twelve Euros fifty how much of a excellent screen you could get Yeah, you'd you'd have to keep it down to a black and white L_C_D_ thing anyway. The other thing is, just ch chucking into mobile phone f design features again, it could h

In [ ]:
# This cell performs the second-stage BART summarization to create the final meeting-level baseline MoM for ES2004a.

final_inputs = tokenizer(
    combined_chunk_summary,
    return_tensors="pt",
    max_length=1024,
    truncation=True
)

final_inputs = {
    key: value.to(DEVICE)
    for key, value in final_inputs.items()
}

with torch.no_grad():
    final_summary_ids = bart_model.generate(
        **final_inputs,
        max_length=220,
        min_length=80,
        num_beams=4,
        length_penalty=2.0,
        no_repeat_ngram_size=3,
        early_stopping=True
    )

es2004a_baseline_summary = tokenizer.decode(
    final_summary_ids[0],
    skip_special_tokens=True
)

print("=" * 70)
print("FINAL BART BASELINE — ES2004a")
print("=" * 70)
print(es2004a_baseline_summary)

FINAL BART BASELINE — ES2004a
The company is aiming for a profit of fifty million Euros in its first year. It is aimed at the international market, not the business market. The company is also targeting the older generation. It could have a flip top remote control so that when you flip over the top, your screen is you can have a bigger screen in Mm-hmm the. flip over. . Just. just a quick thing about Sure the. um about what you're saying about the uh does does it need to be fashionable?


Then we'll compare it with the AMI reference

The reference for ES2004a is:

The Project Manager gave an introduction to the goal of the project,
to create a trendy yet user-friendly remote.

She presented a long-range agenda for the whole project.

The group introduced themselves to each other and practiced with
the meeting room tools by drawing on the board.

The Project Manager presented the project budget, the projected
price point, and the projected profit aim for the project.

Then the group began a discussion about their own experiences with
remote controls to generate initial design ideas for making the
product user-friendly.

They discussed grouping features into a menu and adding an LCD display.

They also discussed the look of various materials that may be used
in the design, in keeping with the company's goal to create
fashionable electronics.

The important baseline question is now:

Can plain BART recover the important meeting-level information from the transcript?

In [ ]:
# This cell records the exact baseline BART configuration so later improvements can be compared fairly.

BASELINE_CONFIG = {
    "model": "facebook/bart-large-cnn",
    "device": str(DEVICE),
    "chunk_tokens": BART_CHUNK_TOKENS,
    "overlap_tokens": BART_OVERLAP_TOKENS,
    "chunk_max_length": 180,
    "chunk_min_length": 40,
    "final_max_length": 220,
    "final_min_length": 80,
    "num_beams": 4,
    "length_penalty": 2.0,
    "no_repeat_ngram_size": 3,
    "dataset": "AMI",
    "meetings": len(baseline_df)
}

print("=" * 70)
print("BART BASELINE CONFIGURATION")
print("=" * 70)

for key, value in BASELINE_CONFIG.items():
    print(f"{key:25}: {value}")

BART BASELINE CONFIGURATION
model                    : facebook/bart-large-cnn
device                   : cuda
chunk_tokens             : 900
overlap_tokens           : 100
chunk_max_length         : 180
chunk_min_length         : 40
final_max_length         : 220
final_min_length         : 80
num_beams                : 4
length_penalty           : 2.0
no_repeat_ngram_size     : 3
dataset                  : AMI
meetings                 : 10


In [ ]:
# This cell runs the same BART baseline pipeline on all 10 AMI meetings and saves each generated MoM.

import os
import json
import time

MOM_OUTPUT_DIR = os.path.join(
    PROJECT_DIR,
    "outputs",
    "mom"
)

os.makedirs(MOM_OUTPUT_DIR, exist_ok=True)

all_baseline_results = []

print("=" * 70)
print("RUNNING BART BASELINE — ALL AMI MEETINGS")
print("=" * 70)

for idx, row in baseline_df.iterrows():

    meeting_id = row["meeting_id"]
    bart_input = row["bart_input"]

    print(
        f"\n[{idx + 1}/{len(baseline_df)}] "
        f"Processing {meeting_id}..."
    )

    start_time = time.time()

    # Create token-aware chunks
    meeting_chunks = chunk_text_by_tokens(
        bart_input,
        tokenizer
    )

    meeting_chunk_summaries = []

    # Generate summary for each chunk
    for chunk_idx, chunk in enumerate(meeting_chunks, 1):

        summary = summarize_chunk(
            chunk["text"]
        )

        meeting_chunk_summaries.append(summary)

    # Combine chunk summaries
    combined_summary = " ".join(
        meeting_chunk_summaries
    )

    # Token count of combined summaries
    combined_token_count = len(
        tokenizer.encode(
            combined_summary,
            add_special_tokens=False
        )
    )

    # Second-stage summarization
    final_inputs = tokenizer(
        combined_summary,
        return_tensors="pt",
        max_length=1024,
        truncation=True
    )

    final_inputs = {
        key: value.to(DEVICE)
        for key, value in final_inputs.items()
    }

    with torch.no_grad():

        final_summary_ids = bart_model.generate(
            **final_inputs,
            max_length=220,
            min_length=80,
            num_beams=4,
            length_penalty=2.0,
            no_repeat_ngram_size=3,
            early_stopping=True
        )

    final_summary = tokenizer.decode(
        final_summary_ids[0],
        skip_special_tokens=True
    )

    elapsed = time.time() - start_time

    # Store complete result
    result = {
        "meeting_id": meeting_id,
        "model": "facebook/bart-large-cnn",
        "num_chunks": len(meeting_chunks),
        "combined_summary_tokens": combined_token_count,
        "chunk_summaries": meeting_chunk_summaries,
        "final_summary": final_summary,
        "processing_time_seconds": round(elapsed, 2)
    }

    all_baseline_results.append(result)

    # Save individual meeting result
    output_path = os.path.join(
        MOM_OUTPUT_DIR,
        f"{meeting_id}_bart_baseline.json"
    )

    with open(
        output_path,
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(
            result,
            f,
            indent=2,
            ensure_ascii=False
        )

    print(
        f"  Chunks: {len(meeting_chunks)}"
    )

    print(
        f"  Combined tokens: {combined_token_count}"
    )

    print(
        f"  Time: {elapsed:.1f} sec"
    )

    print(
        f"  Saved: {output_path}"
    )

print("\n" + "=" * 70)
print("BART BASELINE RUN COMPLETE")
print("=" * 70)

print(
    "Meetings processed:",
    len(all_baseline_results)
)

RUNNING BART BASELINE — ALL AMI MEETINGS

[1/10] Processing ES2004a...
  Chunks: 5
  Combined tokens: 327
  Time: 14.1 sec
  Saved: /content/drive/MyDrive/MTechIndProj/MoM_Project/outputs/mom/ES2004a_bart_baseline.json

[2/10] Processing ES2004b...
  Chunks: 11
  Combined tokens: 794
  Time: 19.2 sec
  Saved: /content/drive/MyDrive/MTechIndProj/MoM_Project/outputs/mom/ES2004b_bart_baseline.json

[3/10] Processing ES2004c...
  Chunks: 11
  Combined tokens: 823
  Time: 18.9 sec
  Saved: /content/drive/MyDrive/MTechIndProj/MoM_Project/outputs/mom/ES2004c_bart_baseline.json

[4/10] Processing ES2004d...
  Chunks: 10
  Combined tokens: 604
  Time: 15.0 sec
  Saved: /content/drive/MyDrive/MTechIndProj/MoM_Project/outputs/mom/ES2004d_bart_baseline.json

[5/10] Processing ES2005a...
  Chunks: 2
  Combined tokens: 158
  Time: 4.4 sec
  Saved: /content/drive/MyDrive/MTechIndProj/MoM_Project/outputs/mom/ES2005a_bart_baseline.json

[6/10] Processing ES2005b...
  Chunks: 10
  Combined tokens: 661
 

In [ ]:
# This cell checks whether the Hugging Face Evaluate library is available for computing ROUGE scores.

import importlib.util

evaluate_available = (
    importlib.util.find_spec("evaluate") is not None
)

print("=" * 70)
print("EVALUATION LIBRARY CHECK")
print("=" * 70)

print("Evaluate installed:", evaluate_available)

EVALUATION LIBRARY CHECK
Evaluate installed: False


In [ ]:
# This cell installs the Hugging Face evaluation package and ROUGE dependency required for baseline evaluation.

!pip install -q evaluate rouge_score

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.7 MB/s eta 0:00:00


In [ ]:
# This cell verifies that the evaluation library and ROUGE metric can now be imported successfully.

import evaluate

print("=" * 70)
print("EVALUATION LIBRARY READY")
print("=" * 70)

print("Evaluate version:", evaluate.__version__)

rouge = evaluate.load("rouge")

print("ROUGE metric loaded successfully.")

EVALUATION LIBRARY READY
Evaluate version: 0.4.6


ROUGE metric loaded successfully.


In [ ]:
# This cell computes ROUGE-1, ROUGE-2, and ROUGE-L for all 10 BART-generated MoMs against the AMI reference summaries.

import os
import json
import pandas as pd

rouge_results = []

print("=" * 70)
print("BART BASELINE — ROUGE EVALUATION")
print("=" * 70)

for _, row in baseline_df.iterrows():

    meeting_id = row["meeting_id"]

    output_path = os.path.join(
        MOM_OUTPUT_DIR,
        f"{meeting_id}_bart_baseline.json"
    )

    with open(
        output_path,
        "r",
        encoding="utf-8"
    ) as f:
        result = json.load(f)

    generated_summary = result["final_summary"]
    reference_summary = row["reference_summary"]

    scores = rouge.compute(
        predictions=[generated_summary],
        references=[reference_summary],
        use_stemmer=True
    )

    rouge_results.append({
        "meeting_id": meeting_id,
        "rouge1": scores["rouge1"],
        "rouge2": scores["rouge2"],
        "rougeL": scores["rougeL"]
    })

    print(
        f"{meeting_id}: "
        f"R1={scores['rouge1']:.4f} | "
        f"R2={scores['rouge2']:.4f} | "
        f"RL={scores['rougeL']:.4f}"
    )


rouge_df = pd.DataFrame(rouge_results)

print("\n" + "=" * 70)
print("AVERAGE BASELINE ROUGE")
print("=" * 70)

print(
    f"ROUGE-1: {rouge_df['rouge1'].mean():.4f}"
)

print(
    f"ROUGE-2: {rouge_df['rouge2'].mean():.4f}"
)

print(
    f"ROUGE-L: {rouge_df['rougeL'].mean():.4f}"
)

BART BASELINE — ROUGE EVALUATION
ES2004a: R1=0.2247 | R2=0.0226 | RL=0.1273
ES2004b: R1=0.1671 | R2=0.0148 | RL=0.1032
ES2004c: R1=0.2143 | R2=0.0240 | RL=0.1310
ES2004d: R1=0.1758 | R2=0.0239 | RL=0.0998
ES2005a: R1=0.1949 | R2=0.0104 | RL=0.1333
ES2005b: R1=0.2382 | R2=0.0891 | RL=0.1496
ES2005c: R1=0.2135 | R2=0.0524 | RL=0.1406
ES2006a: R1=0.2126 | R2=0.0462 | RL=0.1264
ES2006b: R1=0.1818 | R2=0.0400 | RL=0.0966
ES2008a: R1=0.1720 | R2=0.0432 | RL=0.1075

AVERAGE BASELINE ROUGE
ROUGE-1: 0.1995
ROUGE-2: 0.0367
ROUGE-L: 0.1215


In [ ]:
# This cell saves the BART baseline ROUGE scores for use in later comparisons and the final project evaluation.

EVALUATION_DIR = os.path.join(
    PROJECT_DIR,
    "evaluation_results"
)

os.makedirs(
    EVALUATION_DIR,
    exist_ok=True
)

ROUGE_OUTPUT_PATH = os.path.join(
    EVALUATION_DIR,
    "bart_baseline_rouge.csv"
)

rouge_df.to_csv(
    ROUGE_OUTPUT_PATH,
    index=False
)

print("=" * 70)
print("BASELINE EVALUATION SAVED")
print("=" * 70)
print(ROUGE_OUTPUT_PATH)

BASELINE EVALUATION SAVED
/content/drive/MyDrive/MTechIndProj/MoM_Project/evaluation_results/bart_baseline_rouge.csv


08-09-2026

In [ ]:
# ============================================================
# PROJECT PATH SETUP
# Purpose:
# Define the project directories and verify access to the
# saved transcript and MoM artifacts.
# ============================================================

from pathlib import Path

PROJECT_DIR = Path(
    "/content/drive/MyDrive/MTechIndProj/MoM_Project"
)

DATA_DIR = PROJECT_DIR / "data"
TRANSCRIPTS_DIR = DATA_DIR / "transcripts"

SPEAKER_TRANSCRIPT_PATH = (
    TRANSCRIPTS_DIR /
    "ES2004a_speaker_transcript.json"
)

MOM_CANDIDATES_PATH = (
    TRANSCRIPTS_DIR /
    "ES2004a_mom_candidates_rule_based.json"
)

MOM_WORTHY_PATH = (
    TRANSCRIPTS_DIR /
    "ES2004a_mom_worthy_candidates.json"
)

MOM_CLAIMS_PATH = (
    TRANSCRIPTS_DIR /
    "ES2004a_structured_mom_claims.json"
)

print("Project directory:", PROJECT_DIR)
print("Transcripts directory:", TRANSCRIPTS_DIR)

print("\nSaved artifacts:")

for path in [
    SPEAKER_TRANSCRIPT_PATH,
    MOM_CANDIDATES_PATH,
    MOM_WORTHY_PATH,
    MOM_CLAIMS_PATH
]:
    print(
        f"{'✓' if path.exists() else '✗'} "
        f"{path.name}"
    )

Project directory: /content/drive/MyDrive/MTechIndProj/MoM_Project
Transcripts directory: /content/drive/MyDrive/MTechIndProj/MoM_Project/data/transcripts

Saved artifacts:
✓ ES2004a_speaker_transcript.json
✓ ES2004a_mom_candidates_rule_based.json
✓ ES2004a_mom_worthy_candidates.json
✓ ES2004a_structured_mom_claims.json


In [ ]:
# ============================================================
# LOAD STRUCTURED MoM CLAIMS
# Purpose:
# Load the previously generated MoM claims so that the
# evidence retrieval stage can use them.
# ============================================================

import json

with open(MOM_CLAIMS_PATH, "r", encoding="utf-8") as f:
    mom_claims_data = json.load(f)

mom_claims = mom_claims_data["claims"]

print("=" * 80)
print("STRUCTURED MoM CLAIMS LOADED")
print("=" * 80)

print("\nMeeting ID:", mom_claims_data["meeting_id"])
print("Candidate count:", mom_claims_data["candidate_count"])
print("Topic groups:", mom_claims_data["topic_group_count"])
print("Claim count:", mom_claims_data["claim_count"])

print("\nFirst 3 claims:")

for claim in mom_claims[:3]:

    print(
        f"\nClaim {claim['claim_id']:02d} | "
        f"{claim['event_type']}"
    )

    print("Topic:", claim["topic"])
    print("Text:", claim["claim_text"])
    print("Sources:", claim["source_candidate_ids"])
    print(
        f"Time: {claim['start']:.3f}s → "
        f"{claim['end']:.3f}s"
    )
    print("Speaker:", claim["speaker"])

print("\n" + "=" * 80)

STRUCTURED MoM CLAIMS LOADED

Meeting ID: ES2004a
Candidate count: 42
Topic groups: 11
Claim count: 12

First 3 claims:

Claim 01 | INFORMATION
Topic: meeting_introduction_and_agenda
Text: The meeting was introduced as the first meeting, with an agenda covering team interaction, tool training, the project plan, and discussion of ideas.
Sources: [5, 6, 7, 8]
Time: 84.658s → 109.823s
Speaker: SPEAKER_02

Claim 02 | INFORMATION
Topic: product_objective_and_requirements
Text: The team is developing a remote control intended to be original, trendy, appealing to a wide market, and user-friendly for a broad range of users.
Sources: [10, 11, 12, 13]
Time: 117.699s → 141.016s
Speaker: SPEAKER_02

Claim 03 | INFORMATION
Topic: design_process
Text: The design process includes functional design, conceptual design, and detailed design, with individual work addressing product requirements and implementation.
Sources: [14, 15]
Time: 142.933s → 167.740s
Speaker: SPEAKER_02



In [ ]:
# ============================================================
# LOAD SPEAKER-ATTRIBUTED TRANSCRIPT
# Purpose:
# Load the speaker-attributed transcript that will serve as
# the evidence corpus for BGE + FAISS retrieval.
#
# Each utterance retains:
# - speaker
# - start timestamp
# - end timestamp
# - transcript text
# ============================================================

with open(
    SPEAKER_TRANSCRIPT_PATH,
    "r",
    encoding="utf-8"
) as f:
    transcript_data = json.load(f)

speaker_utterances = transcript_data["utterances"]

print("=" * 80)
print("SPEAKER-ATTRIBUTED TRANSCRIPT LOADED")
print("=" * 80)

print("\nMeeting ID:", transcript_data["meeting_id"])
print("Number of speakers:", transcript_data["num_speakers"])
print("Number of utterances:", transcript_data["num_utterances"])

print("\nFirst 3 evidence units:")

for item in speaker_utterances[:3]:

    print(
        f"\nUtterance {item.get('utterance_id', 'N/A')}"
    )

    print("Speaker:", item["speaker"])
    print(
        f"Time: {item['start']:.3f}s → "
        f"{item['end']:.3f}s"
    )
    print("Text:", item["text"])

print("\n" + "=" * 80)

SPEAKER-ATTRIBUTED TRANSCRIPT LOADED

Meeting ID: ES2004a
Number of speakers: 4
Number of utterances: 148

First 3 evidence units:

Utterance N/A
Speaker: SPEAKER_02
Time: 10.998s → 14.521s
Text: Are we, we're not like the dim lights, so we can see that a bit better.

Utterance N/A
Speaker: SPEAKER_01
Time: 17.943s → 18.163s
Text: Yeah.

Utterance N/A
Speaker: SPEAKER_02
Time: 18.944s → 20.945s
Text: Okay, that's fine.



In [ ]:
# ============================================================
# CREATE EVIDENCE DOCUMENTS
# Purpose:
# Convert the speaker-attributed transcript into standardized
# evidence units for semantic retrieval.
#
# Each evidence document contains:
# - evidence_id
# - speaker
# - start timestamp
# - end timestamp
# - duration
# - transcript text
#
# These IDs will later be used to link verified MoM claims
# back to their supporting transcript evidence.
# ============================================================

evidence_documents = []

for idx, utterance in enumerate(speaker_utterances, start=1):

    evidence_documents.append({
        "evidence_id": idx,
        "speaker": utterance["speaker"],
        "start": utterance["start"],
        "end": utterance["end"],
        "duration": utterance["end"] - utterance["start"],
        "text": utterance["text"]
    })


print("=" * 80)
print("EVIDENCE DOCUMENTS CREATED")
print("=" * 80)

print("\nNumber of evidence documents:",
      len(evidence_documents))

print("\nFirst 5 evidence documents:")

for evidence in evidence_documents[:5]:

    print(
        f"\nEvidence {evidence['evidence_id']:03d}"
    )

    print("Speaker:", evidence["speaker"])

    print(
        f"Time: {evidence['start']:.3f}s → "
        f"{evidence['end']:.3f}s"
    )

    print(
        f"Duration: "
        f"{evidence['duration']:.3f}s"
    )

    print("Text:", evidence["text"])

print("\n" + "=" * 80)

EVIDENCE DOCUMENTS CREATED

Number of evidence documents: 148

First 5 evidence documents:

Evidence 001
Speaker: SPEAKER_02
Time: 10.998s → 14.521s
Duration: 3.523s
Text: Are we, we're not like the dim lights, so we can see that a bit better.

Evidence 002
Speaker: SPEAKER_01
Time: 17.943s → 18.163s
Duration: 0.220s
Text: Yeah.

Evidence 003
Speaker: SPEAKER_02
Time: 18.944s → 20.945s
Duration: 2.001s
Text: Okay, that's fine.

Evidence 004
Speaker: SPEAKER_02
Time: 22.406s → 23.707s
Duration: 1.301s
Text: Am I supposed to be standing up there?

Evidence 005
Speaker: SPEAKER_03
Time: 25.128s → 26.509s
Duration: 1.381s
Text: So we've got both of these clipped on.



In [ ]:
# ============================================================
# LOAD BGE EMBEDDING MODEL
# Purpose:
# Load BGE-small to create semantic embeddings for the
# transcript evidence units and MoM claims.
# ============================================================

import torch
from sentence_transformers import SentenceTransformer

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

BGE_MODEL_NAME = "BAAI/bge-small-en-v1.5"

print("=" * 80)
print("LOADING BGE EMBEDDING MODEL")
print("=" * 80)

print("\nDevice:", DEVICE)
print("Model:", BGE_MODEL_NAME)

bge_model = SentenceTransformer(
    BGE_MODEL_NAME,
    device=DEVICE
)

print("\nBGE model loaded successfully.")

print("=" * 80)

LOADING BGE EMBEDDING MODEL

Device: cuda
Model: BAAI/bge-small-en-v1.5


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


BGE model loaded successfully.


In [ ]:
# ============================================================
# GENERATE EVIDENCE EMBEDDINGS
# Purpose:
# Convert each transcript evidence unit into a normalized
# BGE semantic embedding.
#
# Embedding dimension:
# 384
#
# Normalization allows FAISS inner-product similarity
# to behave as cosine similarity.
# ============================================================

evidence_texts = [
    evidence["text"]
    for evidence in evidence_documents
]

print("=" * 80)
print("GENERATING EVIDENCE EMBEDDINGS")
print("=" * 80)

print("\nEvidence documents:", len(evidence_texts))

evidence_embeddings = bge_model.encode(
    evidence_texts,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_numpy=True
)

print("\nEmbedding generation complete.")

print("Embedding shape:", evidence_embeddings.shape)
print("Embedding dtype:", evidence_embeddings.dtype)

print("\nExpected shape:")
print(f"({len(evidence_documents)}, 384)")

print("\n" + "=" * 80)

GENERATING EVIDENCE EMBEDDINGS

Evidence documents: 148


Batches:   0%|          | 0/5 [00:00<?, ?it/s]


Embedding generation complete.
Embedding shape: (148, 384)
Embedding dtype: float32

Expected shape:
(148, 384)



In [ ]:
# ============================================================
# BUILD FAISS EVIDENCE INDEX
# Purpose:
# Build a FAISS vector index over the 148 normalized BGE
# evidence embeddings.
#
# Because the embeddings are normalized, inner-product
# similarity is equivalent to cosine similarity.
# ============================================================

import faiss
import numpy as np

print("=" * 80)
print("BUILDING FAISS EVIDENCE INDEX")
print("=" * 80)

# Ensure embeddings are float32 for FAISS
evidence_embeddings = np.asarray(
    evidence_embeddings,
    dtype="float32"
)

embedding_dimension = evidence_embeddings.shape[1]

# Inner-product index
faiss_index = faiss.IndexFlatIP(
    embedding_dimension
)

# Add evidence embeddings
faiss_index.add(evidence_embeddings)

print("\nEmbedding dimension:", embedding_dimension)
print("Evidence vectors added:", faiss_index.ntotal)
print("FAISS index type:", type(faiss_index).__name__)

print("\nExpected vectors:", len(evidence_documents))

print(
    "\nIndex successfully built:",
    faiss_index.ntotal == len(evidence_documents)
)

print("\n" + "=" * 80)

ModuleNotFoundError: No module named 'faiss'

In [ ]:
# ============================================================
# INSTALL FAISS
# Purpose:
# Install FAISS for vector similarity search.
# ============================================================

!pip install -q faiss-cpu

print("FAISS installation completed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 71.0 MB/s eta 0:00:00
FAISS installation completed.


In [ ]:
# ============================================================
# BUILD FAISS EVIDENCE INDEX
# Purpose:
# Build a FAISS vector index over the 148 normalized BGE
# evidence embeddings.
# ============================================================

import faiss
import numpy as np

print("=" * 80)
print("BUILDING FAISS EVIDENCE INDEX")
print("=" * 80)

evidence_embeddings = np.asarray(
    evidence_embeddings,
    dtype="float32"
)

embedding_dimension = evidence_embeddings.shape[1]

faiss_index = faiss.IndexFlatIP(
    embedding_dimension
)

faiss_index.add(evidence_embeddings)

print("\nEmbedding dimension:", embedding_dimension)
print("Evidence vectors added:", faiss_index.ntotal)
print("FAISS index type:", type(faiss_index).__name__)

print("\nExpected vectors:", len(evidence_documents))

print(
    "\nIndex successfully built:",
    faiss_index.ntotal == len(evidence_documents)
)

print("\n" + "=" * 80)

BUILDING FAISS EVIDENCE INDEX

Embedding dimension: 384
Evidence vectors added: 148
FAISS index type: IndexFlatIP

Expected vectors: 148

Index successfully built: True



In [ ]:
# ============================================================
# EVIDENCE RETRIEVAL FUNCTION
# Purpose:
# Retrieve the most semantically relevant transcript evidence
# for a given MoM claim using BGE + FAISS.
#
# Important:
# Retrieval similarity identifies candidate evidence.
# It does NOT by itself prove that the evidence supports
# the claim. Verification will be performed in the next stage.
# ============================================================

def retrieve_evidence(claim_text, top_k=5):

    # Generate normalized BGE embedding for the claim
    claim_embedding = bge_model.encode(
        [claim_text],
        normalize_embeddings=True,
        convert_to_numpy=True
    ).astype("float32")

    # Search FAISS index
    similarities, indices = faiss_index.search(
        claim_embedding,
        top_k
    )

    retrieved = []

    for similarity, index in zip(
        similarities[0],
        indices[0]
    ):

        evidence = evidence_documents[index].copy()

        evidence["retrieval_similarity"] = float(
            similarity
        )

        retrieved.append(evidence)

    return retrieved


print("=" * 80)
print("EVIDENCE RETRIEVAL FUNCTION CREATED")
print("=" * 80)

print("\nFunction: retrieve_evidence()")
print("Default top-K:", 5)

print("\nRetrieval pipeline:")
print("Claim")
print("  ↓")
print("BGE embedding")
print("  ↓")
print("FAISS similarity search")
print("  ↓")
print("Top-K evidence candidates")

print("\n" + "=" * 80)

EVIDENCE RETRIEVAL FUNCTION CREATED

Function: retrieve_evidence()
Default top-K: 5

Retrieval pipeline:
Claim
  ↓
BGE embedding
  ↓
FAISS similarity search
  ↓
Top-K evidence candidates



In [ ]:
# ============================================================
# TEST EVIDENCE RETRIEVAL
# Purpose:
# Test BGE + FAISS retrieval on one structured MoM claim.
#
# Claim 2 discusses the remote control's objectives and
# usability requirements.
# ============================================================

TEST_CLAIM = mom_claims[1]

print("=" * 90)
print("EVIDENCE RETRIEVAL TEST")
print("=" * 90)

print("\nClaim ID:", TEST_CLAIM["claim_id"])
print("Topic:", TEST_CLAIM["topic"])
print("Claim:")
print(TEST_CLAIM["claim_text"])

print(
    f"\nClaim timestamp: "
    f"{TEST_CLAIM['start']:.3f}s → "
    f"{TEST_CLAIM['end']:.3f}s"
)

print("\n" + "-" * 90)
print("TOP 5 RETRIEVED EVIDENCE")
print("-" * 90)

retrieved_test_evidence = retrieve_evidence(
    TEST_CLAIM["claim_text"],
    top_k=5
)

for rank, evidence in enumerate(
    retrieved_test_evidence,
    start=1
):

    print(
        f"\nRank {rank}"
    )

    print(
        f"Evidence ID: "
        f"{evidence['evidence_id']:03d}"
    )

    print(
        f"Similarity: "
        f"{evidence['retrieval_similarity']:.4f}"
    )

    print(
        f"Speaker: "
        f"{evidence['speaker']}"
    )

    print(
        f"Time: "
        f"{evidence['start']:.3f}s → "
        f"{evidence['end']:.3f}s"
    )

    print(
        f"Text: "
        f"{evidence['text']}"
    )

print("\n" + "=" * 90)

EVIDENCE RETRIEVAL TEST

Claim ID: 2
Topic: product_objective_and_requirements
Claim:
The team is developing a remote control intended to be original, trendy, appealing to a wide market, and user-friendly for a broad range of users.

Claim timestamp: 117.699s → 141.016s

------------------------------------------------------------------------------------------
TOP 5 RETRIEVED EVIDENCE
------------------------------------------------------------------------------------------

Rank 1
Evidence ID: 014
Similarity: 0.7713
Speaker: SPEAKER_02
Time: 117.699s → 120.921s
Text: Now, we're developing a remote control, which you probably already know.

Rank 2
Evidence ID: 105
Similarity: 0.7575
Speaker: SPEAKER_01
Time: 707.248s → 710.889s
Text: remote controls. You want to integrate everything into one.

Rank 3
Evidence ID: 135
Similarity: 0.6684
Speaker: SPEAKER_03
Time: 935.208s → 946.993s
Text: The other thing is, just tacking into mobile phone design features again, you could have a flip top 

In [ ]:
# ============================================================
# IMPROVED EVIDENCE RETRIEVAL FUNCTION
# Purpose:
# Retrieve top-K evidence candidates while preserving their
# retrieval rank.
#
# Retrieval similarity is used only to find candidate evidence.
# It is NOT treated as proof of claim correctness.
# ============================================================

def retrieve_evidence(claim_text, top_k=10):

    # Generate normalized BGE embedding for the claim
    claim_embedding = bge_model.encode(
        [claim_text],
        normalize_embeddings=True,
        convert_to_numpy=True
    ).astype("float32")

    # Search FAISS
    similarities, indices = faiss_index.search(
        claim_embedding,
        top_k
    )

    retrieved = []

    for rank, (similarity, index) in enumerate(
        zip(similarities[0], indices[0]),
        start=1
    ):

        evidence = evidence_documents[index].copy()

        evidence["retrieval_rank"] = rank

        evidence["retrieval_similarity"] = float(
            similarity
        )

        retrieved.append(evidence)

    return retrieved


print("=" * 80)
print("IMPROVED EVIDENCE RETRIEVAL FUNCTION")
print("=" * 80)

print("\nDefault top-K: 10")
print("Retrieval rank: preserved")
print("Similarity: preserved")

print(
    "\nImportant:",
    "Similarity is used for retrieval, not verification."
)

print("=" * 80)

IMPROVED EVIDENCE RETRIEVAL FUNCTION

Default top-K: 10
Retrieval rank: preserved
Similarity: preserved

Important: Similarity is used for retrieval, not verification.


In [ ]:
# ============================================================
# RETRIEVE EVIDENCE FOR ALL MoM CLAIMS
# Purpose:
# Retrieve the top-10 semantically relevant transcript
# evidence units for every structured MoM claim.
#
# These are candidate evidence units only.
# They have NOT yet been verified.
# ============================================================

all_claim_retrievals = []

print("=" * 100)
print("EVIDENCE RETRIEVAL FOR ALL MoM CLAIMS")
print("=" * 100)

for claim in mom_claims:

    retrieved = retrieve_evidence(
        claim["claim_text"],
        top_k=10
    )

    claim_result = {
        "claim_id": claim["claim_id"],
        "topic": claim["topic"],
        "claim_text": claim["claim_text"],
        "claim_speaker": claim["speaker"],
        "claim_start": claim["start"],
        "claim_end": claim["end"],
        "event_type": claim["event_type"],
        "source_candidate_ids": claim[
            "source_candidate_ids"
        ],
        "retrieved_evidence": retrieved
    }

    all_claim_retrievals.append(claim_result)


print("\nTotal claims processed:",
      len(all_claim_retrievals))

print(
    "Expected claims:",
    len(mom_claims)
)

print(
    "\nRetrieval completed successfully:",
    len(all_claim_retrievals) == len(mom_claims)
)

print("\n" + "=" * 100)

EVIDENCE RETRIEVAL FOR ALL MoM CLAIMS

Total claims processed: 12
Expected claims: 12

Retrieval completed successfully: True



In [ ]:
# ============================================================
# COMPACT RETRIEVAL INSPECTION
# Purpose:
# Inspect the top-3 retrieved evidence candidates for each
# MoM claim before moving to evidence verification.
#
# This is a qualitative retrieval checkpoint.
# ============================================================

print("=" * 110)
print("TOP-3 RETRIEVED EVIDENCE FOR EACH MoM CLAIM")
print("=" * 110)

for result in all_claim_retrievals:

    print(
        f"\nCLAIM {result['claim_id']:02d} | "
        f"{result['topic']}"
    )

    print("Claim:")
    print(" ", result["claim_text"])

    print("\nTop retrieved evidence:")

    for evidence in result["retrieved_evidence"][:3]:

        print(
            f"  Rank {evidence['retrieval_rank']} | "
            f"ID {evidence['evidence_id']:03d} | "
            f"Sim {evidence['retrieval_similarity']:.4f} | "
            f"{evidence['speaker']} | "
            f"{evidence['start']:.1f}s–"
            f"{evidence['end']:.1f}s"
        )

        print(
            f"    {evidence['text']}"
        )

    print("-" * 110)

print("\n" + "=" * 110)
print("RETRIEVAL INSPECTION COMPLETE")
print("=" * 110)

TOP-3 RETRIEVED EVIDENCE FOR EACH MoM CLAIM

CLAIM 01 | meeting_introduction_and_agenda
Claim:
  The meeting was introduced as the first meeting, with an agenda covering team interaction, tool training, the project plan, and discussion of ideas.

Top retrieved evidence:
  Rank 1 | ID 009 | Sim 0.7391 | SPEAKER_02 | 84.7s–93.1s
    I'm Sarah, the project manager, and this is our first meeting, surprisingly enough. Okay, this is our agenda.
  Rank 2 | ID 012 | Sim 0.6854 | SPEAKER_02 | 106.3s–109.8s
    talk about the project plan, discuss our own ideas and everything.
  Rank 3 | ID 019 | Sim 0.6029 | SPEAKER_02 | 162.4s–167.7s
    what we're thinking, how it's going to go, and then the detailed design, how we're actually going to put it into practice and make it work.
--------------------------------------------------------------------------------------------------------------

CLAIM 02 | product_objective_and_requirements
Claim:
  The team is developing a remote control intended to be 

In [ ]:
# ============================================================
# LOAD NLI MODEL
# Purpose:
# Load a Natural Language Inference model to determine whether
# retrieved transcript evidence supports a generated MoM claim.
#
# NLI is used for CONTENT CONSISTENCY.
#
# BGE similarity is NOT treated as proof of entailment.
# ============================================================

from transformers import AutoTokenizer, AutoModelForSequenceClassification

NLI_MODEL_NAME = "cross-encoder/nli-deberta-v3-small"

print("=" * 80)
print("LOADING NLI MODEL")
print("=" * 80)

print("\nModel:", NLI_MODEL_NAME)
print("Device:", DEVICE)

nli_tokenizer = AutoTokenizer.from_pretrained(
    NLI_MODEL_NAME
)

nli_model = AutoModelForSequenceClassification.from_pretrained(
    NLI_MODEL_NAME
)

nli_model = nli_model.to(DEVICE)
nli_model.eval()

print("\nNLI model loaded successfully.")

print("=" * 80)

LOADING NLI MODEL

Model: cross-encoder/nli-deberta-v3-small
Device: cuda


config.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.35k [00:00<?, ?B/s]

spm.model: reconstructing file:   0%|          |  0.00B / 2.46MB            

spm.model: downloading bytes:           |  0.00B            

...

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  568MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/106 [00:00<?, ?it/s]


NLI model loaded successfully.


In [ ]:
# ============================================================
# INSPECT NLI LABEL MAPPING
# Purpose:
# Check how the NLI model maps its output indices to
# ENTAILMENT, NEUTRAL, and CONTRADICTION.
#
# This is important because different NLI models can use
# different label-index orders.
# ============================================================

print("=" * 80)
print("NLI LABEL MAPPING")
print("=" * 80)

print("\nModel label mapping:")

for label_id, label_name in nli_model.config.id2label.items():
    print(f"  {label_id} -> {label_name}")

print("\nNumber of labels:", nli_model.config.num_labels)

print("=" * 80)

NLI LABEL MAPPING

Model label mapping:
  0 -> contradiction
  1 -> entailment
  2 -> neutral

Number of labels: 3


In [ ]:
# ============================================================
# NLI SANITY TEST
# Purpose:
# Verify that the NLI model correctly distinguishes:
#   1. Supporting evidence  -> ENTAILMENT
#   2. Unrelated evidence   -> NEUTRAL
#   3. Conflicting evidence -> CONTRADICTION
#
# IMPORTANT:
# Premise    = transcript evidence
# Hypothesis = generated MoM claim
# ============================================================

import torch

def nli_predict(premise, hypothesis):
    """
    Run NLI with:
        premise    = evidence
        hypothesis = MoM claim
    """

    inputs = nli_tokenizer(
        premise,
        hypothesis,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    inputs = {
        key: value.to(DEVICE)
        for key, value in inputs.items()
    }

    with torch.no_grad():
        outputs = nli_model(**inputs)

    probabilities = torch.softmax(outputs.logits, dim=-1)[0]

    label_id = torch.argmax(probabilities).item()
    label = nli_model.config.id2label[label_id].upper()
    confidence = probabilities[label_id].item()

    return label_id, label, confidence


# ------------------------------------------------------------
# Test 1: Supporting evidence
# ------------------------------------------------------------

evidence_1 = (
    "It's got to be accessible and usable by all age groups."
)

claim_1 = (
    "The product should be accessible and usable by all age groups."
)


# ------------------------------------------------------------
# Test 2: Unrelated evidence
# ------------------------------------------------------------

evidence_2 = (
    "We discussed the design process and the detailed design stage."
)

claim_2 = (
    "The product should be accessible and usable by all age groups."
)


# ------------------------------------------------------------
# Test 3: Contradicting evidence
# ------------------------------------------------------------

evidence_3 = (
    "The product is intended only for young adults and is not "
    "designed for older users."
)

claim_3 = (
    "The product should be accessible and usable by all age groups."
)


# ------------------------------------------------------------
# Run tests
# ------------------------------------------------------------

tests = [
    ("SUPPORTING", evidence_1, claim_1),
    ("UNRELATED", evidence_2, claim_2),
    ("CONTRADICTING", evidence_3, claim_3),
]

print("=" * 80)
print("NLI SANITY TEST RESULTS")
print("=" * 80)

for test_name, evidence, claim in tests:

    label_id, label, confidence = nli_predict(
        evidence,
        claim
    )

    print(f"\nTest: {test_name}")
    print("-" * 80)

    print("Evidence:")
    print(evidence)

    print("\nClaim:")
    print(claim)

    print("\nPrediction:")
    print(f"  Label ID   : {label_id}")
    print(f"  Label      : {label}")
    print(f"  Confidence : {confidence:.4f}")

print("\n" + "=" * 80)

NLI SANITY TEST RESULTS

Test: SUPPORTING
--------------------------------------------------------------------------------
Evidence:
It's got to be accessible and usable by all age groups.

Claim:
The product should be accessible and usable by all age groups.

Prediction:
  Label ID   : 2
  Label      : NEUTRAL
  Confidence : 0.9719

Test: UNRELATED
--------------------------------------------------------------------------------
Evidence:
We discussed the design process and the detailed design stage.

Claim:
The product should be accessible and usable by all age groups.

Prediction:
  Label ID   : 2
  Label      : NEUTRAL
  Confidence : 0.9996

Test: CONTRADICTING
--------------------------------------------------------------------------------
Evidence:
The product is intended only for young adults and is not designed for older users.

Claim:
The product should be accessible and usable by all age groups.

Prediction:
  Label ID   : 0
  Label      : CONTRADICTION
  Confidence : 0.9987



In [ ]:
# ============================================================
# REAL PROJECT NLI TEST
# Purpose:
# Test NLI using an actual transcript evidence segment and
# its corresponding structured MoM claim.
#
# This is more meaningful than a manually created example
# because it reflects the actual language produced by the
# meeting transcript and our MoM claim-generation stage.
# ============================================================

# ------------------------------------------------------------
# Select Claim 2
# ------------------------------------------------------------

claim_data = mom_claims_data["claims"][1]

claim_text = claim_data["claim"]

print("=" * 80)
print("REAL PROJECT NLI TEST")
print("=" * 80)

print("\nClaim ID:")
print(claim_data["claim_id"])

print("\nMoM Claim:")
print(claim_text)


# ------------------------------------------------------------
# Retrieve the evidence segments already associated with
# this claim during structured claim construction.
# ------------------------------------------------------------

source_ids = claim_data["source_candidate_ids"]

print("\nSource Candidate IDs:")
print(source_ids)


# ------------------------------------------------------------
# Find the corresponding evidence/utterance text
# ------------------------------------------------------------

print("\n" + "-" * 80)
print("SOURCE EVIDENCE")
print("-" * 80)

for source_id in source_ids:

    # Candidate IDs are stored as integers
    candidate = next(
        (
            item
            for item in mom_candidates_data["candidates"]
            if item["candidate_id"] == source_id
        ),
        None
    )

    if candidate is None:
        print(f"\nCandidate {source_id}: NOT FOUND")
        continue

    evidence_text = candidate["text"]

    print(f"\nCandidate {source_id}")
    print("Evidence:")
    print(evidence_text)

    # --------------------------------------------------------
    # Run NLI
    # --------------------------------------------------------

    label_id, label, confidence = nli_predict(
        evidence_text,
        claim_text
    )

    print("\nNLI Prediction:")
    print(f"  Label ID   : {label_id}")
    print(f"  Label      : {label}")
    print(f"  Confidence : {confidence:.4f}")


print("\n" + "=" * 80)

KeyError: 'claim'

In [ ]:
# ============================================================
# INSPECT STRUCTURED MOM CLAIM FORMAT
# Purpose:
# Check the exact field names used in the structured MoM
# claims JSON before building the NLI verification code.
# ============================================================

print("=" * 80)
print("STRUCTURED MOM CLAIM FORMAT")
print("=" * 80)

print("\nNumber of claims:", len(mom_claims_data["claims"]))

print("\nKeys in first claim:")
print(mom_claims_data["claims"][0].keys())

print("\nFirst claim:")
print(mom_claims_data["claims"][0])

print("\n" + "=" * 80)

STRUCTURED MOM CLAIM FORMAT

Number of claims: 12

Keys in first claim:
dict_keys(['claim_id', 'topic', 'claim_text', 'source_candidate_ids', 'speaker', 'start', 'end', 'event_type'])

First claim:
{'claim_id': 1, 'topic': 'meeting_introduction_and_agenda', 'claim_text': 'The meeting was introduced as the first meeting, with an agenda covering team interaction, tool training, the project plan, and discussion of ideas.', 'source_candidate_ids': [5, 6, 7, 8], 'speaker': 'SPEAKER_02', 'start': 84.658, 'end': 109.823, 'event_type': 'INFORMATION'}



In [ ]:
# ============================================================
# REAL PROJECT NLI TEST
# Purpose:
# Test NLI using an actual structured MoM claim and the
# transcript evidence used to construct that claim.
#
# Premise    = Evidence
# Hypothesis = MoM claim
# ============================================================

print("=" * 80)
print("REAL PROJECT NLI TEST")
print("=" * 80)

# ------------------------------------------------------------
# Select Claim 2
# ------------------------------------------------------------

claim_data = mom_claims_data["claims"][1]

claim_text = claim_data["claim_text"]

print("\nClaim ID:")
print(claim_data["claim_id"])

print("\nTopic:")
print(claim_data["topic"])

print("\nMoM Claim:")
print(claim_text)

print("\nSource Candidate IDs:")
print(claim_data["source_candidate_ids"])


# ------------------------------------------------------------
# Find and test each source evidence
# ------------------------------------------------------------

print("\n" + "-" * 80)
print("SOURCE EVIDENCE + NLI RESULTS")
print("-" * 80)

for source_id in claim_data["source_candidate_ids"]:

    candidate = next(
        (
            item
            for item in mom_candidates_data["candidates"]
            if item["candidate_id"] == source_id
        ),
        None
    )

    if candidate is None:
        print(f"\nCandidate {source_id}: NOT FOUND")
        continue

    evidence_text = candidate["text"]

    print(f"\nCandidate {source_id}")
    print("Evidence:")
    print(evidence_text)

    # --------------------------------------------------------
    # Run NLI
    # --------------------------------------------------------

    label_id, label, confidence = nli_predict(
        evidence_text,
        claim_text
    )

    print("\nNLI Prediction:")
    print(f"  Label ID   : {label_id}")
    print(f"  Label      : {label}")
    print(f"  Confidence : {confidence:.4f}")


print("\n" + "=" * 80)

REAL PROJECT NLI TEST

Claim ID:
2

Topic:
product_objective_and_requirements

MoM Claim:
The team is developing a remote control intended to be original, trendy, appealing to a wide market, and user-friendly for a broad range of users.

Source Candidate IDs:
[10, 11, 12, 13]

--------------------------------------------------------------------------------
SOURCE EVIDENCE + NLI RESULTS
--------------------------------------------------------------------------------


NameError: name 'mom_candidates_data' is not defined

In [ ]:
# ============================================================
# LOAD RULE-BASED MOM CANDIDATES
# Purpose:
# Load the candidate utterances used as source evidence for
# the structured MoM claims.
# ============================================================

import json

print("=" * 80)
print("LOADING MOM CANDIDATES")
print("=" * 80)

with open(MOM_CANDIDATES_PATH, "r", encoding="utf-8") as f:
    mom_candidates_data = json.load(f)

print("\nFile loaded successfully:")
print(MOM_CANDIDATES_PATH)

print("\nNumber of candidates:",
      len(mom_candidates_data["candidates"]))

print("\nAvailable fields:")
print(mom_candidates_data["candidates"][0].keys())

print("=" * 80)

LOADING MOM CANDIDATES

File loaded successfully:
/content/drive/MyDrive/MTechIndProj/MoM_Project/data/transcripts/ES2004a_mom_candidates_rule_based.json

Number of candidates: 50

Available fields:
dict_keys(['candidate_id', 'source_utterance_id', 'speaker', 'start', 'end', 'text'])


In [ ]:
# ============================================================
# REAL PROJECT NLI TEST — CLAIM 2
# Purpose:
# Test whether the actual transcript evidence supports the
# structured MoM claim.
#
# Premise    = Transcript evidence
# Hypothesis = MoM claim
# ============================================================

print("=" * 80)
print("REAL PROJECT NLI TEST — CLAIM 2")
print("=" * 80)

# ------------------------------------------------------------
# Select Claim 2
# ------------------------------------------------------------

claim_data = mom_claims_data["claims"][1]

claim_text = claim_data["claim_text"]

print("\nClaim ID:")
print(claim_data["claim_id"])

print("\nTopic:")
print(claim_data["topic"])

print("\nMoM Claim:")
print(claim_text)

print("\nSource Candidate IDs:")
print(claim_data["source_candidate_ids"])


# ------------------------------------------------------------
# Create candidate lookup
# ------------------------------------------------------------

candidate_lookup = {
    item["candidate_id"]: item
    for item in mom_candidates_data["candidates"]
}


# ------------------------------------------------------------
# Test each source candidate
# ------------------------------------------------------------

print("\n" + "-" * 80)
print("SOURCE EVIDENCE + NLI RESULTS")
print("-" * 80)

for source_id in claim_data["source_candidate_ids"]:

    candidate = candidate_lookup.get(source_id)

    if candidate is None:
        print(f"\nCandidate {source_id}: NOT FOUND")
        continue

    evidence_text = candidate["text"]

    print(f"\nCandidate {source_id}")
    print(
        f"Speaker   : {candidate['speaker']}"
    )
    print(
        f"Timestamp : {candidate['start']:.3f} - "
        f"{candidate['end']:.3f}"
    )

    print("\nEvidence:")
    print(evidence_text)

    # --------------------------------------------------------
    # Run NLI
    # --------------------------------------------------------

    label_id, label, confidence = nli_predict(
        evidence_text,
        claim_text
    )

    print("\nNLI Prediction:")
    print(f"  Label ID   : {label_id}")
    print(f"  Label      : {label}")
    print(f"  Confidence : {confidence:.4f}")


print("\n" + "=" * 80)

REAL PROJECT NLI TEST — CLAIM 2

Claim ID:
2

Topic:
product_objective_and_requirements

MoM Claim:
The team is developing a remote control intended to be original, trendy, appealing to a wide market, and user-friendly for a broad range of users.

Source Candidate IDs:
[10, 11, 12, 13]

--------------------------------------------------------------------------------
SOURCE EVIDENCE + NLI RESULTS
--------------------------------------------------------------------------------

Candidate 10
Speaker   : SPEAKER_02
Timestamp : 117.699 - 120.921

Evidence:
Now, we're developing a remote control, which you probably already know.

NLI Prediction:
  Label ID   : 2
  Label      : NEUTRAL
  Confidence : 0.9900

Candidate 11
Speaker   : SPEAKER_02
Timestamp : 122.503 - 132.290

Evidence:
We want it to be original, something that people haven't thought of. It's not out in the shops. Trendy, appealing to a wide market, but,

NLI Prediction:
  Label ID   : 2
  Label      : NEUTRAL
  Confidence : 0.9

In [ ]:
# ============================================================
# SINGLE-PROPOSITION NLI TEST
# Purpose:
# Test NLI on a claim containing only ONE proposition.
#
# This helps determine whether the NLI model can correctly
# verify individual attributes before we build the
# multi-attribute verification stage.
# ============================================================

evidence = (
    "Now, we're developing a remote control, "
    "which you probably already know."
)

claim = (
    "The team is developing a remote control."
)

label_id, label, confidence = nli_predict(
    evidence,
    claim
)

print("=" * 80)
print("SINGLE-PROPOSITION NLI TEST")
print("=" * 80)

print("\nEvidence:")
print(evidence)

print("\nClaim:")
print(claim)

print("\nPrediction:")
print(f"  Label ID   : {label_id}")
print(f"  Label      : {label}")
print(f"  Confidence : {confidence:.4f}")

print("\n" + "=" * 80)

SINGLE-PROPOSITION NLI TEST

Evidence:
Now, we're developing a remote control, which you probably already know.

Claim:
The team is developing a remote control.

Prediction:
  Label ID   : 1
  Label      : ENTAILMENT
  Confidence : 0.7902



In [ ]:
# ============================================================
# REUSABLE NLI PREDICTION FUNCTION
# Purpose:
# Run NLI between one evidence segment and one proposition.
#
# Output:
#   - contradiction probability
#   - entailment probability
#   - neutral probability
#   - predicted label
#   - confidence
#
# Premise    = Evidence
# Hypothesis = Proposition
# ============================================================

def run_nli(evidence, proposition):
    """
    Perform NLI between transcript evidence and a proposition.

    Parameters
    ----------
    evidence : str
        Transcript evidence.

    proposition : str
        Single factual proposition derived from a MoM claim.

    Returns
    -------
    dict
        NLI probabilities and prediction.
    """

    inputs = nli_tokenizer(
        evidence,
        proposition,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    inputs = {
        key: value.to(DEVICE)
        for key, value in inputs.items()
    }

    with torch.no_grad():
        outputs = nli_model(**inputs)

    probabilities = torch.softmax(
        outputs.logits,
        dim=-1
    )[0]

    # --------------------------------------------------------
    # Model mapping confirmed earlier:
    # 0 = contradiction
    # 1 = entailment
    # 2 = neutral
    # --------------------------------------------------------

    contradiction_prob = probabilities[0].item()
    entailment_prob = probabilities[1].item()
    neutral_prob = probabilities[2].item()

    predicted_id = torch.argmax(probabilities).item()

    predicted_label = (
        nli_model.config.id2label[predicted_id].upper()
    )

    confidence = probabilities[predicted_id].item()

    return {
        "predicted_label": predicted_label,
        "confidence": confidence,
        "contradiction_probability": contradiction_prob,
        "entailment_probability": entailment_prob,
        "neutral_probability": neutral_prob
    }


print("=" * 80)
print("NLI FUNCTION CREATED")
print("=" * 80)

print("\nFunction: run_nli(evidence, proposition)")
print("\nModel mapping:")
print("  0 -> CONTRADICTION")
print("  1 -> ENTAILMENT")
print("  2 -> NEUTRAL")

print("\nReady for attribute-level verification.")

print("=" * 80)

NLI FUNCTION CREATED

Function: run_nli(evidence, proposition)

Model mapping:
  0 -> CONTRADICTION
  1 -> ENTAILMENT
  2 -> NEUTRAL

Ready for attribute-level verification.


In [ ]:
# ============================================================
# TEST REAL PROPOSITION-LEVEL VERIFICATION
# Purpose:
# Verify one individual proposition from Claim 2 against
# its actual transcript evidence.
# ============================================================

evidence = candidate_lookup[10]["text"]

proposition = "The team is developing a remote control."

result = run_nli(
    evidence,
    proposition
)

print("=" * 80)
print("REAL PROPOSITION NLI TEST")
print("=" * 80)

print("\nEvidence:")
print(evidence)

print("\nProposition:")
print(proposition)

print("\nNLI Result:")
print(f"  Predicted Label       : {result['predicted_label']}")
print(f"  Confidence            : {result['confidence']:.4f}")
print(f"  Entailment Probability: {result['entailment_probability']:.4f}")
print(f"  Neutral Probability   : {result['neutral_probability']:.4f}")
print(f"  Contradiction Prob.   : {result['contradiction_probability']:.4f}")

print("\n" + "=" * 80)

REAL PROPOSITION NLI TEST

Evidence:
Now, we're developing a remote control, which you probably already know.

Proposition:
The team is developing a remote control.

NLI Result:
  Predicted Label       : ENTAILMENT
  Confidence            : 0.7902
  Entailment Probability: 0.7902
  Neutral Probability   : 0.2093
  Contradiction Prob.   : 0.0005



In [ ]:
# ============================================================
# CLAIM 2 — ATTRIBUTE / PROPOSITION REPRESENTATION
# Purpose:
# Break the multi-part MoM claim into individual factual
# propositions for evidence verification.
#
# This is the core idea behind the proposed
# Multi-Attribute Evidence Consistency approach.
# ============================================================

claim_2_attributes = [
    {
        "attribute_id": 1,
        "attribute": "product_development",
        "proposition": "The team is developing a remote control.",
        "source_candidate_ids": [10]
    },
    {
        "attribute_id": 2,
        "attribute": "originality",
        "proposition": "The remote control is intended to be original.",
        "source_candidate_ids": [11]
    },
    {
        "attribute_id": 3,
        "attribute": "trendiness",
        "proposition": "The remote control is intended to be trendy.",
        "source_candidate_ids": [11]
    },
    {
        "attribute_id": 4,
        "attribute": "market_appeal",
        "proposition": "The remote control is intended to appeal to a wide market.",
        "source_candidate_ids": [11]
    },
    {
        "attribute_id": 5,
        "attribute": "user_friendliness",
        "proposition": "The remote control is intended to be user-friendly.",
        "source_candidate_ids": [12]
    },
    {
        "attribute_id": 6,
        "attribute": "broad_usability",
        "proposition": "The remote control should be usable by a broad range of users.",
        "source_candidate_ids": [12, 13]
    }
]


print("=" * 80)
print("CLAIM 2 — ATTRIBUTE REPRESENTATION")
print("=" * 80)

print("\nNumber of attributes:", len(claim_2_attributes))

for item in claim_2_attributes:

    print("\nAttribute ID:", item["attribute_id"])
    print("Attribute   :", item["attribute"])
    print("Proposition :", item["proposition"])
    print("Evidence IDs:", item["source_candidate_ids"])

print("\n" + "=" * 80)

CLAIM 2 — ATTRIBUTE REPRESENTATION

Number of attributes: 6

Attribute ID: 1
Attribute   : product_development
Proposition : The team is developing a remote control.
Evidence IDs: [10]

Attribute ID: 2
Attribute   : originality
Proposition : The remote control is intended to be original.
Evidence IDs: [11]

Attribute ID: 3
Attribute   : trendiness
Proposition : The remote control is intended to be trendy.
Evidence IDs: [11]

Attribute ID: 4
Attribute   : market_appeal
Proposition : The remote control is intended to appeal to a wide market.
Evidence IDs: [11]

Attribute ID: 5
Attribute   : user_friendliness
Proposition : The remote control is intended to be user-friendly.
Evidence IDs: [12]

Attribute ID: 6
Attribute   : broad_usability
Proposition : The remote control should be usable by a broad range of users.
Evidence IDs: [12, 13]



In [ ]:
# ============================================================
# CLAIM 2 — ATTRIBUTE-LEVEL NLI VERIFICATION
# Purpose:
# Run NLI separately for each factual proposition in Claim 2.
#
# This demonstrates the core verification mechanism:
#
#   MoM Claim
#       ↓
#   Individual propositions
#       ↓
#   Evidence
#       ↓
#   NLI
# ============================================================

print("=" * 80)
print("CLAIM 2 — ATTRIBUTE-LEVEL NLI RESULTS")
print("=" * 80)

claim2_nli_results = []

for item in claim_2_attributes:

    attribute_id = item["attribute_id"]
    attribute = item["attribute"]
    proposition = item["proposition"]
    source_ids = item["source_candidate_ids"]

    print("\n" + "-" * 80)
    print(f"Attribute {attribute_id}: {attribute}")
    print("-" * 80)

    print("\nProposition:")
    print(proposition)

    best_result = None

    # --------------------------------------------------------
    # Test all designated evidence for this attribute
    # --------------------------------------------------------

    for source_id in source_ids:

        candidate = candidate_lookup.get(source_id)

        if candidate is None:
            print(f"\nCandidate {source_id}: NOT FOUND")
            continue

        evidence = candidate["text"]

        result = run_nli(
            evidence,
            proposition
        )

        print(f"\nEvidence Candidate: {source_id}")
        print(f"Speaker   : {candidate['speaker']}")
        print(
            f"Timestamp : {candidate['start']:.3f} - "
            f"{candidate['end']:.3f}"
        )
        print(f"Evidence  : {evidence}")

        print("\nNLI:")
        print(f"  Label      : {result['predicted_label']}")
        print(f"  Confidence : {result['confidence']:.4f}")
        print(
            f"  Entailment : "
            f"{result['entailment_probability']:.4f}"
        )

        # ----------------------------------------------------
        # Select the evidence with the highest entailment
        # probability.
        # ----------------------------------------------------

        if (
            best_result is None
            or result["entailment_probability"]
            > best_result["entailment_probability"]
        ):
            best_result = {
                "candidate_id": source_id,
                **result
            }

    # --------------------------------------------------------
    # Store best evidence for this attribute
    # --------------------------------------------------------

    if best_result is not None:

        claim2_nli_results.append({
            "attribute_id": attribute_id,
            "attribute": attribute,
            "proposition": proposition,
            "best_evidence_candidate_id":
                best_result["candidate_id"],
            "predicted_label":
                best_result["predicted_label"],
            "confidence":
                best_result["confidence"],
            "entailment_probability":
                best_result["entailment_probability"],
            "neutral_probability":
                best_result["neutral_probability"],
            "contradiction_probability":
                best_result["contradiction_probability"]
        })


print("\n" + "=" * 80)
print("CLAIM 2 ATTRIBUTE-LEVEL NLI COMPLETE")
print("=" * 80)

print("\nAttributes evaluated:",
      len(claim2_nli_results))

print("\nSummary:")

for result in claim2_nli_results:

    print(
        f"\nAttribute {result['attribute_id']} "
        f"({result['attribute']}):"
    )

    print(
        f"  Best Evidence : "
        f"{result['best_evidence_candidate_id']}"
    )

    print(
        f"  Label         : "
        f"{result['predicted_label']}"
    )

    print(
        f"  Entailment    : "
        f"{result['entailment_probability']:.4f}"
    )

print("\n" + "=" * 80)

CLAIM 2 — ATTRIBUTE-LEVEL NLI RESULTS

--------------------------------------------------------------------------------
Attribute 1: product_development
--------------------------------------------------------------------------------

Proposition:
The team is developing a remote control.

Evidence Candidate: 10
Speaker   : SPEAKER_02
Timestamp : 117.699 - 120.921
Evidence  : Now, we're developing a remote control, which you probably already know.

NLI:
  Label      : ENTAILMENT
  Confidence : 0.7902
  Entailment : 0.7902

--------------------------------------------------------------------------------
Attribute 2: originality
--------------------------------------------------------------------------------

Proposition:
The remote control is intended to be original.

Evidence Candidate: 11
Speaker   : SPEAKER_02
Timestamp : 122.503 - 132.290
Evidence  : We want it to be original, something that people haven't thought of. It's not out in the shops. Trendy, appealing to a wide market, but

In [ ]:
# ============================================================
# CONTEXT-EXPANDED NLI TEST
# Purpose:
# Test whether adding nearby transcript context improves
# NLI recognition of conversational/fragmented evidence.
#
# We test the same four attributes from Claim 2 that were
# incorrectly classified as NEUTRAL.
# ============================================================

# ------------------------------------------------------------
# Helper function:
# Get nearby candidates around a source candidate
# ------------------------------------------------------------

def get_context_text(candidate_id, window=1):

    # Sort candidates by candidate ID
    sorted_candidates = sorted(
        mom_candidates_data["candidates"],
        key=lambda x: x["candidate_id"]
    )

    # Find position of target candidate
    target_index = next(
        (
            i
            for i, item in enumerate(sorted_candidates)
            if item["candidate_id"] == candidate_id
        ),
        None
    )

    if target_index is None:
        return None

    start_index = max(0, target_index - window)
    end_index = min(
        len(sorted_candidates),
        target_index + window + 1
    )

    context_items = sorted_candidates[start_index:end_index]

    context_text = " ".join(
        item["text"]
        for item in context_items
    )

    return context_text


# ------------------------------------------------------------
# Test attributes
# ------------------------------------------------------------

context_tests = [
    (
        "originality",
        "The remote control is intended to be original.",
        11
    ),
    (
        "trendiness",
        "The remote control is intended to be trendy.",
        11
    ),
    (
        "market_appeal",
        "The remote control is intended to appeal to a wide market.",
        11
    ),
    (
        "user_friendliness",
        "The remote control is intended to be user-friendly.",
        12
    )
]


print("=" * 80)
print("CONTEXT-EXPANDED NLI TEST")
print("=" * 80)

for attribute, proposition, candidate_id in context_tests:

    context = get_context_text(
        candidate_id,
        window=1
    )

    result = run_nli(
        context,
        proposition
    )

    print("\n" + "-" * 80)
    print("Attribute:", attribute)
    print("Evidence candidate:", candidate_id)

    print("\nExpanded Evidence:")
    print(context)

    print("\nProposition:")
    print(proposition)

    print("\nNLI Result:")
    print(
        f"  Label       : {result['predicted_label']}"
    )
    print(
        f"  Entailment  : "
        f"{result['entailment_probability']:.4f}"
    )
    print(
        f"  Neutral     : "
        f"{result['neutral_probability']:.4f}"
    )
    print(
        f"  Contradiction: "
        f"{result['contradiction_probability']:.4f}"
    )

print("\n" + "=" * 80)

CONTEXT-EXPANDED NLI TEST

--------------------------------------------------------------------------------
Attribute: originality
Evidence candidate: 11

Expanded Evidence:
Now, we're developing a remote control, which you probably already know. We want it to be original, something that people haven't thought of. It's not out in the shops. Trendy, appealing to a wide market, but, you know, not a hunk of metal. And user-friendly, grannies to kids,

Proposition:
The remote control is intended to be original.

NLI Result:
  Label       : ENTAILMENT
  Entailment  : 0.9865
  Neutral     : 0.0124
  Contradiction: 0.0011

--------------------------------------------------------------------------------
Attribute: trendiness
Evidence candidate: 11

Expanded Evidence:
Now, we're developing a remote control, which you probably already know. We want it to be original, something that people haven't thought of. It's not out in the shops. Trendy, appealing to a wide market, but, you know, not a hunk

In [ ]:
# ============================================================
# M7 — EXPLICIT EVIDENCE MATCHING TEST
# Purpose:
# Test whether important words/phrases from a proposition
# are explicitly present in the retrieved transcript evidence.
#
# This is NOT used as proof by itself.
# It is a supporting signal for cases where NLI may fail on
# conversational or fragmented meeting transcripts.
# ============================================================

import re


def normalize_text(text):
    """
    Convert text to lowercase and remove punctuation.
    """
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s-]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def explicit_phrase_match(evidence, phrases):
    """
    Check whether important evidence phrases occur explicitly
    in the transcript evidence.

    Returns:
        matched_phrases
        match_ratio
    """

    normalized_evidence = normalize_text(evidence)

    matched = []

    for phrase in phrases:

        normalized_phrase = normalize_text(phrase)

        if normalized_phrase in normalized_evidence:
            matched.append(phrase)

    match_ratio = (
        len(matched) / len(phrases)
        if phrases
        else 0.0
    )

    return matched, match_ratio


# ------------------------------------------------------------
# Test cases
# ------------------------------------------------------------

tests = [
    {
        "name": "User friendliness",
        "evidence": (
            "We want it to be original, something that people "
            "haven't thought of. It's not out in the shops. "
            "Trendy, appealing to a wide market, but, you know, "
            "not a hunk of metal. And user-friendly, grannies "
            "to kids, maybe even pooches, should be able to use it."
        ),
        "phrases": [
            "user-friendly"
        ]
    },
    {
        "name": "Broad usability",
        "evidence": (
            "And user-friendly, grannies to kids, maybe even "
            "pooches, should be able to use it."
        ),
        "phrases": [
            "grannies",
            "kids",
            "should be able to use it"
        ]
    },
    {
        "name": "Unrelated example",
        "evidence": (
            "We discussed the design process and the detailed "
            "design stage."
        ),
        "phrases": [
            "user-friendly",
            "wide market"
        ]
    }
]


print("=" * 80)
print("M7 — EXPLICIT EVIDENCE MATCHING TEST")
print("=" * 80)

for test in tests:

    matched, ratio = explicit_phrase_match(
        test["evidence"],
        test["phrases"]
    )

    print("\n" + "-" * 80)
    print("Test:", test["name"])

    print("\nEvidence:")
    print(test["evidence"])

    print("\nRequired phrases:")
    print(test["phrases"])

    print("\nMatched phrases:")
    print(matched)

    print(f"\nMatch ratio: {ratio:.4f}")

print("\n" + "=" * 80)

M7 — EXPLICIT EVIDENCE MATCHING TEST

--------------------------------------------------------------------------------
Test: User friendliness

Evidence:
We want it to be original, something that people haven't thought of. It's not out in the shops. Trendy, appealing to a wide market, but, you know, not a hunk of metal. And user-friendly, grannies to kids, maybe even pooches, should be able to use it.

Required phrases:
['user-friendly']

Matched phrases:
['user-friendly']

Match ratio: 1.0000

--------------------------------------------------------------------------------
Test: Broad usability

Evidence:
And user-friendly, grannies to kids, maybe even pooches, should be able to use it.

Required phrases:
['grannies', 'kids', 'should be able to use it']

Matched phrases:
['grannies', 'kids', 'should be able to use it']

Match ratio: 1.0000

--------------------------------------------------------------------------------
Test: Unrelated example

Evidence:
We discussed the design proces

In [ ]:
# ============================================================
# M7 — HYBRID EVIDENCE SUPPORT TEST
# Purpose:
# Combine:
#   1. NLI entailment probability
#   2. Explicit evidence phrase matching
#
# This is an experimental verification signal.
# We will NOT finalize thresholds yet.
# ============================================================

def hybrid_evidence_score(
    evidence,
    proposition,
    required_phrases
):
    """
    Calculate a hybrid evidence-support score.

    Components:
        NLI entailment probability
        Explicit phrase match ratio

    The two components are kept separate so that we can
    inspect their behavior before defining the final rule.
    """

    # --------------------------------------------------------
    # NLI
    # --------------------------------------------------------

    nli_result = run_nli(
        evidence,
        proposition
    )

    # --------------------------------------------------------
    # Explicit phrase matching
    # --------------------------------------------------------

    matched_phrases, match_ratio = explicit_phrase_match(
        evidence,
        required_phrases
    )

    # --------------------------------------------------------
    # Experimental combined score
    #
    # 70% NLI + 30% explicit evidence
    #
    # IMPORTANT:
    # This weighting is temporary for experimentation only.
    # We will NOT use it as the final research threshold.
    # --------------------------------------------------------

    hybrid_score = (
        0.70 * nli_result["entailment_probability"]
        +
        0.30 * match_ratio
    )

    return {
        "nli_label": nli_result["predicted_label"],
        "nli_entailment": nli_result["entailment_probability"],
        "nli_neutral": nli_result["neutral_probability"],
        "nli_contradiction": nli_result["contradiction_probability"],
        "matched_phrases": matched_phrases,
        "phrase_match_ratio": match_ratio,
        "hybrid_score": hybrid_score
    }


# ------------------------------------------------------------
# Test cases
# ------------------------------------------------------------

tests = [
    {
        "name": "Strong semantic + explicit evidence",
        "evidence": (
            "Now, we're developing a remote control, "
            "which you probably already know."
        ),
        "proposition": (
            "The team is developing a remote control."
        ),
        "phrases": [
            "developing a remote control"
        ]
    },
    {
        "name": "Explicit evidence but weak NLI",
        "evidence": (
            "We want it to be original, something that people "
            "haven't thought of. It's not out in the shops. "
            "Trendy, appealing to a wide market, but, you know, "
            "not a hunk of metal. And user-friendly, grannies "
            "to kids, maybe even pooches, should be able to use it."
        ),
        "proposition": (
            "The remote control is intended to be user-friendly."
        ),
        "phrases": [
            "user-friendly"
        ]
    },
    {
        "name": "No supporting evidence",
        "evidence": (
            "We discussed the design process and the detailed "
            "design stage."
        ),
        "proposition": (
            "The remote control is intended to be user-friendly."
        ),
        "phrases": [
            "user-friendly"
        ]
    },
    {
        "name": "Potential contradiction",
        "evidence": (
            "The product is intended only for young adults and "
            "is not designed for older users."
        ),
        "proposition": (
            "The product should be usable by all age groups."
        ),
        "phrases": [
            "all age groups"
        ]
    }
]


print("=" * 80)
print("M7 — HYBRID EVIDENCE SUPPORT TEST")
print("=" * 80)

for test in tests:

    result = hybrid_evidence_score(
        test["evidence"],
        test["proposition"],
        test["phrases"]
    )

    print("\n" + "-" * 80)
    print("Test:", test["name"])

    print("\nEvidence:")
    print(test["evidence"])

    print("\nProposition:")
    print(test["proposition"])

    print("\nResults:")
    print(
        f"  NLI Label          : "
        f"{result['nli_label']}"
    )

    print(
        f"  NLI Entailment     : "
        f"{result['nli_entailment']:.4f}"
    )

    print(
        f"  NLI Neutral        : "
        f"{result['nli_neutral']:.4f}"
    )

    print(
        f"  NLI Contradiction  : "
        f"{result['nli_contradiction']:.4f}"
    )

    print(
        f"  Matched Phrases    : "
        f"{result['matched_phrases']}"
    )

    print(
        f"  Phrase Match Ratio : "
        f"{result['phrase_match_ratio']:.4f}"
    )

    print(
        f"  Experimental Score : "
        f"{result['hybrid_score']:.4f}"
    )

print("\n" + "=" * 80)


M7 — HYBRID EVIDENCE SUPPORT TEST


NameError: name 'run_nli' is not defined

In [ ]:
# ============================================================
# M7 — RESTORE NLI FUNCTION
# Purpose:
# Recreate the reusable NLI function in the current runtime.
#
# The NLI model was already loaded and validated earlier.
# No model re-download or reinstallation is required.
# ============================================================

import torch

def run_nli(evidence, proposition):
    """
    Perform NLI between transcript evidence and a proposition.

    Premise    = evidence
    Hypothesis = proposition
    """

    inputs = nli_tokenizer(
        evidence,
        proposition,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    inputs = {
        key: value.to(DEVICE)
        for key, value in inputs.items()
    }

    with torch.no_grad():
        outputs = nli_model(**inputs)

    probabilities = torch.softmax(
        outputs.logits,
        dim=-1
    )[0]

    # Confirmed model mapping:
    # 0 = contradiction
    # 1 = entailment
    # 2 = neutral

    contradiction_prob = probabilities[0].item()
    entailment_prob = probabilities[1].item()
    neutral_prob = probabilities[2].item()

    predicted_id = torch.argmax(probabilities).item()

    predicted_label = (
        nli_model.config.id2label[predicted_id].upper()
    )

    confidence = probabilities[predicted_id].item()

    return {
        "predicted_label": predicted_label,
        "confidence": confidence,
        "contradiction_probability": contradiction_prob,
        "entailment_probability": entailment_prob,
        "neutral_probability": neutral_prob
    }


print("=" * 80)
print("NLI FUNCTION RESTORED")
print("=" * 80)

print("\nFunction: run_nli()")
print("Premise    = Evidence")
print("Hypothesis = Proposition")

print("\nModel mapping:")
print("  0 -> CONTRADICTION")
print("  1 -> ENTAILMENT")
print("  2 -> NEUTRAL")

print("\nReady for M7 hybrid verification.")

print("=" * 80)

NLI FUNCTION RESTORED

Function: run_nli()
Premise    = Evidence
Hypothesis = Proposition

Model mapping:
  0 -> CONTRADICTION
  1 -> ENTAILMENT
  2 -> NEUTRAL

Ready for M7 hybrid verification.


In [ ]:
# ============================================================
# M7 — QUICK HYBRID FUNCTION CHECK
# Purpose:
# Confirm that both NLI and explicit phrase matching work
# together before running the complete test set.
# ============================================================

evidence = (
    "And user-friendly, grannies to kids, "
    "maybe even pooches, should be able to use it."
)

proposition = (
    "The remote control is intended to be user-friendly."
)

required_phrases = [
    "user-friendly"
]

# ------------------------------------------------------------
# NLI
# ------------------------------------------------------------

nli_result = run_nli(
    evidence,
    proposition
)

# ------------------------------------------------------------
# Explicit phrase matching
# ------------------------------------------------------------

matched_phrases, match_ratio = explicit_phrase_match(
    evidence,
    required_phrases
)

# ------------------------------------------------------------
# Display results
# ------------------------------------------------------------

print("=" * 80)
print("M7 — QUICK HYBRID FUNCTION CHECK")
print("=" * 80)

print("\nNLI:")
print(f"  Label       : {nli_result['predicted_label']}")
print(
    f"  Entailment  : "
    f"{nli_result['entailment_probability']:.4f}"
)
print(
    f"  Neutral     : "
    f"{nli_result['neutral_probability']:.4f}"
)
print(
    f"  Contradiction: "
    f"{nli_result['contradiction_probability']:.4f}"
)

print("\nExplicit Evidence:")
print(f"  Matched     : {matched_phrases}")
print(f"  Match Ratio : {match_ratio:.4f}")

print("\n" + "=" * 80)

NameError: name 'nli_tokenizer' is not defined

In [ ]:
# ============================================================
# M7 — RESTORE NLI TOKENIZER
# Purpose:
# Recreate the tokenizer object in the current runtime.
#
# The NLI model is already loaded. No model reinstallation
# is required.
# ============================================================

from transformers import AutoTokenizer

NLI_MODEL_NAME = "cross-encoder/nli-deberta-v3-small"

print("=" * 80)
print("RESTORING NLI TOKENIZER")
print("=" * 80)

nli_tokenizer = AutoTokenizer.from_pretrained(
    NLI_MODEL_NAME
)

print("\nNLI tokenizer restored successfully.")
print("Model:", NLI_MODEL_NAME)

print("=" * 80)

RESTORING NLI TOKENIZER


config.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.35k [00:00<?, ?B/s]

spm.model: reconstructing file:   0%|          |  0.00B / 2.46MB            

spm.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/8.66M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]


NLI tokenizer restored successfully.
Model: cross-encoder/nli-deberta-v3-small


In [ ]:
# ============================================================
# M7 — QUICK HYBRID FUNCTION CHECK
# Purpose:
# Confirm that NLI and explicit phrase matching work together.
# ============================================================

evidence = (
    "And user-friendly, grannies to kids, "
    "maybe even pooches, should be able to use it."
)

proposition = (
    "The remote control is intended to be user-friendly."
)

required_phrases = [
    "user-friendly"
]

# ------------------------------------------------------------
# NLI
# ------------------------------------------------------------

nli_result = run_nli(
    evidence,
    proposition
)

# ------------------------------------------------------------
# Explicit phrase matching
# ------------------------------------------------------------

matched_phrases, match_ratio = explicit_phrase_match(
    evidence,
    required_phrases
)

# ------------------------------------------------------------
# Display results
# ------------------------------------------------------------

print("=" * 80)
print("M7 — QUICK HYBRID FUNCTION CHECK")
print("=" * 80)

print("\nNLI:")
print(f"  Label        : {nli_result['predicted_label']}")
print(
    f"  Entailment   : "
    f"{nli_result['entailment_probability']:.4f}"
)
print(
    f"  Neutral      : "
    f"{nli_result['neutral_probability']:.4f}"
)
print(
    f"  Contradiction: "
    f"{nli_result['contradiction_probability']:.4f}"
)

print("\nExplicit Evidence:")
print(f"  Matched      : {matched_phrases}")
print(f"  Match Ratio  : {match_ratio:.4f}")

print("\n" + "=" * 80)

NameError: name 'DEVICE' is not defined

In [ ]:
# ============================================================
# M7 — RESTORE DEVICE
# Purpose:
# Recreate the DEVICE variable used by the NLI verification
# functions.
#
# Automatically uses CUDA when available; otherwise CPU.
# ============================================================

import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("=" * 80)
print("M7 — DEVICE RESTORED")
print("=" * 80)

print("\nDevice:", DEVICE)

if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("CUDA is not available. Using CPU.")

print("=" * 80)

M7 — DEVICE RESTORED

Device: cpu
CUDA is not available. Using CPU.


M7 initialization/setup cell that:

Mounts Drive
Sets project paths
Loads the structured MoM claims
Loads the candidate data
Loads the NLI model/tokenizer
Sets DEVICE = "cuda"
Recreates the verification functions

In [1]:
# ============================================================
# M7 — RESTORE DEVICE
# Purpose:
# Recreate the DEVICE variable used by the NLI verification
# functions.
#
# Automatically uses CUDA when available; otherwise CPU.
# ============================================================

import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("=" * 80)
print("M7 — DEVICE RESTORED")
print("=" * 80)

print("\nDevice:", DEVICE)

if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("CUDA is not available. Using CPU.")

print("=" * 80)

M7 — DEVICE RESTORED

Device: cuda
GPU: Tesla T4


In [2]:
# ============================================================
# M7 — EVIDENCE VERIFICATION INITIALIZATION
# Purpose:
# Initialize the M7 verification environment after a runtime
# restart.
#
# This cell:
#   1. Mounts Google Drive
#   2. Defines project paths
#   3. Loads MoM claims and candidates
#   4. Sets the computation device
#   5. Loads the NLI model and tokenizer
#
# No earlier project stages are re-run.
# ============================================================

import os
import json
import torch

from google.colab import drive
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

# ------------------------------------------------------------
# 1. Mount Google Drive
# ------------------------------------------------------------

drive.mount("/content/drive")


# ------------------------------------------------------------
# 2. Project paths
# ------------------------------------------------------------

PROJECT_DIR = "/content/drive/MyDrive/MTechIndProj/MoM_Project"

DATA_DIR = os.path.join(
    PROJECT_DIR,
    "data"
)

TRANSCRIPTS_DIR = os.path.join(
    DATA_DIR,
    "transcripts"
)

MOM_CANDIDATES_PATH = os.path.join(
    TRANSCRIPTS_DIR,
    "ES2004a_mom_candidates_rule_based.json"
)

MOM_CLAIMS_PATH = os.path.join(
    TRANSCRIPTS_DIR,
    "ES2004a_structured_mom_claims.json"
)


# ------------------------------------------------------------
# 3. Device
# ------------------------------------------------------------

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


# ------------------------------------------------------------
# 4. Load MoM claims
# ------------------------------------------------------------

with open(
    MOM_CLAIMS_PATH,
    "r",
    encoding="utf-8"
) as f:

    mom_claims_data = json.load(f)


# ------------------------------------------------------------
# 5. Load MoM candidates
# ------------------------------------------------------------

with open(
    MOM_CANDIDATES_PATH,
    "r",
    encoding="utf-8"
) as f:

    mom_candidates_data = json.load(f)


# ------------------------------------------------------------
# 6. Create candidate lookup
# ------------------------------------------------------------

candidate_lookup = {
    item["candidate_id"]: item
    for item in mom_candidates_data["candidates"]
}


# ------------------------------------------------------------
# 7. Load NLI model and tokenizer
# ------------------------------------------------------------

NLI_MODEL_NAME = (
    "cross-encoder/nli-deberta-v3-small"
)

print("=" * 80)
print("M7 — EVIDENCE VERIFICATION INITIALIZATION")
print("=" * 80)

print("\nDevice:", DEVICE)

if DEVICE == "cuda":
    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

print("\nLoading NLI tokenizer...")

nli_tokenizer = AutoTokenizer.from_pretrained(
    NLI_MODEL_NAME
)

print("Tokenizer loaded.")

print("\nLoading NLI model...")

nli_model = AutoModelForSequenceClassification.from_pretrained(
    NLI_MODEL_NAME
)

nli_model = nli_model.to(DEVICE)
nli_model.eval()

print("NLI model loaded.")


# ------------------------------------------------------------
# 8. Verify loaded project data
# ------------------------------------------------------------

print("\nProject data:")
print(
    "  MoM claims     :",
    len(mom_claims_data["claims"])
)
print(
    "  MoM candidates :",
    len(mom_candidates_data["candidates"])
)

print("\nNLI label mapping:")

for label_id, label_name in nli_model.config.id2label.items():
    print(
        f"  {label_id} -> {label_name}"
    )

print("\n" + "=" * 80)
print("M7 INITIALIZATION COMPLETE")
print("=" * 80)

Mounted at /content/drive
M7 — EVIDENCE VERIFICATION INITIALIZATION

Device: cuda
GPU: Tesla T4

Loading NLI tokenizer...


config.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.35k [00:00<?, ?B/s]

spm.model: reconstructing file:   0%|          |  0.00B / 2.46MB            

spm.model: downloading bytes:           |  0.00B            

...

Tokenizer loaded.

Loading NLI model...


model.safetensors: reconstructing file:   0%|          |  0.00B /  568MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/106 [00:00<?, ?it/s]

NLI model loaded.

Project data:
  MoM claims     : 12
  MoM candidates : 50

NLI label mapping:
  0 -> contradiction
  1 -> entailment
  2 -> neutral

M7 INITIALIZATION COMPLETE


In [3]:
# ============================================================
# M7 — NLI VERIFICATION FUNCTION
# Purpose:
# Provide a reusable function for checking whether a transcript
# evidence segment supports a single MoM proposition.
#
# Premise    = Transcript evidence
# Hypothesis = MoM proposition
#
# Model:
# cross-encoder/nli-deberta-v3-small
#
# Label mapping:
#   0 = CONTRADICTION
#   1 = ENTAILMENT
#   2 = NEUTRAL
# ============================================================

def run_nli(evidence, proposition):
    """
    Perform Natural Language Inference between evidence
    and a single factual proposition.
    """

    inputs = nli_tokenizer(
        evidence,
        proposition,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    inputs = {
        key: value.to(DEVICE)
        for key, value in inputs.items()
    }

    with torch.no_grad():
        outputs = nli_model(**inputs)

    probabilities = torch.softmax(
        outputs.logits,
        dim=-1
    )[0]

    # --------------------------------------------------------
    # Extract probabilities
    # --------------------------------------------------------

    contradiction_probability = probabilities[0].item()
    entailment_probability = probabilities[1].item()
    neutral_probability = probabilities[2].item()

    # --------------------------------------------------------
    # Determine predicted label
    # --------------------------------------------------------

    predicted_id = torch.argmax(probabilities).item()

    predicted_label = (
        nli_model.config.id2label[predicted_id].upper()
    )

    confidence = probabilities[predicted_id].item()

    return {
        "predicted_label": predicted_label,
        "confidence": confidence,
        "contradiction_probability":
            contradiction_probability,
        "entailment_probability":
            entailment_probability,
        "neutral_probability":
            neutral_probability
    }


print("=" * 80)
print("M7 — NLI VERIFICATION FUNCTION READY")
print("=" * 80)

print("\nFunction: run_nli()")
print("Premise    = Evidence")
print("Hypothesis = Proposition")

print("\nOutput:")
print("  Predicted label")
print("  Confidence")
print("  Entailment probability")
print("  Neutral probability")
print("  Contradiction probability")

print("\n" + "=" * 80)

M7 — NLI VERIFICATION FUNCTION READY

Function: run_nli()
Premise    = Evidence
Hypothesis = Proposition

Output:
  Predicted label
  Confidence
  Entailment probability
  Neutral probability
  Contradiction probability



In [4]:
# ============================================================
# M7 — NLI SANITY TEST
# Purpose:
# Test NLI on one real transcript evidence segment
# and one simple MoM proposition.
# ============================================================

evidence = (
    "Now, we're developing a remote control, "
    "which you probably already know."
)

proposition = (
    "The team is developing a remote control."
)

result = run_nli(evidence, proposition)

print("=" * 80)
print("M7 — NLI SANITY TEST")
print("=" * 80)

print("\nEvidence:")
print(evidence)

print("\nProposition:")
print(proposition)

print("\nNLI Result:")
for key, value in result.items():
    print(f"  {key}: {value:.4f}" if isinstance(value, float)
          else f"  {key}: {value}")

print("\n" + "=" * 80)

M7 — NLI SANITY TEST

Evidence:
Now, we're developing a remote control, which you probably already know.

Proposition:
The team is developing a remote control.

NLI Result:
  predicted_label: ENTAILMENT
  confidence: 0.7902
  contradiction_probability: 0.0005
  entailment_probability: 0.7902
  neutral_probability: 0.2093



In [5]:
# ============================================================
# M7 — NLI CONTROLLED TEST
# Purpose:
# Test NLI behavior for:
#   1. Supporting evidence
#   2. Contradictory evidence
#   3. Unrelated evidence
#
# This helps validate the NLI component before integrating it
# into the full evidence verification pipeline.
# ============================================================

test_cases = [
    {
        "name": "SUPPORTING",
        "evidence": (
            "Now, we're developing a remote control, "
            "which you probably already know."
        ),
        "proposition": (
            "The team is developing a remote control."
        )
    },
    {
        "name": "CONTRADICTORY",
        "evidence": (
            "We're not developing a remote control. "
            "The project is focused on a different product."
        ),
        "proposition": (
            "The team is developing a remote control."
        )
    },
    {
        "name": "UNRELATED",
        "evidence": (
            "The team discussed the meeting schedule "
            "and the next meeting date."
        ),
        "proposition": (
            "The team is developing a remote control."
        )
    }
]

print("=" * 80)
print("M7 — NLI CONTROLLED TEST")
print("=" * 80)

for case in test_cases:

    result = run_nli(
        case["evidence"],
        case["proposition"]
    )

    print(f"\n[{case['name']}]")

    print("Evidence:")
    print(case["evidence"])

    print("\nProposition:")
    print(case["proposition"])

    print("\nResult:")
    print("  Predicted label :", result["predicted_label"])
    print("  Confidence      :", round(result["confidence"], 4))
    print("  Entailment      :", round(result["entailment_probability"], 4))
    print("  Neutral         :", round(result["neutral_probability"], 4))
    print("  Contradiction   :", round(result["contradiction_probability"], 4))

print("\n" + "=" * 80)

M7 — NLI CONTROLLED TEST

[SUPPORTING]
Evidence:
Now, we're developing a remote control, which you probably already know.

Proposition:
The team is developing a remote control.

Result:
  Predicted label : ENTAILMENT
  Confidence      : 0.7902
  Entailment      : 0.7902
  Neutral         : 0.2093
  Contradiction   : 0.0005

[CONTRADICTORY]
Evidence:
We're not developing a remote control. The project is focused on a different product.

Proposition:
The team is developing a remote control.

Result:
  Predicted label : CONTRADICTION
  Confidence      : 0.9967
  Entailment      : 0.0012
  Neutral         : 0.0021
  Contradiction   : 0.9967

[UNRELATED]
Evidence:
The team discussed the meeting schedule and the next meeting date.

Proposition:
The team is developing a remote control.

Result:
  Predicted label : CONTRADICTION
  Confidence      : 0.9975
  Entailment      : 0.0
  Neutral         : 0.0025
  Contradiction   : 0.9975



In [6]:
# ============================================================
# M7 — REAL MEETING NLI TEST
# Purpose:
# Test NLI using actual meeting evidence and a simple
# proposition derived from that evidence.
# ============================================================

evidence = (
    "We want it to be original. "
    "Trendy, appealing to a wide market."
)

proposition = (
    "The remote control is intended to be original."
)

result = run_nli(
    evidence,
    proposition
)

print("=" * 80)
print("M7 — REAL MEETING NLI TEST")
print("=" * 80)

print("\nEvidence:")
print(evidence)

print("\nProposition:")
print(proposition)

print("\nNLI Result:")
print("  Predicted label :", result["predicted_label"])
print("  Confidence      :", round(result["confidence"], 4))
print("  Entailment      :", round(result["entailment_probability"], 4))
print("  Neutral         :", round(result["neutral_probability"], 4))
print("  Contradiction   :", round(result["contradiction_probability"], 4))

print("\n" + "=" * 80)

M7 — REAL MEETING NLI TEST

Evidence:
We want it to be original. Trendy, appealing to a wide market.

Proposition:
The remote control is intended to be original.

NLI Result:
  Predicted label : NEUTRAL
  Confidence      : 0.9996
  Entailment      : 0.0002
  Neutral         : 0.9996
  Contradiction   : 0.0002



In [7]:
# ============================================================
# M7 — EVIDENCE RETRIEVAL INITIALIZATION
# Purpose:
# Rebuild the speaker-level evidence collection and BGE + FAISS
# retrieval resources after a Colab runtime restart.
#
# Source:
#   ES2004a speaker-level transcript
#
# Retrieval model:
#   BAAI/bge-small-en-v1.5
#
# FAISS:
#   IndexFlatIP on normalized embeddings
#   → inner product approximates cosine similarity
# ============================================================

import os
import json
import numpy as np
import torch

from sentence_transformers import SentenceTransformer
import faiss

# ------------------------------------------------------------
# 1. Paths
# ------------------------------------------------------------

SPEAKER_TRANSCRIPT_PATH = os.path.join(
    TRANSCRIPTS_DIR,
    "ES2004a_speaker_transcript.json"
)

# ------------------------------------------------------------
# 2. Load speaker-level transcript
# ------------------------------------------------------------

with open(SPEAKER_TRANSCRIPT_PATH, "r", encoding="utf-8") as f:
    speaker_transcript_data = json.load(f)

speaker_utterances = speaker_transcript_data["speaker_utterances"]

# ------------------------------------------------------------
# 3. Create evidence documents
# ------------------------------------------------------------

evidence_documents = []

for i, utterance in enumerate(speaker_utterances):

    evidence_documents.append({
        "evidence_id": i + 1,
        "speaker": utterance["speaker"],
        "start": utterance["start"],
        "end": utterance["end"],
        "duration": utterance["end"] - utterance["start"],
        "text": utterance["text"]
    })

# ------------------------------------------------------------
# 4. Load BGE embedding model
# ------------------------------------------------------------

BGE_MODEL_NAME = "BAAI/bge-small-en-v1.5"

print("=" * 80)
print("M7 — EVIDENCE RETRIEVAL INITIALIZATION")
print("=" * 80)

print("\nDevice:", DEVICE)

print("\nLoading BGE model...")
bge_model = SentenceTransformer(
    BGE_MODEL_NAME,
    device=DEVICE
)

print("BGE model loaded.")

# ------------------------------------------------------------
# 5. Generate normalized evidence embeddings
# ------------------------------------------------------------

evidence_texts = [
    item["text"]
    for item in evidence_documents
]

print("\nGenerating evidence embeddings...")

evidence_embeddings = bge_model.encode(
    evidence_texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

evidence_embeddings = evidence_embeddings.astype(
    np.float32
)

# ------------------------------------------------------------
# 6. Build FAISS index
# ------------------------------------------------------------

embedding_dimension = evidence_embeddings.shape[1]

faiss_index = faiss.IndexFlatIP(
    embedding_dimension
)

faiss_index.add(evidence_embeddings)

# ------------------------------------------------------------
# 7. Retrieval function
# ------------------------------------------------------------

def retrieve_evidence(claim_text, top_k=10):

    query_embedding = bge_model.encode(
        [claim_text],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype(np.float32)

    similarities, indices = faiss_index.search(
        query_embedding,
        top_k
    )

    results = []

    for rank, (index, similarity) in enumerate(
        zip(indices[0], similarities[0]),
        start=1
    ):

        evidence = evidence_documents[index]

        results.append({
            "retrieval_rank": rank,
            "retrieval_similarity": float(similarity),
            **evidence
        })

    return results

# ------------------------------------------------------------
# 8. Summary
# ------------------------------------------------------------

print("\nEvidence documents :", len(evidence_documents))
print("Embedding shape    :", evidence_embeddings.shape)
print("FAISS vectors      :", faiss_index.ntotal)
print("Embedding dimension:", embedding_dimension)

print("\n" + "=" * 80)
print("M7 EVIDENCE RETRIEVAL INITIALIZATION COMPLETE")
print("=" * 80)

ModuleNotFoundError: No module named 'faiss'

In [8]:
# ============================================================
# M7 — INSTALL FAISS
# Purpose:
# Install FAISS for vector similarity search.
# ============================================================

!pip install -q faiss-cpu

print("FAISS installation complete.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 54.3 MB/s eta 0:00:00
FAISS installation complete.


In [9]:
# ============================================================
# M7 — EVIDENCE RETRIEVAL INITIALIZATION
# Purpose:
# Rebuild the speaker-level evidence collection and BGE + FAISS
# retrieval resources after a Colab runtime restart.
#
# Source:
#   ES2004a speaker-level transcript
#
# Retrieval model:
#   BAAI/bge-small-en-v1.5
#
# FAISS:
#   IndexFlatIP on normalized embeddings
#   → inner product approximates cosine similarity
# ============================================================

import os
import json
import numpy as np
import torch

from sentence_transformers import SentenceTransformer
import faiss

# ------------------------------------------------------------
# 1. Paths
# ------------------------------------------------------------

SPEAKER_TRANSCRIPT_PATH = os.path.join(
    TRANSCRIPTS_DIR,
    "ES2004a_speaker_transcript.json"
)

# ------------------------------------------------------------
# 2. Load speaker-level transcript
# ------------------------------------------------------------

with open(SPEAKER_TRANSCRIPT_PATH, "r", encoding="utf-8") as f:
    speaker_transcript_data = json.load(f)

speaker_utterances = speaker_transcript_data["speaker_utterances"]

# ------------------------------------------------------------
# 3. Create evidence documents
# ------------------------------------------------------------

evidence_documents = []

for i, utterance in enumerate(speaker_utterances):

    evidence_documents.append({
        "evidence_id": i + 1,
        "speaker": utterance["speaker"],
        "start": utterance["start"],
        "end": utterance["end"],
        "duration": utterance["end"] - utterance["start"],
        "text": utterance["text"]
    })

# ------------------------------------------------------------
# 4. Load BGE embedding model
# ------------------------------------------------------------

BGE_MODEL_NAME = "BAAI/bge-small-en-v1.5"

print("=" * 80)
print("M7 — EVIDENCE RETRIEVAL INITIALIZATION")
print("=" * 80)

print("\nDevice:", DEVICE)

print("\nLoading BGE model...")
bge_model = SentenceTransformer(
    BGE_MODEL_NAME,
    device=DEVICE
)

print("BGE model loaded.")

# ------------------------------------------------------------
# 5. Generate normalized evidence embeddings
# ------------------------------------------------------------

evidence_texts = [
    item["text"]
    for item in evidence_documents
]

print("\nGenerating evidence embeddings...")

evidence_embeddings = bge_model.encode(
    evidence_texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

evidence_embeddings = evidence_embeddings.astype(
    np.float32
)

# ------------------------------------------------------------
# 6. Build FAISS index
# ------------------------------------------------------------

embedding_dimension = evidence_embeddings.shape[1]

faiss_index = faiss.IndexFlatIP(
    embedding_dimension
)

faiss_index.add(evidence_embeddings)

# ------------------------------------------------------------
# 7. Retrieval function
# ------------------------------------------------------------

def retrieve_evidence(claim_text, top_k=10):

    query_embedding = bge_model.encode(
        [claim_text],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype(np.float32)

    similarities, indices = faiss_index.search(
        query_embedding,
        top_k
    )

    results = []

    for rank, (index, similarity) in enumerate(
        zip(indices[0], similarities[0]),
        start=1
    ):

        evidence = evidence_documents[index]

        results.append({
            "retrieval_rank": rank,
            "retrieval_similarity": float(similarity),
            **evidence
        })

    return results

# ------------------------------------------------------------
# 8. Summary
# ------------------------------------------------------------

print("\nEvidence documents :", len(evidence_documents))
print("Embedding shape    :", evidence_embeddings.shape)
print("FAISS vectors      :", faiss_index.ntotal)
print("Embedding dimension:", embedding_dimension)

print("\n" + "=" * 80)
print("M7 EVIDENCE RETRIEVAL INITIALIZATION COMPLETE")
print("=" * 80)

KeyError: 'speaker_utterances'

In [10]:
# ============================================================
# M7 — INSPECT SPEAKER TRANSCRIPT JSON
# Purpose:
# Identify the actual structure of the saved transcript JSON.
# No data is modified.
# ============================================================

print("=" * 80)
print("M7 — SPEAKER TRANSCRIPT JSON STRUCTURE")
print("=" * 80)

print("\nTop-level keys:")
for key in speaker_transcript_data.keys():
    print(" ", key)

print("\nData types:")
for key, value in speaker_transcript_data.items():
    print(f"  {key}: {type(value).__name__}")

print("\n" + "=" * 80)

M7 — SPEAKER TRANSCRIPT JSON STRUCTURE

Top-level keys:
  meeting_id
  source
  gap_threshold_seconds
  num_speakers
  speakers
  num_utterances
  utterances

Data types:
  meeting_id: str
  source: str
  gap_threshold_seconds: float
  num_speakers: int
  speakers: list
  num_utterances: int
  utterances: list



In [11]:
# ============================================================
# M7 — EVIDENCE RETRIEVAL INITIALIZATION
# Purpose:
# Build speaker-level evidence documents and the BGE + FAISS
# retrieval index from the saved speaker transcript.
#
# Source:
#   ES2004a_speaker_transcript.json
#
# Retrieval model:
#   BAAI/bge-small-en-v1.5
#
# FAISS:
#   IndexFlatIP with normalized embeddings
# ============================================================

import numpy as np
from sentence_transformers import SentenceTransformer
import faiss

# ------------------------------------------------------------
# 1. Use the correct transcript structure
# ------------------------------------------------------------

speaker_utterances = speaker_transcript_data["utterances"]

# ------------------------------------------------------------
# 2. Create evidence documents
# ------------------------------------------------------------

evidence_documents = []

for i, utterance in enumerate(speaker_utterances):

    evidence_documents.append({
        "evidence_id": i + 1,
        "speaker": utterance["speaker"],
        "start": utterance["start"],
        "end": utterance["end"],
        "duration": utterance["end"] - utterance["start"],
        "text": utterance["text"]
    })

# ------------------------------------------------------------
# 3. Load BGE embedding model
# ------------------------------------------------------------

BGE_MODEL_NAME = "BAAI/bge-small-en-v1.5"

print("=" * 80)
print("M7 — EVIDENCE RETRIEVAL INITIALIZATION")
print("=" * 80)

print("\nDevice:", DEVICE)

print("\nLoading BGE model...")

bge_model = SentenceTransformer(
    BGE_MODEL_NAME,
    device=DEVICE
)

print("BGE model loaded.")

# ------------------------------------------------------------
# 4. Generate evidence embeddings
# ------------------------------------------------------------

evidence_texts = [
    item["text"]
    for item in evidence_documents
]

print("\nGenerating evidence embeddings...")

evidence_embeddings = bge_model.encode(
    evidence_texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

evidence_embeddings = evidence_embeddings.astype(
    np.float32
)

# ------------------------------------------------------------
# 5. Build FAISS index
# ------------------------------------------------------------

embedding_dimension = evidence_embeddings.shape[1]

faiss_index = faiss.IndexFlatIP(
    embedding_dimension
)

faiss_index.add(evidence_embeddings)

# ------------------------------------------------------------
# 6. Retrieval function
# ------------------------------------------------------------

def retrieve_evidence(claim_text, top_k=10):

    query_embedding = bge_model.encode(
        [claim_text],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype(np.float32)

    similarities, indices = faiss_index.search(
        query_embedding,
        top_k
    )

    results = []

    for rank, (index, similarity) in enumerate(
        zip(indices[0], similarities[0]),
        start=1
    ):

        evidence = evidence_documents[index]

        results.append({
            "retrieval_rank": rank,
            "retrieval_similarity": float(similarity),
            **evidence
        })

    return results

# ------------------------------------------------------------
# 7. Summary
# ------------------------------------------------------------

print("\nEvidence documents :", len(evidence_documents))
print("Embedding shape    :", evidence_embeddings.shape)
print("FAISS vectors      :", faiss_index.ntotal)
print("Embedding dimension:", embedding_dimension)

print("\n" + "=" * 80)
print("M7 EVIDENCE RETRIEVAL INITIALIZATION COMPLETE")
print("=" * 80)

M7 — EVIDENCE RETRIEVAL INITIALIZATION

Device: cuda

Loading BGE model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

BGE model loaded.

Generating evidence embeddings...


Batches:   0%|          | 0/5 [00:00<?, ?it/s]


Evidence documents : 148
Embedding shape    : (148, 384)
FAISS vectors      : 148
Embedding dimension: 384

M7 EVIDENCE RETRIEVAL INITIALIZATION COMPLETE


In [12]:
# ============================================================
# M7 — BGE RETRIEVAL TEST
# Purpose:
# Test semantic retrieval for one factual MoM proposition.
#
# Note:
# BGE similarity indicates semantic relevance.
# It does NOT by itself prove that the evidence supports
# the proposition.
# ============================================================

proposition = (
    "The remote control is intended to be original."
)

results = retrieve_evidence(
    proposition,
    top_k=5
)

print("=" * 80)
print("M7 — BGE RETRIEVAL TEST")
print("=" * 80)

print("\nProposition:")
print(proposition)

print("\nTop retrieved evidence:")

for item in results:

    print("\n" + "-" * 80)
    print(
        f"Rank       : {item['retrieval_rank']}"
    )
    print(
        f"Similarity : {item['retrieval_similarity']:.4f}"
    )
    print(
        f"Evidence ID: {item['evidence_id']}"
    )
    print(
        f"Speaker    : {item['speaker']}"
    )
    print(
        f"Time       : {item['start']:.3f} - {item['end']:.3f}"
    )
    print(
        f"Text       : {item['text']}"
    )

print("\n" + "=" * 80)

M7 — BGE RETRIEVAL TEST

Proposition:
The remote control is intended to be original.

Top retrieved evidence:

--------------------------------------------------------------------------------
Rank       : 1
Similarity : 0.7917
Evidence ID: 105
Speaker    : SPEAKER_01
Time       : 707.248 - 710.889
Text       : remote controls. You want to integrate everything into one.

--------------------------------------------------------------------------------
Rank       : 2
Similarity : 0.7810
Evidence ID: 14
Speaker    : SPEAKER_02
Time       : 117.699 - 120.921
Text       : Now, we're developing a remote control, which you probably already know.

--------------------------------------------------------------------------------
Rank       : 3
Similarity : 0.7408
Evidence ID: 107
Speaker    : SPEAKER_03
Time       : 713.150 - 729.357
Text       : experience has only been given the remote control with the object I buy, not doing any tampering with it and programming, using it to program TV videos 

In [13]:
# ============================================================
# M7 — CONTEXT EXPANSION TEST
# Purpose:
# Check whether neighboring transcript segments around the
# retrieved evidence contain the specific information needed
# to verify the proposition.
#
# This is an experimental inspection step.
# No verification decision is made yet.
# ============================================================

proposition = (
    "The remote control is intended to be original."
)

# Retrieve the top evidence
results = retrieve_evidence(
    proposition,
    top_k=5
)

# Select the highest-ranked evidence
top_evidence_id = results[0]["evidence_id"]

# Convert to zero-based list position
top_index = top_evidence_id - 1

# Number of neighboring evidence segments to inspect
CONTEXT_WINDOW = 3

start_index = max(
    0,
    top_index - CONTEXT_WINDOW
)

end_index = min(
    len(evidence_documents),
    top_index + CONTEXT_WINDOW + 1
)

context_documents = evidence_documents[
    start_index:end_index
]

print("=" * 80)
print("M7 — CONTEXT EXPANSION TEST")
print("=" * 80)

print("\nProposition:")
print(proposition)

print("\nTop retrieved evidence:")
print(
    f"Evidence ID {results[0]['evidence_id']} "
    f"(similarity={results[0]['retrieval_similarity']:.4f})"
)

print("\nContext window:")
print(
    f"Evidence IDs {context_documents[0]['evidence_id']} "
    f"to {context_documents[-1]['evidence_id']}"
)

for item in context_documents:

    marker = (
        " <-- TOP RETRIEVED"
        if item["evidence_id"] == top_evidence_id
        else ""
    )

    print("\n" + "-" * 80)
    print(
        f"Evidence ID: {item['evidence_id']}{marker}"
    )
    print(
        f"Speaker    : {item['speaker']}"
    )
    print(
        f"Time       : {item['start']:.3f} - "
        f"{item['end']:.3f}"
    )
    print(
        f"Text       : {item['text']}"
    )

print("\n" + "=" * 80)

M7 — CONTEXT EXPANSION TEST

Proposition:
The remote control is intended to be original.

Top retrieved evidence:
Evidence ID 105 (similarity=0.7917)

Context window:
Evidence IDs 102 to 108

--------------------------------------------------------------------------------
Evidence ID: 102
Speaker    : SPEAKER_02
Time       : 682.011 - 704.502
Text       : we had three videos, a TV and a sort of amp thing all set up. So we got one of the universal remote controls that you program each of your things into. But that kept losing the signals, so we'd have to reprogram it every now and again. I think it was quite cheapy as well. So that might have had something to do with it. But that was quite good, the fact that you could

--------------------------------------------------------------------------------
Evidence ID: 103
Speaker    : SPEAKER_00
Time       : 705.387 - 706.668
Text       : use all the ones. You didn't have

----------------------------------------------------------------------

In [14]:
# ============================================================
# M7 — TOP-20 BGE RETRIEVAL TEST
# Purpose:
# Determine whether the correct evidence for an attribute
# appears within a larger semantic retrieval candidate pool.
# ============================================================

proposition = (
    "The remote control is intended to be original."
)

results = retrieve_evidence(
    proposition,
    top_k=20
)

print("=" * 80)
print("M7 — TOP-20 BGE RETRIEVAL TEST")
print("=" * 80)

print("\nProposition:")
print(proposition)

print("\nTop 20 retrieved evidence segments:")

for item in results:

    print("\n" + "-" * 80)

    print(
        f"Rank       : {item['retrieval_rank']}"
    )

    print(
        f"Similarity : {item['retrieval_similarity']:.4f}"
    )

    print(
        f"Evidence ID: {item['evidence_id']}"
    )

    print(
        f"Speaker    : {item['speaker']}"
    )

    print(
        f"Time       : {item['start']:.3f} - "
        f"{item['end']:.3f}"
    )

    print(
        f"Text       : {item['text']}"
    )

print("\n" + "=" * 80)

M7 — TOP-20 BGE RETRIEVAL TEST

Proposition:
The remote control is intended to be original.

Top 20 retrieved evidence segments:

--------------------------------------------------------------------------------
Rank       : 1
Similarity : 0.7917
Evidence ID: 105
Speaker    : SPEAKER_01
Time       : 707.248 - 710.889
Text       : remote controls. You want to integrate everything into one.

--------------------------------------------------------------------------------
Rank       : 2
Similarity : 0.7810
Evidence ID: 14
Speaker    : SPEAKER_02
Time       : 117.699 - 120.921
Text       : Now, we're developing a remote control, which you probably already know.

--------------------------------------------------------------------------------
Rank       : 3
Similarity : 0.7408
Evidence ID: 107
Speaker    : SPEAKER_03
Time       : 713.150 - 729.357
Text       : experience has only been given the remote control with the object I buy, not doing any tampering with it and programming, using it to

In [15]:
# ============================================================
# M7.4 — Attribute-Level Lexical Evidence Support
# ============================================================
#
# Purpose:
#   Identify whether important content words from a proposition
#   are explicitly present in retrieved evidence.
#
# Important:
#   This is NOT the final verification decision.
#   It is one signal that will later be combined with:
#       1. BGE retrieval
#       2. NLI
#       3. Speaker consistency
#       4. Timestamp consistency
#       5. Event-type consistency
#
# The function is intentionally lightweight and restart-safe.
# No model is loaded and no existing files are modified.
# ============================================================

import re

# Common English stopwords.
# These are ignored when calculating meaningful lexical overlap.
STOPWORDS = {
    "the", "a", "an", "and", "or", "but", "if", "then",
    "is", "are", "was", "were", "be", "been", "being",
    "to", "of", "in", "on", "for", "with", "by", "from",
    "at", "as", "into", "about", "over", "after", "before",
    "this", "that", "these", "those", "it", "its",
    "they", "them", "their", "there", "here",
    "he", "she", "we", "you", "i",
    "will", "would", "should", "could", "can", "may", "might",
    "do", "does", "did", "done",
    "have", "has", "had",
    "not", "no",
    "be", "being"
}


def normalize_tokens(text):
    """
    Convert text into lowercase content-word tokens.
    """
    text = text.lower()

    # Keep alphabetic words.
    tokens = re.findall(r"[a-z]+", text)

    # Remove stopwords and very short tokens.
    tokens = [
        token for token in tokens
        if token not in STOPWORDS and len(token) > 2
    ]

    return tokens


def lexical_support(proposition, evidence):
    """
    Calculate simple lexical support between a proposition
    and an evidence sentence.

    Returns:
        proposition_terms
        matched_terms
        missing_terms
        overlap_ratio
    """

    proposition_terms = list(dict.fromkeys(normalize_tokens(proposition)))
    evidence_terms = set(normalize_tokens(evidence))

    matched_terms = [
        term for term in proposition_terms
        if term in evidence_terms
    ]

    missing_terms = [
        term for term in proposition_terms
        if term not in evidence_terms
    ]

    if len(proposition_terms) == 0:
        overlap_ratio = 0.0
    else:
        overlap_ratio = len(matched_terms) / len(proposition_terms)

    return {
        "proposition_terms": proposition_terms,
        "matched_terms": matched_terms,
        "missing_terms": missing_terms,
        "overlap_ratio": round(overlap_ratio, 4)
    }


# ------------------------------------------------------------
# Controlled test using the known "original" proposition
# ------------------------------------------------------------

test_proposition = "The remote control is intended to be original."

test_evidence = (
    "We want it to be original. Trendy, appealing to a wide market."
)

result = lexical_support(test_proposition, test_evidence)

print("PROPOSITION:")
print(test_proposition)

print("\nEVIDENCE:")
print(test_evidence)

print("\nLEXICAL SUPPORT:")
print("Proposition terms :", result["proposition_terms"])
print("Matched terms     :", result["matched_terms"])
print("Missing terms     :", result["missing_terms"])
print("Overlap ratio     :", result["overlap_ratio"])

PROPOSITION:
The remote control is intended to be original.

EVIDENCE:
We want it to be original. Trendy, appealing to a wide market.

LEXICAL SUPPORT:
Proposition terms : ['remote', 'control', 'intended', 'original']
Matched terms     : ['original']
Missing terms     : ['remote', 'control', 'intended']
Overlap ratio     : 0.25


In [16]:
# ============================================================
# M7.4 — Lexical Support Diagnostic Tests
# ============================================================
#
# Purpose:
#   Test lexical overlap on:
#       1. Direct supporting evidence
#       2. Conversational/paraphrased evidence
#       3. Contradictory evidence
#       4. Unrelated evidence
#
# This is a diagnostic only.
# No verification threshold is introduced here.
# ============================================================

test_cases = [
    {
        "name": "Direct support",
        "proposition": "The remote control is intended to be original.",
        "evidence": "The remote control is intended to be original."
    },
    {
        "name": "Conversational support",
        "proposition": "The remote control is intended to be original.",
        "evidence": "We want it to be original. Trendy, appealing to a wide market."
    },
    {
        "name": "Contradictory evidence",
        "proposition": "The remote control is intended to be original.",
        "evidence": "We don't want it to be original. We want something similar to existing products."
    },
    {
        "name": "Unrelated evidence",
        "proposition": "The remote control is intended to be original.",
        "evidence": "The meeting will take place next week and everyone should attend."
    }
]


for i, case in enumerate(test_cases, start=1):

    result = lexical_support(
        case["proposition"],
        case["evidence"]
    )

    print("=" * 70)
    print(f"TEST {i}: {case['name']}")
    print("-" * 70)

    print("Proposition:")
    print(case["proposition"])

    print("\nEvidence:")
    print(case["evidence"])

    print("\nMatched terms:")
    print(result["matched_terms"])

    print("Missing terms:")
    print(result["missing_terms"])

    print("Overlap ratio:")
    print(result["overlap_ratio"])

print("=" * 70)

TEST 1: Direct support
----------------------------------------------------------------------
Proposition:
The remote control is intended to be original.

Evidence:
The remote control is intended to be original.

Matched terms:
['remote', 'control', 'intended', 'original']
Missing terms:
[]
Overlap ratio:
1.0
TEST 2: Conversational support
----------------------------------------------------------------------
Proposition:
The remote control is intended to be original.

Evidence:
We want it to be original. Trendy, appealing to a wide market.

Matched terms:
['original']
Missing terms:
['remote', 'control', 'intended']
Overlap ratio:
0.25
TEST 3: Contradictory evidence
----------------------------------------------------------------------
Proposition:
The remote control is intended to be original.

Evidence:
We don't want it to be original. We want something similar to existing products.

Matched terms:
['original']
Missing terms:
['remote', 'control', 'intended']
Overlap ratio:
0.25
TES

In [19]:
# ============================================================
# M7.5 — Contextual Evidence Window
# ============================================================
#
# Purpose:
#   Return nearby transcript evidence around a selected
#   evidence document.
#
# Evidence IDs in the current saved data are INTEGER values.
# ============================================================

def get_context_window(evidence_id, window_size=2):
    """
    Return nearby evidence documents around evidence_id.

    Parameters
    ----------
    evidence_id : int
        Integer evidence ID.

    window_size : int
        Number of evidence documents before and after
        the selected document.

    Returns
    -------
    list
        Chronological context window.
    """

    # Find the selected evidence document
    target_index = None

    for i, doc in enumerate(evidence_documents):

        if int(doc["evidence_id"]) == int(evidence_id):
            target_index = i
            break

    if target_index is None:

        available_ids = [
            doc["evidence_id"]
            for doc in evidence_documents[:10]
        ]

        raise ValueError(
            f"Evidence ID {evidence_id} not found.\n"
            f"Example available IDs: {available_ids}"
        )

    # Determine context boundaries
    start_index = max(
        0,
        target_index - window_size
    )

    end_index = min(
        len(evidence_documents),
        target_index + window_size + 1
    )

    # Return chronological context
    return evidence_documents[start_index:end_index]


# ------------------------------------------------------------
# Test with the actual evidence ID
# ------------------------------------------------------------

test_evidence_id = 15

context = get_context_window(
    test_evidence_id,
    window_size=2
)

print("=" * 70)
print(f"CONTEXT WINDOW AROUND EVIDENCE {test_evidence_id}")
print("=" * 70)

for doc in context:

    print(
        f"Evidence {doc['evidence_id']} | "
        f"{doc['speaker']} | "
        f"{doc['start']:.3f} → {doc['end']:.3f}"
    )

    print(f"  {doc['text']}")
    print()

CONTEXT WINDOW AROUND EVIDENCE 15
Evidence 13 | SPEAKER_02 | 112.185 → 115.748
  And we've got 25 minutes to do that as far as I can understand.

Evidence 14 | SPEAKER_02 | 117.699 → 120.921
  Now, we're developing a remote control, which you probably already know.

Evidence 15 | SPEAKER_02 | 122.503 → 132.290
  We want it to be original, something that people haven't thought of. It's not out in the shops. Trendy, appealing to a wide market, but,

Evidence 16 | SPEAKER_02 | 133.291 → 138.174
  you know, not a hunk of metal. And user-friendly, grannies to kids,

Evidence 17 | SPEAKER_02 | 139.375 → 141.016
  maybe even pooches, should be able to use it.



In [20]:
# ============================================================
# M7.6 — Contextual NLI Verification Test
# ============================================================
#
# Purpose:
#   Compare NLI performance using:
#
#       A. Single evidence utterance
#       B. Expanded contextual evidence
#
# This is a diagnostic experiment.
# It does NOT yet define the final verification rule.
# ============================================================


# ------------------------------------------------------------
# Proposition to verify
# ------------------------------------------------------------

proposition = (
    "The remote control is intended to be original."
)


# ------------------------------------------------------------
# Single evidence
# ------------------------------------------------------------

single_evidence = (
    "We want it to be original. "
    "something that people haven't thought of. "
    "It's not out in the shops. "
    "Trendy, appealing to a wide market, but,"
)


# ------------------------------------------------------------
# Contextual evidence
# ------------------------------------------------------------

context_evidence = " ".join(
    doc["text"]
    for doc in context
)


# ------------------------------------------------------------
# Run NLI on single evidence
# ------------------------------------------------------------

single_result = run_nli(
    single_evidence,
    proposition
)


# ------------------------------------------------------------
# Run NLI on contextual evidence
# ------------------------------------------------------------

context_result = run_nli(
    context_evidence,
    proposition
)


# ------------------------------------------------------------
# Display results
# ------------------------------------------------------------

print("=" * 70)
print("PROPOSITION")
print("=" * 70)
print(proposition)


print("\n" + "=" * 70)
print("SINGLE EVIDENCE")
print("=" * 70)
print(single_evidence)

print("\nNLI RESULT:")
print(single_result)


print("\n" + "=" * 70)
print("CONTEXTUAL EVIDENCE")
print("=" * 70)
print(context_evidence)

print("\nNLI RESULT:")
print(context_result)

PROPOSITION
The remote control is intended to be original.

SINGLE EVIDENCE
We want it to be original. something that people haven't thought of. It's not out in the shops. Trendy, appealing to a wide market, but,

NLI RESULT:
{'predicted_label': 'NEUTRAL', 'confidence': 0.9994945526123047, 'contradiction_probability': 0.0003343396238051355, 'entailment_probability': 0.0001710433280095458, 'neutral_probability': 0.9994945526123047}

CONTEXTUAL EVIDENCE
And we've got 25 minutes to do that as far as I can understand. Now, we're developing a remote control, which you probably already know. We want it to be original, something that people haven't thought of. It's not out in the shops. Trendy, appealing to a wide market, but, you know, not a hunk of metal. And user-friendly, grannies to kids, maybe even pooches, should be able to use it.

NLI RESULT:
{'predicted_label': 'ENTAILMENT', 'confidence': 0.9787161946296692, 'contradiction_probability': 0.0013938040938228369, 'entailment_probability

In [21]:
# ============================================================
# M7.7 — Contextual NLI Controlled Evaluation
# ============================================================
#
# Purpose:
#   Evaluate DeBERTa NLI on three controlled cases:
#
#       1. Supporting evidence
#       2. Contradictory evidence
#       3. Unrelated evidence
#
# This helps us understand whether NLI can safely
# participate in the final verification pipeline.
#
# NOTE:
#   We will NOT use these results to create arbitrary
#   thresholds yet.
# ============================================================


controlled_cases = [

    {
        "name": "SUPPORTING",
        "evidence": (
            "Now, we're developing a remote control, "
            "which you probably already know. "
            "We want it to be original, "
            "something that people haven't thought of."
        ),
        "proposition": (
            "The remote control is intended to be original."
        )
    },

    {
        "name": "CONTRADICTORY",
        "evidence": (
            "Now, we're developing a remote control, "
            "which you probably already know. "
            "We don't want it to be original. "
            "We want something similar to existing products."
        ),
        "proposition": (
            "The remote control is intended to be original."
        )
    },

    {
        "name": "UNRELATED",
        "evidence": (
            "The meeting will take place next week. "
            "Everyone should attend the next meeting."
        ),
        "proposition": (
            "The remote control is intended to be original."
        )
    }
]


print("=" * 70)
print("CONTEXTUAL NLI CONTROLLED EVALUATION")
print("=" * 70)


for i, case in enumerate(controlled_cases, start=1):

    result = run_nli(
        case["evidence"],
        case["proposition"]
    )

    print("\n" + "=" * 70)
    print(f"TEST {i}: {case['name']}")
    print("-" * 70)

    print("Evidence:")
    print(case["evidence"])

    print("\nProposition:")
    print(case["proposition"])

    print("\nNLI Result:")
    print(
        f"Predicted label : {result['predicted_label']}"
    )

    print(
        f"Contradiction   : "
        f"{result['contradiction_probability']:.4f}"
    )

    print(
        f"Entailment      : "
        f"{result['entailment_probability']:.4f}"
    )

    print(
        f"Neutral         : "
        f"{result['neutral_probability']:.4f}"
    )

print("\n" + "=" * 70)
print("Evaluation complete.")
print("=" * 70)

CONTEXTUAL NLI CONTROLLED EVALUATION

TEST 1: SUPPORTING
----------------------------------------------------------------------
Evidence:
Now, we're developing a remote control, which you probably already know. We want it to be original, something that people haven't thought of.

Proposition:
The remote control is intended to be original.

NLI Result:
Predicted label : ENTAILMENT
Contradiction   : 0.0086
Entailment      : 0.9643
Neutral         : 0.0270

TEST 2: CONTRADICTORY
----------------------------------------------------------------------
Evidence:
Now, we're developing a remote control, which you probably already know. We don't want it to be original. We want something similar to existing products.

Proposition:
The remote control is intended to be original.

NLI Result:
Predicted label : CONTRADICTION
Contradiction   : 0.9972
Entailment      : 0.0006
Neutral         : 0.0021

TEST 3: UNRELATED
----------------------------------------------------------------------
Evidence:
The

In [22]:
# ============================================================
# M7.8 — Real Meeting Attribute-Level NLI Evaluation
# ============================================================
#
# Purpose:
#   Test contextual NLI on real attributes extracted from
#   Claim 2 of the meeting.
#
# Claim 2:
#   "The team is developing a remote control intended to be
#    original, trendy, appealing to a wide market, and
#    user-friendly for a broad range of users."
#
# We evaluate each attribute separately using its known
# source evidence.
#
# This is an evaluation step only.
# It does NOT modify any saved project files.
# ============================================================


# ------------------------------------------------------------
# Claim 2 attributes
# ------------------------------------------------------------

claim_2_attributes = [
    {
        "attribute": "product_development",
        "proposition": "The team is developing a remote control.",
        "source_ids": [14]
    },
    {
        "attribute": "originality",
        "proposition": "The remote control is intended to be original.",
        "source_ids": [15]
    },
    {
        "attribute": "trendiness",
        "proposition": "The remote control is intended to be trendy.",
        "source_ids": [15]
    },
    {
        "attribute": "market_appeal",
        "proposition": "The remote control is intended to appeal to a wide market.",
        "source_ids": [15]
    },
    {
        "attribute": "user_friendliness",
        "proposition": "The remote control is intended to be user-friendly.",
        "source_ids": [16]
    },
    {
        "attribute": "broad_usability",
        "proposition": "The remote control should be usable by a broad range of users.",
        "source_ids": [16, 17]
    }
]


# ------------------------------------------------------------
# Build contextual evidence for each attribute
# ------------------------------------------------------------

print("=" * 80)
print("REAL MEETING ATTRIBUTE-LEVEL NLI EVALUATION")
print("=" * 80)


attribute_results = []


for item in claim_2_attributes:

    # Use the first source evidence as the center of the
    # contextual window.
    source_id = item["source_ids"][0]

    context_docs = get_context_window(
        source_id,
        window_size=2
    )

    context_text = " ".join(
        doc["text"]
        for doc in context_docs
    )

    # Run contextual NLI
    nli_result = run_nli(
        context_text,
        item["proposition"]
    )

    result = {
        "attribute": item["attribute"],
        "proposition": item["proposition"],
        "source_ids": item["source_ids"],
        "nli_label": nli_result["predicted_label"],
        "entailment_probability": nli_result["entailment_probability"],
        "contradiction_probability": nli_result["contradiction_probability"],
        "neutral_probability": nli_result["neutral_probability"]
    }

    attribute_results.append(result)

    print("\n" + "-" * 80)
    print(f"ATTRIBUTE: {item['attribute']}")
    print("-" * 80)

    print("Proposition:")
    print(item["proposition"])

    print("\nSource evidence IDs:")
    print(item["source_ids"])

    print("\nContext:")
    print(context_text)

    print("\nNLI:")
    print(f"Label         : {nli_result['predicted_label']}")
    print(
        f"Entailment    : "
        f"{nli_result['entailment_probability']:.4f}"
    )
    print(
        f"Contradiction : "
        f"{nli_result['contradiction_probability']:.4f}"
    )
    print(
        f"Neutral       : "
        f"{nli_result['neutral_probability']:.4f}"
    )


# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("ATTRIBUTE-LEVEL SUMMARY")
print("=" * 80)

for result in attribute_results:

    print(
        f"{result['attribute']:22s} | "
        f"{result['nli_label']:13s} | "
        f"Entailment = "
        f"{result['entailment_probability']:.4f}"
    )

REAL MEETING ATTRIBUTE-LEVEL NLI EVALUATION

--------------------------------------------------------------------------------
ATTRIBUTE: product_development
--------------------------------------------------------------------------------
Proposition:
The team is developing a remote control.

Source evidence IDs:
[14]

Context:
talk about the project plan, discuss our own ideas and everything. And we've got 25 minutes to do that as far as I can understand. Now, we're developing a remote control, which you probably already know. We want it to be original, something that people haven't thought of. It's not out in the shops. Trendy, appealing to a wide market, but, you know, not a hunk of metal. And user-friendly, grannies to kids,

NLI:
Label         : ENTAILMENT
Entailment    : 0.9799
Contradiction : 0.0013
Neutral       : 0.0188

--------------------------------------------------------------------------------
ATTRIBUTE: originality
-------------------------------------------------------

In [23]:
# ============================================================
# M7.9 — Multi-Source Evidence NLI
# ============================================================
#
# Purpose:
#   Test whether combining all known source utterances
#   improves NLI for an attribute.
#
# This is especially important for attributes whose meaning
# is distributed across multiple conversational utterances.
#
# No files are modified.
# ============================================================


# ------------------------------------------------------------
# Attribute under investigation
# ------------------------------------------------------------

proposition = (
    "The remote control should be usable by a broad range "
    "of users."
)

source_ids = [16, 17]


# ------------------------------------------------------------
# Retrieve the exact source evidence
# ------------------------------------------------------------

source_documents = []

for source_id in source_ids:

    for doc in evidence_documents:

        if int(doc["evidence_id"]) == int(source_id):
            source_documents.append(doc)
            break


# ------------------------------------------------------------
# Build evidence from the exact source utterances
# ------------------------------------------------------------

exact_source_text = " ".join(
    doc["text"]
    for doc in source_documents
)


# ------------------------------------------------------------
# Run NLI on exact source evidence
# ------------------------------------------------------------

exact_result = run_nli(
    exact_source_text,
    proposition
)


# ------------------------------------------------------------
# Also create a slightly larger context around ALL sources
# ------------------------------------------------------------

all_context_documents = []

for source_id in source_ids:

    context_docs = get_context_window(
        source_id,
        window_size=2
    )

    for doc in context_docs:

        if doc["evidence_id"] not in [
            d["evidence_id"] for d in all_context_documents
        ]:
            all_context_documents.append(doc)


# Sort chronologically
all_context_documents = sorted(
    all_context_documents,
    key=lambda x: x["start"]
)


combined_context_text = " ".join(
    doc["text"]
    for doc in all_context_documents
)


# ------------------------------------------------------------
# Run NLI on combined context
# ------------------------------------------------------------

combined_result = run_nli(
    combined_context_text,
    proposition
)


# ------------------------------------------------------------
# Display results
# ------------------------------------------------------------

print("=" * 80)
print("MULTI-SOURCE EVIDENCE NLI TEST")
print("=" * 80)

print("\nPROPOSITION:")
print(proposition)


print("\n" + "-" * 80)
print("EXACT SOURCE EVIDENCE")
print("-" * 80)

for doc in source_documents:

    print(
        f"Evidence {doc['evidence_id']} | "
        f"{doc['speaker']} | "
        f"{doc['start']:.3f} → {doc['end']:.3f}"
    )

    print(doc["text"])
    print()


print("NLI RESULT:")
print(exact_result)


print("\n" + "-" * 80)
print("COMBINED CONTEXT")
print("-" * 80)

for doc in all_context_documents:

    print(
        f"Evidence {doc['evidence_id']} | "
        f"{doc['speaker']} | "
        f"{doc['start']:.3f} → {doc['end']:.3f}"
    )

    print(doc["text"])
    print()


print("NLI RESULT:")
print(combined_result)

MULTI-SOURCE EVIDENCE NLI TEST

PROPOSITION:
The remote control should be usable by a broad range of users.

--------------------------------------------------------------------------------
EXACT SOURCE EVIDENCE
--------------------------------------------------------------------------------
Evidence 16 | SPEAKER_02 | 133.291 → 138.174
you know, not a hunk of metal. And user-friendly, grannies to kids,

Evidence 17 | SPEAKER_02 | 139.375 → 141.016
maybe even pooches, should be able to use it.

NLI RESULT:
{'predicted_label': 'NEUTRAL', 'confidence': 0.9989136457443237, 'contradiction_probability': 0.00027211118140257895, 'entailment_probability': 0.0008143104496411979, 'neutral_probability': 0.9989136457443237}

--------------------------------------------------------------------------------
COMBINED CONTEXT
--------------------------------------------------------------------------------
Evidence 14 | SPEAKER_02 | 117.699 → 120.921
Now, we're developing a remote control, which you prob

In [24]:
# ============================================================
# M7.10 — BGE Semantic Evidence Support Test
# ============================================================
#
# Purpose:
#   Test whether BGE semantic similarity can recognize
#   conversational/paraphrased evidence that NLI failed
#   to recognize.
#
# Example:
#
# Proposition:
#   "The remote control should be usable by a broad range
#    of users."
#
# Evidence:
#   "grannies to kids, maybe even pooches, should be able
#    to use it."
#
# BGE similarity is treated as a retrieval/support signal,
# NOT as proof of truth.
#
# No files are modified.
# ============================================================


# ------------------------------------------------------------
# Proposition
# ------------------------------------------------------------

proposition = (
    "The remote control should be usable by a broad range "
    "of users."
)


# ------------------------------------------------------------
# Exact evidence documents
# ------------------------------------------------------------

source_ids = [16, 17]

source_documents = []

for source_id in source_ids:

    for doc in evidence_documents:

        if int(doc["evidence_id"]) == int(source_id):

            source_documents.append(doc)
            break


# ------------------------------------------------------------
# Combine the source evidence
# ------------------------------------------------------------

source_text = " ".join(
    doc["text"]
    for doc in source_documents
)


# ------------------------------------------------------------
# Generate proposition embedding
# ------------------------------------------------------------

proposition_embedding = bge_model.encode(
    [proposition],
    normalize_embeddings=True,
    convert_to_numpy=True
)


# ------------------------------------------------------------
# Generate evidence embedding
# ------------------------------------------------------------

evidence_embedding = bge_model.encode(
    [source_text],
    normalize_embeddings=True,
    convert_to_numpy=True
)


# ------------------------------------------------------------
# Cosine similarity
#
# Because both embeddings are normalized,
# dot product = cosine similarity.
# ------------------------------------------------------------

similarity = float(
    proposition_embedding[0] @ evidence_embedding[0]
)


# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

print("=" * 80)
print("BGE SEMANTIC EVIDENCE SUPPORT TEST")
print("=" * 80)

print("\nPROPOSITION:")
print(proposition)

print("\nEVIDENCE:")
print(source_text)

print("\nBGE COSINE SIMILARITY:")
print(f"{similarity:.4f}")

print("\nInterpretation:")
print(
    "This score is a semantic similarity signal only. "
    "It is NOT being used as proof or as a verification "
    "threshold at this stage."
)

BGE SEMANTIC EVIDENCE SUPPORT TEST

PROPOSITION:
The remote control should be usable by a broad range of users.

EVIDENCE:
you know, not a hunk of metal. And user-friendly, grannies to kids, maybe even pooches, should be able to use it.

BGE COSINE SIMILARITY:
0.5906

Interpretation:
This score is a semantic similarity signal only. It is NOT being used as proof or as a verification threshold at this stage.


In [25]:
# ============================================================
# M7.11 — Speaker & Timestamp Consistency Test
# ============================================================
#
# Purpose:
#   Test deterministic metadata consistency between a
#   generated MoM claim and its supporting evidence.
#
# These checks do NOT require another ML model.
#
# Speaker:
#   Exact comparison with evidence speaker.
#
# Timestamp:
#   Evidence timestamps are taken directly from the
#   speaker-attributed transcript.
#
# This is a diagnostic step only.
# ============================================================


def check_speaker_consistency(claim_speaker, evidence_speaker):
    """
    Check whether the claimed speaker matches the evidence speaker.
    """

    if claim_speaker is None:
        return {
            "applicable": False,
            "passed": True,
            "reason": "No speaker specified in claim."
        }

    if evidence_speaker is None:
        return {
            "applicable": False,
            "passed": True,
            "reason": "Evidence speaker unavailable."
        }

    passed = (
        str(claim_speaker).strip().upper()
        ==
        str(evidence_speaker).strip().upper()
    )

    return {
        "applicable": True,
        "passed": passed,
        "reason": (
            "Speaker matches evidence."
            if passed
            else "Speaker does not match evidence."
        )
    }


def check_timestamp_consistency(
    evidence_start,
    evidence_end,
    expected_start=None,
    expected_end=None,
    tolerance_seconds=2.0
):
    """
    Check timestamp consistency.

    If no expected timestamp is provided, the evidence timestamp
    itself is considered valid and is simply returned.

    If expected timestamps are available, allow a small tolerance.
    """

    # No expected timestamp means we use the evidence timestamp
    if expected_start is None and expected_end is None:
        return {
            "applicable": False,
            "passed": True,
            "reason": "Evidence timestamp accepted directly.",
            "evidence_start": evidence_start,
            "evidence_end": evidence_end
        }

    start_ok = True
    end_ok = True

    if expected_start is not None:
        start_ok = abs(
            float(evidence_start) - float(expected_start)
        ) <= tolerance_seconds

    if expected_end is not None:
        end_ok = abs(
            float(evidence_end) - float(expected_end)
        ) <= tolerance_seconds

    passed = start_ok and end_ok

    return {
        "applicable": True,
        "passed": passed,
        "reason": (
            "Timestamp is consistent."
            if passed
            else "Timestamp is inconsistent."
        ),
        "evidence_start": evidence_start,
        "evidence_end": evidence_end
    }


# ------------------------------------------------------------
# Test 1 — Correct speaker
# ------------------------------------------------------------

evidence = next(
    doc for doc in evidence_documents
    if int(doc["evidence_id"]) == 15
)

correct_speaker_result = check_speaker_consistency(
    "SPEAKER_02",
    evidence["speaker"]
)

correct_time_result = check_timestamp_consistency(
    evidence["start"],
    evidence["end"]
)


# ------------------------------------------------------------
# Test 2 — Incorrect speaker
# ------------------------------------------------------------

wrong_speaker_result = check_speaker_consistency(
    "SPEAKER_03",
    evidence["speaker"]
)


# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

print("=" * 80)
print("SPEAKER & TIMESTAMP CONSISTENCY TEST")
print("=" * 80)

print("\nEvidence:")
print(f"Evidence ID : {evidence['evidence_id']}")
print(f"Speaker     : {evidence['speaker']}")
print(
    f"Timestamp   : "
    f"{evidence['start']:.3f} → {evidence['end']:.3f}"
)
print(f"Text        : {evidence['text']}")


print("\n" + "-" * 80)
print("TEST 1 — CORRECT SPEAKER")
print("-" * 80)

print("Claim speaker   : SPEAKER_02")
print("Evidence speaker:", evidence["speaker"])
print("Result          :", correct_speaker_result)


print("\n" + "-" * 80)
print("TEST 2 — INCORRECT SPEAKER")
print("-" * 80)

print("Claim speaker   : SPEAKER_03")
print("Evidence speaker:", evidence["speaker"])
print("Result          :", wrong_speaker_result)


print("\n" + "-" * 80)
print("TIMESTAMP")
print("-" * 80)

print("Result:", correct_time_result)

SPEAKER & TIMESTAMP CONSISTENCY TEST

Evidence:
Evidence ID : 15
Speaker     : SPEAKER_02
Timestamp   : 122.503 → 132.290
Text        : We want it to be original, something that people haven't thought of. It's not out in the shops. Trendy, appealing to a wide market, but,

--------------------------------------------------------------------------------
TEST 1 — CORRECT SPEAKER
--------------------------------------------------------------------------------
Claim speaker   : SPEAKER_02
Evidence speaker: SPEAKER_02
Result          : {'applicable': True, 'passed': True, 'reason': 'Speaker matches evidence.'}

--------------------------------------------------------------------------------
TEST 2 — INCORRECT SPEAKER
--------------------------------------------------------------------------------
Claim speaker   : SPEAKER_03
Evidence speaker: SPEAKER_02
Result          : {'applicable': True, 'passed': False, 'reason': 'Speaker does not match evidence.'}

------------------------------------

In [26]:
# ============================================================
# M7.12 — Attribute Verification Result Structure
# ============================================================
#
# Purpose:
#   Define a standardized structure for storing verification
#   results for every MoM attribute.
#
# This structure will later be used by:
#
#   M7  → Evidence Verification
#   M8  → Verified MoM Generation
#   M9  → Dashboard
#   M11 → Evaluation
#
# No model inference is performed here.
# No existing files are modified.
# ============================================================

from dataclasses import dataclass, asdict
from typing import Optional, List


@dataclass
class AttributeVerificationResult:
    """
    Stores the verification result of one MoM attribute.
    """

    attribute: str

    proposition: str

    evidence_ids: List[int]

    evidence_start: Optional[float]

    evidence_end: Optional[float]

    evidence_speakers: List[str]

    bge_similarity: Optional[float]

    nli_label: Optional[str]

    nli_entailment: Optional[float]

    nli_contradiction: Optional[float]

    nli_neutral: Optional[float]

    lexical_overlap: Optional[float]

    speaker_check: Optional[bool]

    timestamp_check: Optional[bool]

    content_supported: Optional[bool]

    final_status: str

    verification_reason: str


# ------------------------------------------------------------
# Create a small example
# ------------------------------------------------------------

example_result = AttributeVerificationResult(

    attribute="originality",

    proposition=(
        "The remote control is intended to be original."
    ),

    evidence_ids=[15],

    evidence_start=122.503,

    evidence_end=132.290,

    evidence_speakers=["SPEAKER_02"],

    bge_similarity=0.0,

    nli_label="ENTAILMENT",

    nli_entailment=0.9787,

    nli_contradiction=0.0014,

    nli_neutral=0.0199,

    lexical_overlap=0.25,

    speaker_check=True,

    timestamp_check=True,

    content_supported=True,

    final_status="VERIFIED",

    verification_reason=(
        "Evidence context supports the proposition."
    )
)


# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

print("=" * 80)
print("ATTRIBUTE VERIFICATION RESULT STRUCTURE")
print("=" * 80)

print()

for key, value in asdict(example_result).items():
    print(f"{key:25s}: {value}")

ATTRIBUTE VERIFICATION RESULT STRUCTURE

attribute                : originality
proposition              : The remote control is intended to be original.
evidence_ids             : [15]
evidence_start           : 122.503
evidence_end             : 132.29
evidence_speakers        : ['SPEAKER_02']
bge_similarity           : 0.0
nli_label                : ENTAILMENT
nli_entailment           : 0.9787
nli_contradiction        : 0.0014
nli_neutral              : 0.0199
lexical_overlap          : 0.25
speaker_check            : True
timestamp_check          : True
content_supported        : True
final_status             : VERIFIED
verification_reason      : Evidence context supports the proposition.


In [27]:
# ============================================================
# M7.13 — Integrated Attribute Evidence Analysis
# ============================================================
#
# Purpose:
#   Combine the existing verification components for ONE
#   attribute:
#
#       1. BGE + FAISS retrieval
#       2. Context expansion
#       3. Contextual NLI
#       4. Lexical support
#       5. Speaker consistency
#       6. Evidence timestamps
#
# IMPORTANT:
#   This cell intentionally DOES NOT make the final
#   VERIFIED / FLAGGED decision.
#
#   It produces all relevant signals first.
#   We will use real results to design the final
#   aggregation rule.
#
# No existing files are modified.
# ============================================================


def analyze_attribute_evidence(
    proposition,
    claim_speaker=None,
    top_k=10,
    context_window=2
):
    """
    Analyze evidence for one MoM attribute.

    Parameters
    ----------
    proposition : str
        Attribute proposition to verify.

    claim_speaker : str or None
        Speaker specified by the generated MoM claim.

    top_k : int
        Number of BGE evidence candidates to retrieve.

    context_window : int
        Number of neighboring utterances on each side.

    Returns
    -------
    dict
        Retrieval and verification signals.
    """

    # --------------------------------------------------------
    # 1. Retrieve candidate evidence using BGE + FAISS
    # --------------------------------------------------------

    retrieved = retrieve_evidence(
        proposition,
        top_k=top_k
    )

    if len(retrieved) == 0:
        return {
            "proposition": proposition,
            "claim_speaker": claim_speaker,
            "retrieved_candidates": [],
            "message": "No evidence candidates retrieved."
        }


    # --------------------------------------------------------
    # 2. Analyze each retrieved candidate
    # --------------------------------------------------------

    analyzed_candidates = []


    for candidate in retrieved:

        evidence_id = int(candidate["evidence_id"])

        # Find the actual evidence document
        evidence_doc = next(
            (
                doc for doc in evidence_documents
                if int(doc["evidence_id"]) == evidence_id
            ),
            None
        )

        if evidence_doc is None:
            continue


        # ----------------------------------------------------
        # Context expansion
        # ----------------------------------------------------

        context_docs = get_context_window(
            evidence_id,
            window_size=context_window
        )

        context_text = " ".join(
            doc["text"]
            for doc in context_docs
        )


        # ----------------------------------------------------
        # Contextual NLI
        # ----------------------------------------------------

        nli_result = run_nli(
            context_text,
            proposition
        )


        # ----------------------------------------------------
        # Lexical support
        # ----------------------------------------------------

        lexical_result = lexical_support(
            proposition,
            context_text
        )


        # ----------------------------------------------------
        # Speaker consistency
        # ----------------------------------------------------

        speaker_result = check_speaker_consistency(
            claim_speaker,
            evidence_doc["speaker"]
        )


        # ----------------------------------------------------
        # Store all signals
        # ----------------------------------------------------

        analyzed_candidates.append({

            "evidence_id": evidence_id,

            "speaker": evidence_doc["speaker"],

            "start": evidence_doc["start"],

            "end": evidence_doc["end"],

            "text": evidence_doc["text"],

            "retrieval_similarity": candidate[
                "retrieval_similarity"
            ],

            "nli_label": nli_result[
                "predicted_label"
            ],

            "nli_entailment": nli_result[
                "entailment_probability"
            ],

            "nli_contradiction": nli_result[
                "contradiction_probability"
            ],

            "nli_neutral": nli_result[
                "neutral_probability"
            ],

            "lexical_overlap": lexical_result[
                "overlap_ratio"
            ],

            "matched_terms": lexical_result[
                "matched_terms"
            ],

            "missing_terms": lexical_result[
                "missing_terms"
            ],

            "speaker_check": speaker_result,

            "context_evidence_ids": [
                int(doc["evidence_id"])
                for doc in context_docs
            ]
        })


    # --------------------------------------------------------
    # 3. Return complete analysis
    # --------------------------------------------------------

    return {

        "proposition": proposition,

        "claim_speaker": claim_speaker,

        "retrieved_candidates": analyzed_candidates
    }


# ============================================================
# Test on the originality attribute
# ============================================================

test_proposition = (
    "The remote control is intended to be original."
)

analysis_result = analyze_attribute_evidence(
    proposition=test_proposition,
    claim_speaker="SPEAKER_02",
    top_k=10,
    context_window=2
)


# ------------------------------------------------------------
# Display compact results
# ------------------------------------------------------------

print("=" * 90)
print("INTEGRATED ATTRIBUTE EVIDENCE ANALYSIS")
print("=" * 90)

print("\nProposition:")
print(analysis_result["proposition"])

print("\nClaim speaker:")
print(analysis_result["claim_speaker"])

print("\n" + "-" * 90)
print("RETRIEVED EVIDENCE CANDIDATES")
print("-" * 90)


for rank, candidate in enumerate(
    analysis_result["retrieved_candidates"],
    start=1
):

    print(
        f"\nRank {rank} | "
        f"Evidence {candidate['evidence_id']}"
    )

    print(
        f"Speaker: {candidate['speaker']} | "
        f"Time: "
        f"{candidate['start']:.3f} → "
        f"{candidate['end']:.3f}"
    )

    print(
        f"BGE similarity: "
        f"{candidate['retrieval_similarity']:.4f}"
    )

    print(
        f"NLI: {candidate['nli_label']} | "
        f"Entailment: "
        f"{candidate['nli_entailment']:.4f} | "
        f"Contradiction: "
        f"{candidate['nli_contradiction']:.4f} | "
        f"Neutral: "
        f"{candidate['nli_neutral']:.4f}"
    )

    print(
        f"Lexical overlap: "
        f"{candidate['lexical_overlap']:.4f}"
    )

    print(
        f"Matched terms: "
        f"{candidate['matched_terms']}"
    )

    print(
        f"Speaker check: "
        f"{candidate['speaker_check']}"
    )

    print(
        f"Context IDs: "
        f"{candidate['context_evidence_ids']}"
    )

    print(
        f"Evidence text: "
        f"{candidate['text']}"
    )


print("\n" + "=" * 90)
print("Analysis complete — no final verification decision made.")
print("=" * 90)

INTEGRATED ATTRIBUTE EVIDENCE ANALYSIS

Proposition:
The remote control is intended to be original.

Claim speaker:
SPEAKER_02

------------------------------------------------------------------------------------------
RETRIEVED EVIDENCE CANDIDATES
------------------------------------------------------------------------------------------

Rank 1 | Evidence 105
Speaker: SPEAKER_01 | Time: 707.248 → 710.889
BGE similarity: 0.7917
NLI: NEUTRAL | Entailment: 0.0013 | Contradiction: 0.0053 | Neutral: 0.9934
Lexical overlap: 0.5000
Matched terms: ['remote', 'control']
Speaker check: {'applicable': True, 'passed': False, 'reason': 'Speaker does not match evidence.'}
Context IDs: [103, 104, 105, 106, 107]
Evidence text: remote controls. You want to integrate everything into one.

Rank 2 | Evidence 14
Speaker: SPEAKER_02 | Time: 117.699 → 120.921
BGE similarity: 0.7810
NLI: ENTAILMENT | Entailment: 0.9812 | Contradiction: 0.0012 | Neutral: 0.0177
Lexical overlap: 0.7500
Matched terms: ['remote'

In [28]:
# ============================================================
# M7.14 — Evidence Candidate Selection
# ============================================================
#
# Purpose:
#   Select plausible supporting evidence from the integrated
#   evidence analysis.
#
# Selection principles:
#
#   1. Contradiction is not supporting evidence.
#   2. Neutral evidence is not treated as support.
#   3. Entailment is the strongest semantic signal.
#   4. Lexical overlap provides additional support.
#   5. Speaker mismatch is rejected when a speaker is claimed.
#   6. BGE similarity is used for retrieval, not proof.
#
# IMPORTANT:
#   No arbitrary numerical weighting is used yet.
#   No final VERIFIED / FLAGGED decision is made.
# ============================================================


def select_supporting_candidates(
    analysis_result,
    min_lexical_overlap=0.0
):
    """
    Select plausible supporting evidence candidates.

    Parameters
    ----------
    analysis_result : dict
        Output from analyze_attribute_evidence().

    min_lexical_overlap : float
        Optional minimum lexical overlap.
        Default is 0 because lexical overlap is only
        a supporting signal and not mandatory for
        semantic entailment.

    Returns
    -------
    list
        Supporting evidence candidates.
    """

    candidates = analysis_result.get(
        "retrieved_candidates",
        []
    )

    supporting = []
    rejected = []


    for candidate in candidates:

        nli_label = candidate["nli_label"]

        speaker_check = candidate["speaker_check"]

        lexical_overlap = candidate["lexical_overlap"]


        # ----------------------------------------------------
        # Reject explicit contradiction
        # ----------------------------------------------------

        if nli_label == "CONTRADICTION":

            rejected.append({
                "evidence_id": candidate["evidence_id"],
                "reason": "NLI contradiction"
            })

            continue


        # ----------------------------------------------------
        # Reject neutral evidence
        # ----------------------------------------------------

        if nli_label == "NEUTRAL":

            rejected.append({
                "evidence_id": candidate["evidence_id"],
                "reason": "NLI neutral"
            })

            continue


        # ----------------------------------------------------
        # Require NLI entailment
        # ----------------------------------------------------

        if nli_label != "ENTAILMENT":

            rejected.append({
                "evidence_id": candidate["evidence_id"],
                "reason": "No semantic support"
            })

            continue


        # ----------------------------------------------------
        # Speaker consistency
        # ----------------------------------------------------

        if (
            speaker_check["applicable"]
            and not speaker_check["passed"]
        ):

            rejected.append({
                "evidence_id": candidate["evidence_id"],
                "reason": "Speaker mismatch"
            })

            continue


        # ----------------------------------------------------
        # Optional lexical condition
        # ----------------------------------------------------

        if lexical_overlap < min_lexical_overlap:

            rejected.append({
                "evidence_id": candidate["evidence_id"],
                "reason": "Insufficient lexical support"
            })

            continue


        # ----------------------------------------------------
        # Candidate accepted
        # ----------------------------------------------------

        supporting.append(candidate)


    # --------------------------------------------------------
    # Sort supporting candidates
    #
    # Stronger NLI support first.
    # BGE similarity is used as a secondary ordering signal.
    # --------------------------------------------------------

    supporting = sorted(
        supporting,
        key=lambda x: (
            x["nli_entailment"],
            x["retrieval_similarity"]
        ),
        reverse=True
    )


    return {
        "supporting_candidates": supporting,
        "rejected_candidates": rejected
    }


# ============================================================
# Test using the originality attribute
# ============================================================

selection_result = select_supporting_candidates(
    analysis_result
)


# ------------------------------------------------------------
# Display supporting candidates
# ------------------------------------------------------------

print("=" * 90)
print("SUPPORTING EVIDENCE CANDIDATES")
print("=" * 90)

for rank, candidate in enumerate(
    selection_result["supporting_candidates"],
    start=1
):

    print(
        f"\nRank {rank} | "
        f"Evidence {candidate['evidence_id']}"
    )

    print(
        f"BGE similarity : "
        f"{candidate['retrieval_similarity']:.4f}"
    )

    print(
        f"NLI entailment : "
        f"{candidate['nli_entailment']:.4f}"
    )

    print(
        f"Lexical overlap: "
        f"{candidate['lexical_overlap']:.4f}"
    )

    print(
        f"Speaker check  : "
        f"{candidate['speaker_check']}"
    )

    print(
        f"Time           : "
        f"{candidate['start']:.3f} → "
        f"{candidate['end']:.3f}"
    )


# ------------------------------------------------------------
# Display rejected candidates
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("REJECTED EVIDENCE CANDIDATES")
print("=" * 90)

for item in selection_result["rejected_candidates"]:

    print(
        f"Evidence {item['evidence_id']} "
        f"→ {item['reason']}"
    )


print("\n" + "=" * 90)
print("Candidate selection complete.")
print("No final VERIFIED / FLAGGED decision made.")
print("=" * 90)

SUPPORTING EVIDENCE CANDIDATES

Rank 1 | Evidence 14
BGE similarity : 0.7810
NLI entailment : 0.9812
Lexical overlap: 0.7500
Speaker check  : {'applicable': True, 'passed': True, 'reason': 'Speaker matches evidence.'}
Time           : 117.699 → 120.921

Rank 2 | Evidence 15
BGE similarity : 0.6708
NLI entailment : 0.9787
Lexical overlap: 0.7500
Speaker check  : {'applicable': True, 'passed': True, 'reason': 'Speaker matches evidence.'}
Time           : 122.503 → 132.290

REJECTED EVIDENCE CANDIDATES
Evidence 105 → NLI neutral
Evidence 107 → NLI neutral
Evidence 145 → NLI neutral
Evidence 102 → NLI neutral
Evidence 135 → NLI neutral
Evidence 117 → NLI neutral
Evidence 112 → NLI neutral
Evidence 134 → NLI neutral

Candidate selection complete.
No final VERIFIED / FLAGGED decision made.


In [29]:
# ============================================================
# M7.15 — Best Supporting Evidence Context
# ============================================================
#
# Purpose:
#   Select the strongest supporting evidence context for
#   one MoM attribute.
#
# The selected unit consists of:
#
#   - Primary evidence utterance
#   - Its surrounding context
#   - NLI support
#   - Lexical support
#   - Speaker consistency
#
# The primary utterance provides the citation timestamp.
# The context provides conversational meaning.
#
# No final VERIFIED / FLAGGED decision is made yet.
# ============================================================


def select_best_evidence_context(selection_result):
    """
    Select the strongest supporting evidence candidate.

    Ranking priority:
        1. NLI entailment
        2. BGE retrieval similarity
        3. Lexical overlap

    Returns
    -------
    dict or None
        Best evidence context.
    """

    candidates = selection_result.get(
        "supporting_candidates",
        []
    )

    if not candidates:
        return None


    # --------------------------------------------------------
    # Candidates are already ordered primarily by NLI
    # entailment and secondarily by BGE similarity.
    # --------------------------------------------------------

    best_candidate = candidates[0]


    # --------------------------------------------------------
    # Retrieve the complete context
    # --------------------------------------------------------

    context_docs = get_context_window(
        best_candidate["evidence_id"],
        window_size=2
    )


    # --------------------------------------------------------
    # Context timestamps
    # --------------------------------------------------------

    context_start = min(
        doc["start"]
        for doc in context_docs
    )

    context_end = max(
        doc["end"]
        for doc in context_docs
    )


    # --------------------------------------------------------
    # Return structured evidence context
    # --------------------------------------------------------

    return {

        "primary_evidence_id":
            best_candidate["evidence_id"],

        "primary_start":
            best_candidate["start"],

        "primary_end":
            best_candidate["end"],

        "primary_speaker":
            best_candidate["speaker"],

        "primary_text":
            best_candidate["text"],

        "context_evidence_ids": [
            int(doc["evidence_id"])
            for doc in context_docs
        ],

        "context_start":
            context_start,

        "context_end":
            context_end,

        "context_speakers": list(
            dict.fromkeys(
                doc["speaker"]
                for doc in context_docs
            )
        ),

        "context_text": " ".join(
            doc["text"]
            for doc in context_docs
        ),

        "bge_similarity":
            best_candidate["retrieval_similarity"],

        "nli_label":
            best_candidate["nli_label"],

        "nli_entailment":
            best_candidate["nli_entailment"],

        "nli_contradiction":
            best_candidate["nli_contradiction"],

        "nli_neutral":
            best_candidate["nli_neutral"],

        "lexical_overlap":
            best_candidate["lexical_overlap"],

        "matched_terms":
            best_candidate["matched_terms"],

        "speaker_check":
            best_candidate["speaker_check"]
    }


# ============================================================
# Test using the originality analysis
# ============================================================

best_evidence = select_best_evidence_context(
    selection_result
)


print("=" * 90)
print("BEST SUPPORTING EVIDENCE CONTEXT")
print("=" * 90)


if best_evidence is None:

    print("\nNo supporting evidence was found.")

else:

    print(
        f"\nPrimary Evidence ID : "
        f"{best_evidence['primary_evidence_id']}"
    )

    print(
        f"Primary Timestamp   : "
        f"{best_evidence['primary_start']:.3f} → "
        f"{best_evidence['primary_end']:.3f}"
    )

    print(
        f"Primary Speaker     : "
        f"{best_evidence['primary_speaker']}"
    )

    print(
        f"\nContext Evidence IDs: "
        f"{best_evidence['context_evidence_ids']}"
    )

    print(
        f"Context Timestamp   : "
        f"{best_evidence['context_start']:.3f} → "
        f"{best_evidence['context_end']:.3f}"
    )

    print(
        f"Context Speakers    : "
        f"{best_evidence['context_speakers']}"
    )

    print(
        f"\nBGE Similarity      : "
        f"{best_evidence['bge_similarity']:.4f}"
    )

    print(
        f"NLI Label           : "
        f"{best_evidence['nli_label']}"
    )

    print(
        f"NLI Entailment      : "
        f"{best_evidence['nli_entailment']:.4f}"
    )

    print(
        f"Lexical Overlap     : "
        f"{best_evidence['lexical_overlap']:.4f}"
    )

    print(
        f"Speaker Check       : "
        f"{best_evidence['speaker_check']}"
    )

    print("\nPrimary Evidence Text:")
    print(best_evidence["primary_text"])

    print("\nFull Context:")
    print(best_evidence["context_text"])


print("\n" + "=" * 90)
print("Best evidence context selection complete.")
print("=" * 90)

BEST SUPPORTING EVIDENCE CONTEXT

Primary Evidence ID : 14
Primary Timestamp   : 117.699 → 120.921
Primary Speaker     : SPEAKER_02

Context Evidence IDs: [12, 13, 14, 15, 16]
Context Timestamp   : 106.340 → 138.174
Context Speakers    : ['SPEAKER_02']

BGE Similarity      : 0.7810
NLI Label           : ENTAILMENT
NLI Entailment      : 0.9812
Lexical Overlap     : 0.7500
Speaker Check       : {'applicable': True, 'passed': True, 'reason': 'Speaker matches evidence.'}

Primary Evidence Text:
Now, we're developing a remote control, which you probably already know.

Full Context:
talk about the project plan, discuss our own ideas and everything. And we've got 25 minutes to do that as far as I can understand. Now, we're developing a remote control, which you probably already know. We want it to be original, something that people haven't thought of. It's not out in the shops. Trendy, appealing to a wide market, but, you know, not a hunk of metal. And user-friendly, grannies to kids,

Best e

In [30]:
# ============================================================
# M7.16 — Direct Evidence Identification
# ============================================================
#
# Purpose:
#   Within a selected contextual evidence window, identify
#   the individual utterance that is most directly related
#   to the proposition.
#
# Important distinction:
#
#   Context window
#       → used for understanding the conversation
#
#   Direct evidence utterance
#       → used for the primary citation/timestamp
#
# This prevents a contextual NLI match from producing a
# misleading primary timestamp.
#
# No existing files are modified.
# ============================================================


def identify_direct_evidence(
    proposition,
    context_docs
):
    """
    Identify the most directly relevant utterance inside
    a contextual evidence window.

    Uses BGE semantic similarity between the proposition
    and each individual utterance.

    Returns
    -------
    dict or None
        Direct evidence document and similarity.
    """

    if not context_docs:
        return None


    # --------------------------------------------------------
    # Extract individual evidence texts
    # --------------------------------------------------------

    texts = [
        doc["text"]
        for doc in context_docs
    ]


    # --------------------------------------------------------
    # Generate proposition embedding
    # --------------------------------------------------------

    proposition_embedding = bge_model.encode(
        [proposition],
        normalize_embeddings=True,
        convert_to_numpy=True
    )


    # --------------------------------------------------------
    # Generate individual evidence embeddings
    # --------------------------------------------------------

    evidence_embeddings = bge_model.encode(
        texts,
        normalize_embeddings=True,
        convert_to_numpy=True
    )


    # --------------------------------------------------------
    # Calculate cosine similarities
    # --------------------------------------------------------

    similarities = (
        evidence_embeddings
        @ proposition_embedding[0]
    )


    # --------------------------------------------------------
    # Find most relevant utterance
    # --------------------------------------------------------

    best_index = int(
        similarities.argmax()
    )

    best_doc = context_docs[best_index]

    best_similarity = float(
        similarities[best_index]
    )


    return {
        "evidence_id": int(
            best_doc["evidence_id"]
        ),

        "speaker": best_doc["speaker"],

        "start": best_doc["start"],

        "end": best_doc["end"],

        "text": best_doc["text"],

        "similarity": best_similarity
    }


# ============================================================
# Test using the selected originality context
# ============================================================

proposition = (
    "The remote control is intended to be original."
)


direct_evidence = identify_direct_evidence(
    proposition,
    context
)


print("=" * 90)
print("DIRECT EVIDENCE IDENTIFICATION")
print("=" * 90)


if direct_evidence is None:

    print("\nNo direct evidence identified.")

else:

    print(
        f"\nEvidence ID : "
        f"{direct_evidence['evidence_id']}"
    )

    print(
        f"Speaker     : "
        f"{direct_evidence['speaker']}"
    )

    print(
        f"Timestamp   : "
        f"{direct_evidence['start']:.3f} → "
        f"{direct_evidence['end']:.3f}"
    )

    print(
        f"BGE Similarity: "
        f"{direct_evidence['similarity']:.4f}"
    )

    print("\nEvidence Text:")
    print(direct_evidence["text"])


print("\n" + "=" * 90)
print("Direct evidence identification complete.")
print("=" * 90)

DIRECT EVIDENCE IDENTIFICATION

Evidence ID : 14
Speaker     : SPEAKER_02
Timestamp   : 117.699 → 120.921
BGE Similarity: 0.7810

Evidence Text:
Now, we're developing a remote control, which you probably already know.

Direct evidence identification complete.


In [31]:
# ============================================================
# M7.17 — Direct Evidence Signal Diagnostic
# ============================================================
#
# Purpose:
#   Compare multiple evidence signals for every utterance
#   inside the selected context window.
#
# Signals:
#   1. BGE semantic similarity
#   2. Lexical overlap
#   3. NLI entailment
#   4. NLI label
#
# This is diagnostic only.
# No final ranking formula is introduced yet.
# ============================================================


def diagnose_context_evidence(
    proposition,
    context_docs
):
    """
    Calculate evidence-support signals for each utterance
    inside a contextual evidence window.
    """

    if not context_docs:
        return []


    # --------------------------------------------------------
    # BGE embeddings for proposition and utterances
    # --------------------------------------------------------

    proposition_embedding = bge_model.encode(
        [proposition],
        normalize_embeddings=True,
        convert_to_numpy=True
    )

    evidence_texts = [
        doc["text"]
        for doc in context_docs
    ]

    evidence_embeddings = bge_model.encode(
        evidence_texts,
        normalize_embeddings=True,
        convert_to_numpy=True
    )

    similarities = (
        evidence_embeddings
        @ proposition_embedding[0]
    )


    # --------------------------------------------------------
    # Analyze every utterance
    # --------------------------------------------------------

    results = []


    for i, doc in enumerate(context_docs):

        text = doc["text"]


        # Lexical support
        lexical_result = lexical_support(
            proposition,
            text
        )


        # NLI
        nli_result = run_nli(
            text,
            proposition
        )


        results.append({

            "evidence_id":
                int(doc["evidence_id"]),

            "speaker":
                doc["speaker"],

            "start":
                doc["start"],

            "end":
                doc["end"],

            "text":
                text,

            "bge_similarity":
                float(similarities[i]),

            "lexical_overlap":
                lexical_result["overlap_ratio"],

            "matched_terms":
                lexical_result["matched_terms"],

            "nli_label":
                nli_result["predicted_label"],

            "nli_entailment":
                nli_result["entailment_probability"],

            "nli_contradiction":
                nli_result["contradiction_probability"],

            "nli_neutral":
                nli_result["neutral_probability"]
        })


    return results


# ============================================================
# Run diagnostic on originality
# ============================================================

proposition = (
    "The remote control is intended to be original."
)

diagnostic_results = diagnose_context_evidence(
    proposition,
    context
)


# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

print("=" * 100)
print("DIRECT EVIDENCE SIGNAL DIAGNOSTIC")
print("=" * 100)

print("\nProposition:")
print(proposition)


for result in diagnostic_results:

    print("\n" + "-" * 100)

    print(
        f"Evidence {result['evidence_id']} | "
        f"{result['speaker']} | "
        f"{result['start']:.3f} → "
        f"{result['end']:.3f}"
    )

    print(
        f"BGE similarity     : "
        f"{result['bge_similarity']:.4f}"
    )

    print(
        f"Lexical overlap    : "
        f"{result['lexical_overlap']:.4f}"
    )

    print(
        f"Matched terms      : "
        f"{result['matched_terms']}"
    )

    print(
        f"NLI label          : "
        f"{result['nli_label']}"
    )

    print(
        f"NLI entailment     : "
        f"{result['nli_entailment']:.4f}"
    )

    print(
        f"NLI contradiction  : "
        f"{result['nli_contradiction']:.4f}"
    )

    print(
        f"NLI neutral        : "
        f"{result['nli_neutral']:.4f}"
    )

    print("\nText:")
    print(result["text"])


print("\n" + "=" * 100)
print("Diagnostic complete.")
print("=" * 100)

DIRECT EVIDENCE SIGNAL DIAGNOSTIC

Proposition:
The remote control is intended to be original.

----------------------------------------------------------------------------------------------------
Evidence 13 | SPEAKER_02 | 112.185 → 115.748
BGE similarity     : 0.4592
Lexical overlap    : 0.0000
Matched terms      : []
NLI label          : NEUTRAL
NLI entailment     : 0.0002
NLI contradiction  : 0.0009
NLI neutral        : 0.9989

Text:
And we've got 25 minutes to do that as far as I can understand.

----------------------------------------------------------------------------------------------------
Evidence 14 | SPEAKER_02 | 117.699 → 120.921
BGE similarity     : 0.7810
Lexical overlap    : 0.5000
Matched terms      : ['remote', 'control']
NLI label          : NEUTRAL
NLI entailment     : 0.0012
NLI contradiction  : 0.0112
NLI neutral        : 0.9876

Text:
Now, we're developing a remote control, which you probably already know.

------------------------------------------------------

In [32]:
# ============================================================
# M7.18 — Evidence Group Representation
# ============================================================
#
# Purpose:
#   Represent supporting evidence as a GROUP of utterances
#   rather than forcing every attribute to have one evidence
#   sentence.
#
# Three levels are maintained:
#
#   1. Supporting evidence IDs
#      → utterances directly contributing to the claim
#
#   2. Context evidence IDs
#      → surrounding utterances used for interpretation
#
#   3. Display interval
#      → timestamp range shown to the user
#
# This structure will later be used by:
#
#   M7 → Verification
#   M8 → Verified MoM
#   M9 → Dashboard
# ============================================================


def build_evidence_group(
    supporting_evidence_ids,
    context_evidence_ids
):
    """
    Build a structured evidence group.

    Parameters
    ----------
    supporting_evidence_ids : list[int]
        Evidence utterances directly supporting the attribute.

    context_evidence_ids : list[int]
        Surrounding evidence used to interpret the support.

    Returns
    -------
    dict
        Structured evidence group.
    """

    # --------------------------------------------------------
    # Retrieve supporting documents
    # --------------------------------------------------------

    supporting_docs = [
        doc
        for doc in evidence_documents
        if int(doc["evidence_id"])
        in [int(x) for x in supporting_evidence_ids]
    ]


    # --------------------------------------------------------
    # Retrieve context documents
    # --------------------------------------------------------

    context_docs = [
        doc
        for doc in evidence_documents
        if int(doc["evidence_id"])
        in [int(x) for x in context_evidence_ids]
    ]


    # --------------------------------------------------------
    # Sort chronologically
    # --------------------------------------------------------

    supporting_docs = sorted(
        supporting_docs,
        key=lambda x: x["start"]
    )

    context_docs = sorted(
        context_docs,
        key=lambda x: x["start"]
    )


    # --------------------------------------------------------
    # Determine display interval
    # --------------------------------------------------------

    all_display_docs = (
        supporting_docs
        if supporting_docs
        else context_docs
    )

    if all_display_docs:

        display_start = min(
            doc["start"]
            for doc in all_display_docs
        )

        display_end = max(
            doc["end"]
            for doc in all_display_docs
        )

    else:

        display_start = None
        display_end = None


    # --------------------------------------------------------
    # Build result
    # --------------------------------------------------------

    return {

        "supporting_evidence_ids": [
            int(doc["evidence_id"])
            for doc in supporting_docs
        ],

        "supporting_evidence": [
            {
                "evidence_id": int(doc["evidence_id"]),
                "speaker": doc["speaker"],
                "start": doc["start"],
                "end": doc["end"],
                "text": doc["text"]
            }
            for doc in supporting_docs
        ],

        "context_evidence_ids": [
            int(doc["evidence_id"])
            for doc in context_docs
        ],

        "context_start": display_start,

        "context_end": display_end,

        "context_text": " ".join(
            doc["text"]
            for doc in context_docs
        ),

        "speakers": list(
            dict.fromkeys(
                doc["speaker"]
                for doc in context_docs
            )
        )
    }


# ============================================================
# Test using the originality example
# ============================================================

evidence_group = build_evidence_group(
    supporting_evidence_ids=[14, 15],
    context_evidence_ids=[13, 14, 15, 16, 17]
)


print("=" * 90)
print("EVIDENCE GROUP")
print("=" * 90)

print(
    "\nSupporting evidence IDs:",
    evidence_group["supporting_evidence_ids"]
)

print(
    "Context evidence IDs:",
    evidence_group["context_evidence_ids"]
)

print(
    "Display interval:",
    f"{evidence_group['context_start']:.3f}"
    f" → "
    f"{evidence_group['context_end']:.3f}"
)

print(
    "Speakers:",
    evidence_group["speakers"]
)


print("\nSupporting evidence:")

for item in evidence_group["supporting_evidence"]:

    print(
        f"\nEvidence {item['evidence_id']} | "
        f"{item['speaker']} | "
        f"{item['start']:.3f} → "
        f"{item['end']:.3f}"
    )

    print(item["text"])


print("\nContext text:")
print(evidence_group["context_text"])


print("\n" + "=" * 90)
print("Evidence group construction complete.")
print("=" * 90)

EVIDENCE GROUP

Supporting evidence IDs: [14, 15]
Context evidence IDs: [13, 14, 15, 16, 17]
Display interval: 117.699 → 132.290
Speakers: ['SPEAKER_02']

Supporting evidence:

Evidence 14 | SPEAKER_02 | 117.699 → 120.921
Now, we're developing a remote control, which you probably already know.

Evidence 15 | SPEAKER_02 | 122.503 → 132.290
We want it to be original, something that people haven't thought of. It's not out in the shops. Trendy, appealing to a wide market, but,

Context text:
And we've got 25 minutes to do that as far as I can understand. Now, we're developing a remote control, which you probably already know. We want it to be original, something that people haven't thought of. It's not out in the shops. Trendy, appealing to a wide market, but, you know, not a hunk of metal. And user-friendly, grannies to kids, maybe even pooches, should be able to use it.

Evidence group construction complete.


In [33]:
# ============================================================
# M7 - Reusable Evidence Group Builder
# ============================================================
# Purpose:
#   Build a structured evidence group for one verified
#   proposition/attribute.
#
# The group distinguishes:
#   1. Supporting evidence -> directly contributes to the claim
#   2. Context evidence    -> surrounding utterances for interpretation
#   3. Display interval    -> timestamp range shown to the user
# ============================================================

def create_evidence_group(
    supporting_evidence_ids,
    context_evidence_ids
):
    """
    Create a structured evidence group from evidence IDs.

    Parameters
    ----------
    supporting_evidence_ids : list
        Evidence IDs that directly support the proposition.

    context_evidence_ids : list
        Evidence IDs used as surrounding conversational context.

    Returns
    -------
    dict
        Structured evidence group.
    """

    supporting_ids = [int(x) for x in supporting_evidence_ids]
    context_ids = [int(x) for x in context_evidence_ids]

    # Lookup evidence documents
    evidence_lookup = {
        int(doc["evidence_id"]): doc
        for doc in evidence_documents
    }

    # Retrieve supporting evidence
    supporting_docs = [
        evidence_lookup[eid]
        for eid in supporting_ids
        if eid in evidence_lookup
    ]

    # Retrieve context evidence
    context_docs = [
        evidence_lookup[eid]
        for eid in context_ids
        if eid in evidence_lookup
    ]

    # Sort chronologically
    supporting_docs = sorted(
        supporting_docs,
        key=lambda x: x["start"]
    )

    context_docs = sorted(
        context_docs,
        key=lambda x: x["start"]
    )

    # Display interval should cover the supporting evidence.
    # If no supporting evidence exists, fall back to context.
    display_docs = supporting_docs if supporting_docs else context_docs

    if display_docs:
        display_start = min(
            doc["start"] for doc in display_docs
        )
        display_end = max(
            doc["end"] for doc in display_docs
        )
    else:
        display_start = None
        display_end = None

    # Unique speakers from supporting evidence
    speakers = sorted(
        set(
            doc["speaker"]
            for doc in supporting_docs
            if doc.get("speaker") is not None
        )
    )

    # Combined context text
    context_text = " ".join(
        doc["text"].strip()
        for doc in context_docs
    )

    return {
        "supporting_evidence_ids": [
            int(doc["evidence_id"])
            for doc in supporting_docs
        ],

        "supporting_evidence": supporting_docs,

        "context_evidence_ids": [
            int(doc["evidence_id"])
            for doc in context_docs
        ],

        "context_start": display_start,
        "context_end": display_end,

        "context_text": context_text,

        "speakers": speakers
    }


# ------------------------------------------------------------
# Test the reusable function with the originality example
# ------------------------------------------------------------

test_group = create_evidence_group(
    supporting_evidence_ids=[14, 15],
    context_evidence_ids=[13, 14, 15, 16, 17]
)

print("=" * 90)
print("REUSABLE EVIDENCE GROUP TEST")
print("=" * 90)

print(f"Supporting IDs : {test_group['supporting_evidence_ids']}")
print(f"Context IDs    : {test_group['context_evidence_ids']}")
print(
    f"Display interval: "
    f"{test_group['context_start']:.3f} → "
    f"{test_group['context_end']:.3f}"
)
print(f"Speakers       : {test_group['speakers']}")

print("\nEvidence group function is ready.")

REUSABLE EVIDENCE GROUP TEST
Supporting IDs : [14, 15]
Context IDs    : [13, 14, 15, 16, 17]
Display interval: 117.699 → 132.290
Speakers       : ['SPEAKER_02']

Evidence group function is ready.


In [34]:
# ============================================================
# M7 - Attribute-Level Evidence Verification
# ============================================================
# Purpose:
#   Verify one MoM proposition using:
#   BGE retrieval + contextual NLI + lexical support
#   + speaker consistency.
#
# Important:
#   BGE similarity is used for retrieval, NOT as proof.
#   NLI is evaluated on conversational context because a
#   proposition may require information from multiple utterances.
# ============================================================

def verify_attribute(
    attribute,
    proposition,
    claim_speaker=None,
    top_k=10,
    context_window=2
):
    """
    Verify one atomic proposition.

    Parameters
    ----------
    attribute : str
        Attribute name, e.g. "originality".

    proposition : str
        Atomic proposition to verify.

    claim_speaker : str or None
        Expected speaker if the proposition contains a
        speaker attribution.

    top_k : int
        Number of BGE candidates to inspect.

    context_window : int
        Number of neighboring utterances on each side.

    Returns
    -------
    dict
        Verification result.
    """

    # --------------------------------------------------------
    # Step 1: Retrieve candidate evidence using BGE
    # --------------------------------------------------------

    analysis = analyze_attribute_evidence(
        proposition=proposition,
        claim_speaker=claim_speaker,
        top_k=top_k,
        context_window=context_window
    )

    # --------------------------------------------------------
    # Step 2: Select candidates supported by contextual NLI
    # --------------------------------------------------------

    supporting_candidates = select_supporting_candidates(
        analysis,
        min_lexical_overlap=0.0
    )

    # --------------------------------------------------------
    # Step 3: No supporting evidence found
    # --------------------------------------------------------

    if not supporting_candidates:

        return {
            "attribute": attribute,
            "proposition": proposition,
            "status": "FLAGGED",
            "reason": "No supporting evidence found.",
            "supporting_evidence_ids": [],
            "context_evidence_ids": [],
            "display_start": None,
            "display_end": None,
            "speakers": [],
            "candidates_checked": len(analysis)
        }

    # --------------------------------------------------------
    # Step 4: Keep supporting candidates
    # --------------------------------------------------------

    supporting_ids = [
        int(item["evidence_id"])
        for item in supporting_candidates
    ]

    # --------------------------------------------------------
    # Step 5: Build context around all supporting evidence
    # --------------------------------------------------------

    context_ids = set()

    for item in supporting_candidates:

        item_context = item.get("context_ids", [])

        for eid in item_context:
            context_ids.add(int(eid))

    context_ids = sorted(context_ids)

    # --------------------------------------------------------
    # Step 6: Build final evidence group
    # --------------------------------------------------------

    evidence_group = create_evidence_group(
        supporting_evidence_ids=supporting_ids,
        context_evidence_ids=context_ids
    )

    # --------------------------------------------------------
    # Step 7: Speaker consistency
    # --------------------------------------------------------

    speaker_check = True

    if claim_speaker is not None:

        evidence_speakers = evidence_group["speakers"]

        if evidence_speakers:

            speaker_check = (
                claim_speaker in evidence_speakers
            )

    # --------------------------------------------------------
    # Step 8: Final status
    # --------------------------------------------------------

    if speaker_check:
        status = "VERIFIED"
        reason = (
            "The proposition is supported by retrieved "
            "evidence and contextual NLI, with speaker "
            "consistency satisfied."
        )
    else:
        status = "FLAGGED"
        reason = (
            "The proposition has supporting textual evidence, "
            "but the claimed speaker does not match the "
            "supporting evidence."
        )

    # --------------------------------------------------------
    # Step 9: Return structured verification result
    # --------------------------------------------------------

    return {
        "attribute": attribute,
        "proposition": proposition,

        "status": status,
        "reason": reason,

        "supporting_evidence_ids":
            evidence_group["supporting_evidence_ids"],

        "context_evidence_ids":
            evidence_group["context_evidence_ids"],

        "display_start":
            evidence_group["context_start"],

        "display_end":
            evidence_group["context_end"],

        "speakers":
            evidence_group["speakers"],

        "speaker_check":
            speaker_check,

        "candidates_checked":
            len(analysis),

        "supporting_candidates":
            supporting_candidates
    }


# ============================================================
# Test with the originality attribute
# ============================================================

test_verification = verify_attribute(
    attribute="originality",
    proposition="The remote control is intended to be original.",
    claim_speaker="SPEAKER_02",
    top_k=10,
    context_window=2
)

print("=" * 90)
print("ATTRIBUTE VERIFICATION TEST")
print("=" * 90)

print(f"Attribute       : {test_verification['attribute']}")
print(f"Proposition     : {test_verification['proposition']}")
print(f"Status          : {test_verification['status']}")
print(f"Speaker check   : {test_verification['speaker_check']}")
print(f"Supporting IDs  : {test_verification['supporting_evidence_ids']}")
print(f"Context IDs     : {test_verification['context_evidence_ids']}")
print(
    f"Display interval: "
    f"{test_verification['display_start']:.3f} → "
    f"{test_verification['display_end']:.3f}"
)
print(f"Speakers        : {test_verification['speakers']}")
print(f"Reason          : {test_verification['reason']}")

print("\nSupporting evidence:")
for item in test_verification["supporting_candidates"]:
    print(
        f"  ID {item['evidence_id']} | "
        f"BGE={item['retrieval_similarity']:.4f} | "
        f"NLI={item['nli_label']} "
        f"{item['nli_entailment']:.4f}"
    )

print("\nAttribute verification test complete.")

TypeError: string indices must be integers, not 'str'

In [35]:
# ============================================================
# M7 - Inspect Supporting Candidate Output
# ============================================================
# Purpose:
#   Check the exact structure returned by
#   select_supporting_candidates().
#
# This is a lightweight diagnostic and does NOT rerun
# WhisperX, Pyannote, BGE model loading, or FAISS creation.
# ============================================================

test_analysis = analyze_attribute_evidence(
    proposition="The remote control is intended to be original.",
    claim_speaker="SPEAKER_02",
    top_k=10,
    context_window=2
)

test_supporting = select_supporting_candidates(
    test_analysis,
    min_lexical_overlap=0.0
)

print("=" * 90)
print("SUPPORTING CANDIDATE STRUCTURE")
print("=" * 90)

print("Python type:")
print(type(test_supporting))

print("\nObject:")
print(test_supporting)

print("\nKeys / length:")
try:
    print(len(test_supporting))
except Exception:
    print("Length not available")

if isinstance(test_supporting, dict):
    print("\nDictionary keys:")
    print(list(test_supporting.keys()))

print("\nDiagnostic complete.")

SUPPORTING CANDIDATE STRUCTURE
Python type:
<class 'dict'>

Object:
{'supporting_candidates': [{'evidence_id': 14, 'speaker': 'SPEAKER_02', 'start': 117.699, 'end': 120.921, 'text': "Now, we're developing a remote control, which you probably already know.", 'retrieval_similarity': 0.7810472249984741, 'nli_label': 'ENTAILMENT', 'nli_entailment': 0.9811810255050659, 'nli_contradiction': 0.001153470017015934, 'nli_neutral': 0.017665470018982887, 'lexical_overlap': 0.75, 'matched_terms': ['remote', 'control', 'original'], 'missing_terms': ['intended'], 'speaker_check': {'applicable': True, 'passed': True, 'reason': 'Speaker matches evidence.'}, 'context_evidence_ids': [12, 13, 14, 15, 16]}, {'evidence_id': 15, 'speaker': 'SPEAKER_02', 'start': 122.503, 'end': 132.29, 'text': "We want it to be original, something that people haven't thought of. It's not out in the shops. Trendy, appealing to a wide market, but,", 'retrieval_similarity': 0.6708011627197266, 'nli_label': 'ENTAILMENT', 'nli_en

In [36]:
# ============================================================
# M7 - Corrected Attribute-Level Evidence Verification
# ============================================================
# Purpose:
#   Verify one atomic proposition using:
#   BGE retrieval + contextual NLI + speaker consistency
#   + structured evidence grouping.
#
# No model reloading is required.
# ============================================================

def verify_attribute(
    attribute,
    proposition,
    claim_speaker=None,
    top_k=10,
    context_window=2
):
    """
    Verify one atomic proposition.

    Returns a structured verification result containing:
    - verification status
    - supporting evidence
    - context evidence
    - timestamp interval
    - speaker consistency
    """

    # --------------------------------------------------------
    # 1. Retrieve and analyze candidate evidence
    # --------------------------------------------------------

    analysis = analyze_attribute_evidence(
        proposition=proposition,
        claim_speaker=claim_speaker,
        top_k=top_k,
        context_window=context_window
    )

    # --------------------------------------------------------
    # 2. Select candidates supported by contextual NLI
    # --------------------------------------------------------

    supporting_result = select_supporting_candidates(
        analysis,
        min_lexical_overlap=0.0
    )

    # IMPORTANT:
    # select_supporting_candidates() returns a dictionary.
    supporting_candidates = supporting_result[
        "supporting_candidates"
    ]

    # --------------------------------------------------------
    # 3. No supporting evidence
    # --------------------------------------------------------

    if not supporting_candidates:

        return {
            "attribute": attribute,
            "proposition": proposition,
            "status": "FLAGGED",
            "reason": "No supporting evidence found.",
            "supporting_evidence_ids": [],
            "context_evidence_ids": [],
            "display_start": None,
            "display_end": None,
            "speakers": [],
            "speaker_check": False,
            "candidates_checked": len(analysis),
            "supporting_candidates": [],
            "rejected_candidates":
                supporting_result["rejected_candidates"]
        }

    # --------------------------------------------------------
    # 4. Extract supporting evidence IDs
    # --------------------------------------------------------

    supporting_ids = [
        int(item["evidence_id"])
        for item in supporting_candidates
    ]

    # --------------------------------------------------------
    # 5. Collect context IDs from all supporting candidates
    # --------------------------------------------------------

    context_ids = set()

    for item in supporting_candidates:

        for eid in item.get(
            "context_evidence_ids",
            []
        ):
            context_ids.add(int(eid))

    context_ids = sorted(context_ids)

    # --------------------------------------------------------
    # 6. Build evidence group
    # --------------------------------------------------------

    evidence_group = create_evidence_group(
        supporting_evidence_ids=supporting_ids,
        context_evidence_ids=context_ids
    )

    # --------------------------------------------------------
    # 7. Speaker consistency
    # --------------------------------------------------------

    speaker_check = True

    if claim_speaker is not None:

        evidence_speakers = evidence_group["speakers"]

        if evidence_speakers:

            speaker_check = (
                claim_speaker in evidence_speakers
            )

    # --------------------------------------------------------
    # 8. Determine final verification status
    # --------------------------------------------------------

    if speaker_check:

        status = "VERIFIED"

        reason = (
            "The proposition is supported by contextual "
            "NLI evidence and the claimed speaker is "
            "consistent with the supporting evidence."
        )

    else:

        status = "FLAGGED"

        reason = (
            "Supporting evidence was found, but the "
            "claimed speaker does not match the "
            "supporting evidence."
        )

    # --------------------------------------------------------
    # 9. Return complete result
    # --------------------------------------------------------

    return {
        "attribute": attribute,
        "proposition": proposition,

        "status": status,
        "reason": reason,

        "supporting_evidence_ids":
            evidence_group["supporting_evidence_ids"],

        "context_evidence_ids":
            evidence_group["context_evidence_ids"],

        "display_start":
            evidence_group["context_start"],

        "display_end":
            evidence_group["context_end"],

        "speakers":
            evidence_group["speakers"],

        "speaker_check":
            speaker_check,

        "candidates_checked":
            len(analysis),

        "supporting_candidates":
            supporting_candidates,

        "rejected_candidates":
            supporting_result["rejected_candidates"]
    }


# ============================================================
# Test with the originality attribute
# ============================================================

test_verification = verify_attribute(
    attribute="originality",
    proposition="The remote control is intended to be original.",
    claim_speaker="SPEAKER_02",
    top_k=10,
    context_window=2
)

print("=" * 90)
print("ATTRIBUTE VERIFICATION TEST")
print("=" * 90)

print(
    f"Attribute       : "
    f"{test_verification['attribute']}"
)

print(
    f"Proposition     : "
    f"{test_verification['proposition']}"
)

print(
    f"Status          : "
    f"{test_verification['status']}"
)

print(
    f"Speaker check   : "
    f"{test_verification['speaker_check']}"
)

print(
    f"Supporting IDs  : "
    f"{test_verification['supporting_evidence_ids']}"
)

print(
    f"Context IDs     : "
    f"{test_verification['context_evidence_ids']}"
)

print(
    f"Display interval: "
    f"{test_verification['display_start']:.3f} → "
    f"{test_verification['display_end']:.3f}"
)

print(
    f"Speakers        : "
    f"{test_verification['speakers']}"
)

print(
    f"Reason          : "
    f"{test_verification['reason']}"
)

print("\nSupporting evidence:")

for item in test_verification[
    "supporting_candidates"
]:

    print(
        f"  ID {item['evidence_id']} | "
        f"BGE={item['retrieval_similarity']:.4f} | "
        f"NLI={item['nli_label']} "
        f"{item['nli_entailment']:.4f}"
    )

print("\nAttribute verification test complete.")

ATTRIBUTE VERIFICATION TEST
Attribute       : originality
Proposition     : The remote control is intended to be original.
Status          : VERIFIED
Speaker check   : True
Supporting IDs  : [14, 15]
Context IDs     : [12, 13, 14, 15, 16, 17]
Display interval: 117.699 → 132.290
Speakers        : ['SPEAKER_02']
Reason          : The proposition is supported by contextual NLI evidence and the claimed speaker is consistent with the supporting evidence.

Supporting evidence:
  ID 14 | BGE=0.7810 | NLI=ENTAILMENT 0.9812
  ID 15 | BGE=0.6708 | NLI=ENTAILMENT 0.9787

Attribute verification test complete.


In [37]:
# ============================================================
# M7 - Speaker Consistency Negative Test
# ============================================================
# Purpose:
#   Verify that the system flags a proposition when the
#   claimed speaker is incorrect.
#
# Expected:
#   The evidence should still be found, but the speaker
#   consistency check should fail.
# ============================================================

wrong_speaker_test = verify_attribute(
    attribute="originality",
    proposition="The remote control is intended to be original.",
    claim_speaker="SPEAKER_03",   # Deliberately incorrect
    top_k=10,
    context_window=2
)

print("=" * 90)
print("SPEAKER CONSISTENCY NEGATIVE TEST")
print("=" * 90)

print(
    f"Attribute       : "
    f"{wrong_speaker_test['attribute']}"
)

print(
    f"Proposition     : "
    f"{wrong_speaker_test['proposition']}"
)

print(
    f"Claimed speaker : SPEAKER_03"
)

print(
    f"Evidence speaker: "
    f"{wrong_speaker_test['speakers']}"
)

print(
    f"Speaker check   : "
    f"{wrong_speaker_test['speaker_check']}"
)

print(
    f"Status          : "
    f"{wrong_speaker_test['status']}"
)

print(
    f"Supporting IDs  : "
    f"{wrong_speaker_test['supporting_evidence_ids']}"
)

print(
    f"Reason          : "
    f"{wrong_speaker_test['reason']}"
)

print("\nSpeaker consistency negative test complete.")

SPEAKER CONSISTENCY NEGATIVE TEST
Attribute       : originality
Proposition     : The remote control is intended to be original.
Claimed speaker : SPEAKER_03
Evidence speaker: []
Speaker check   : False
Status          : FLAGGED
Supporting IDs  : []
Reason          : No supporting evidence found.

Speaker consistency negative test complete.
